## Análise Exploratória dos Dados Brutos

## Hemoprod Alagoas

In [1]:
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)


In [2]:
import pandas as pd
import os
import numpy as np

# --- 1. Defina os caminhos para seus arquivos ---
# Ajuste os caminhos conforme a estrutura do seu projeto no notebook
dados_brutos_path = 'dados_brutos'
dicionario_path = 'dicionario_colunas_269.xlsx'
arquivo_dados_path = os.path.join(dados_brutos_path, 'Hemoprod_AL.xlsx')
nome_planilha = 'HEMOPROD - ALAGOAS'

# --- 2. Carregue os dados e o dicionário ---
try:
    # Carrega o arquivo de dados
    hemoprod_al = pd.read_excel(arquivo_dados_path, sheet_name=nome_planilha)
    print("Arquivo de dados carregado com sucesso.")
    print(f"Número de colunas original: {len(hemoprod_al.columns)}")

    # Carrega o arquivo de dicionário
    dicionario = pd.read_excel(dicionario_path)
    print("Arquivo de dicionário carregado com sucesso.")

    # --- 3. Limpeza Inicial dos Nomes das Colunas (ADICIONADO) ---
    # Mapeia as colunas, limpando espaços em branco no início e fim
    # e substituindo o caractere \xa0 (Non-breaking space) por um espaço normal.
    mapa_limpeza = {
        col: col.strip().replace('\xa0', ' ') 
        for col in hemoprod_al.columns
    }

    # Aplica a renomeação para limpar os nomes
    hemoprod_al = hemoprod_al.rename(columns=mapa_limpeza)
    print("Nomes das colunas do DataFrame limpos e padronizados (strip/\\xa0).")

    # --- 4. Crie o dicionário de mapeamento ---
    # Cria um dicionário no formato {'nome_original': 'nome_sql'} para uso no .rename()
    # NOTA: O 'nome_original' no dicionário TAMBÉM deve estar limpo para combinar.
    # Se o dicionário não estiver limpo, adicione a mesma limpeza aqui:
    dicionario['nome_original'] = dicionario['nome_original'].astype(str).str.strip().str.replace('\xa0', ' ')
    
    mapa_renomeacao = pd.Series(dicionario['nome_sql'].values, index=dicionario['nome_original']).to_dict()
    
    # Lista dos nomes de coluna no DataFrame atual APÓS a limpeza inicial
    colunas_atuais = set(hemoprod_al.columns)
    
    # Lista dos nomes originais de coluna no dicionário APÓS a limpeza (se aplicada)
    colunas_originais_dicionario = set(dicionario['nome_original'].tolist())
    
    # Lista dos nomes SQL (os desejados) no dicionário
    colunas_sql_desejadas = set(dicionario['nome_sql'].tolist())
    
    # --- 5. Renomeie as colunas com base no mapeamento (nome_original -> nome_sql) ---
    print("\n--- Processo de Renomeação ---")
    hemoprod_al.rename(columns=mapa_renomeacao, inplace=True)
    print("Colunas renomeadas com sucesso (apenas as que existiam no dicionário foram modificadas).")


   # --- 4. Inferir Tipos e Criar Mapeamento de Tipos ---
    
    # --- Lógica de Inferência customizada para 'Int64' ---
    tipos_inferidos = {}
    for col in hemoprod_al.columns:
        
        # 4.1. Tenta converter a coluna para o tipo que melhor representa seus dados (incluindo Int64/string/datetime)
        # O errors='ignore' é crucial para não falhar se a coluna não puder ser convertida
        coluna_convertida = pd.to_numeric(hemoprod_al[col], errors='coerce')
        
        # Se a conversão for bem-sucedida (não é totalmente NaN, nem totalmente string)
        if not coluna_convertida.isna().all() and coluna_convertida.dtype.kind in 'fi': # 'f' para float, 'i' para int
            
            # Se a coluna parecer um número, tentamos forçar para Int64
            # O .astype('Int64') usa o tipo inteiro que suporta NaN
            try:
                # Se for possível converter para Int64, use 'Int64'
                hemoprod_al[col] = hemoprod_al[col].astype('Int64')
                tipos_inferidos[col] = 'Int64'
            except Exception:
                # Se for numérico mas não puder ser Int64 (ex: float com muitas casas decimais), use 'float64'
                tipos_inferidos[col] = 'float64'
        
        # Verifica se é data/hora
        elif pd.api.types.is_datetime64_any_dtype(hemoprod_al[col]):
             tipos_inferidos[col] = 'datetime64[ns]'
             
        # Caso contrário, assume-se que é um texto/string
        else:
            tipos_inferidos[col] = 'object' # O padrão para string no pandas

    
    print("\nTipos de dados inferidos (amostra):")
    for i, (col, dtype) in enumerate(tipos_inferidos.items()):
        if i < 5:
            print(f"  {col}: {dtype}")
        if i == 5:
            print("  ...")
    
    # --- 5. Atualizar o DataFrame Dicionário ---
    
    # 5.1. Cria a coluna 'tipo_dados' no dicionário e preenche com os tipos inferidos
    dicionario['tipo_dados'] = dicionario['nome_sql'].map(tipos_inferidos)
    
    # 5.2. Trata colunas não encontradas no DataFrame de dados
    dicionario['tipo_dados'] = dicionario['tipo_dados'].fillna('object')
    
    print("\n--- Dicionário Atualizado ---")
    print("Coluna 'tipo_dados' criada com sucesso, com numéricos definidos como 'Int64' ou 'float64'.")
    
    # --- 6. Salvar o Dicionário Atualizado ---

    # Sugestão de novo caminho para salvar
    novo_dicionario_path = 'dicionario_colunas_269_COM_TIPOS_V2.xlsx'
    
    dicionario.to_excel(novo_dicionario_path, index=False)
    
    print(f"\n✅ Dicionário salvo com a nova coluna 'tipo_dados' em: {novo_dicionario_path}")
    print("\nPrimeiras linhas do dicionário atualizado:")
    display(dicionario.head())

    # --- 6. Análise de Colunas (Qualidade dos Dados) ---

    # 6.1. Colunas que não puderam ser renomeadas (existem no DF, mas não no 'nome_original' do dicionário)
    # Aqui usamos as colunas atuais ANTES da renomeação para ver o que sobrou.
    colunas_nao_mapeadas = [
        col for col in colunas_atuais 
        if col not in colunas_originais_dicionario
    ]

    print("-" * 30)
    print("Análise de Colunas do DataFrame (hemoprod_al):")
    print(f"Número de colunas não mapeadas: {len(colunas_nao_mapeadas)}")
    print(f"Colunas não mapeadas (manterão nome original): {colunas_nao_mapeadas}")
    
    # 6.2. Análise de Colunas Faltantes/A Mais (Comparação com 'nome_sql' desejado)
    colunas_apos_renomeacao = set(hemoprod_al.columns)
    
    colunas_faltantes = list(colunas_sql_desejadas - colunas_apos_renomeacao)
    colunas_a_mais = list(colunas_apos_renomeacao - colunas_sql_desejadas)

    print("-" * 30)
    print("Análise de Colunas (Comparação com a lista SQL DESEJADA):")
    print(f"Número de colunas FALTANTES: {len(colunas_faltantes)}")
    print(f"Colunas FALTANTES (deveriam estar, mas não estão): {colunas_faltantes}")
    print("-" * 30)
    print(f"Número de colunas A MAIS: {len(colunas_a_mais)}")
    print(f"Colunas A MAIS (estão no DF, mas não na lista SQL desejada): {colunas_a_mais}")
    print("-" * 30)

    # --- 7. Verifique o resultado ---
    print("\nInformações do DataFrame com as novas colunas:")
    hemoprod_al.info()
    
    print("\nAs 5 primeiras linhas com as novas colunas:")
    display(hemoprod_al.head())

except FileNotFoundError as e:
    print(f"\nErro de arquivo não encontrado: {e}")
except KeyError as e:
    print(f"\nErro de coluna não encontrada: {e}. Verifique se as colunas 'nome_original' e 'nome_sql' existem no seu arquivo de dicionário.")
except Exception as e:
    print(f"\nOcorreu um erro inesperado: {e}")

Arquivo de dados carregado com sucesso.
Número de colunas original: 269
Arquivo de dicionário carregado com sucesso.
Nomes das colunas do DataFrame limpos e padronizados (strip/\xa0).

--- Processo de Renomeação ---
Colunas renomeadas com sucesso (apenas as que existiam no dicionário foram modificadas).

Tipos de dados inferidos (amostra):
  id: Int64
  data_envio: object
  ultima_pagina: Int64
  idioma_inicial: object
  semente: Int64
  ...

--- Dicionário Atualizado ---
Coluna 'tipo_dados' criada com sucesso, com numéricos definidos como 'Int64' ou 'float64'.

✅ Dicionário salvo com a nova coluna 'tipo_dados' em: dicionario_colunas_269_COM_TIPOS_V2.xlsx

Primeiras linhas do dicionário atualizado:


,nome_original,nome_sql,comentario,tipo_dados
0,ID da resposta,id,Identificador único e chave primária da submis...,Int64
1,Data de envio,data_envio,Data e hora exatas do envio final do formulário.,object
2,Última página,ultima_pagina,Título da última página do formulário acessada...,Int64
3,Idioma inicial,idioma_inicial,Idioma selecionado no início do preenchimento ...,object
4,Semente,semente,Valor interno de semente/aleatorização (uso té...,Int64


------------------------------
Análise de Colunas do DataFrame (hemoprod_al):
Número de colunas não mapeadas: 0
Colunas não mapeadas (manterão nome original): []
------------------------------
Análise de Colunas (Comparação com a lista SQL DESEJADA):
Número de colunas FALTANTES: 0
Colunas FALTANTES (deveriam estar, mas não estão): []
------------------------------
Número de colunas A MAIS: 0
Colunas A MAIS (estão no DF, mas não na lista SQL desejada): []
------------------------------

Informações do DataFrame com as novas colunas:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 315 entries, 0 to 314
Columns: 269 entries, id to hemoprod_3_observacoes
dtypes: Int64(235), float64(16), object(18)
memory usage: 734.4+ KB

As 5 primeiras linhas com as novas colunas:


,id,data_envio,ultima_pagina,idioma_inicial,semente,codigo_acesso,data_inicio,data_ultima_acao,ip,identificacao_dado,tipo_envio,ano_referencia,periodo_referencia,identificacao_estabelecimento,municipio,razao_social_nome_fantasia,razao_social_nome_fantasia_outros,cnpj,tipo_estabelecimento,natureza_estabelecimento,dados_informados_referem_se,rede_estabelecimento,cnes,endereco,triagem_clinica_total_doacao_espontanea_aptos,triagem_clinica_total_doacao_espontanea_inaptos,triagem_clinica_total_doacao_reposicao_aptos,triagem_clinica_total_doacao_reposicao_inaptos,triagem_clinica_total_doacao_autologa_aptos,triagem_clinica_total_doacao_autologa_inaptos,total_doador_primeira_vez_aptos,total_doador_primeira_vez_inaptos,total_doador_repeticao_aptos,total_doador_repeticao_inaptos,total_doador_esporadico_aptos,total_doador_esporadico_inaptos,total_doador_masculino_aptos,total_doador_masculino_inaptos,total_doador_feminino_aptos,total_doador_feminino_inaptos,total_doador_menor_de_18_anos_aptos,total_doador_menor_de_18_anos_inaptos,total_doador_18_ate_29_anos_aptos,total_doador_18_ate_29_anos_inaptos,total_doador_acima_de_29_anos_aptos,total_doador_acima_de_29_anos_inaptos,total_candidatos_inaptos_anemia_masculino,total_candidatos_inaptos_anemia_feminino,total_candidatos_inaptos_anemia_total,total_candidatos_inaptos_hipertensao_masculino,total_candidatos_inaptos_hipertensao_feminino,total_candidatos_inaptos_hipertensao_total,total_candidatos_inaptos_hipotensao_masculino,total_candidatos_inaptos_hipotensao_feminino,total_candidatos_inaptos_hipotensao_total,total_candidatos_inaptos_alcoolismo_masculino,total_candidatos_inaptos_alcoolismo_feminino,total_candidatos_inaptos_alcoolismo_total,total_candidatos_inaptos_comportamento_risco_dst_masculino,total_candidatos_inaptos_comportamento_risco_dst_feminino,total_candidatos_inaptos_comportamento_risco_dst_total,total_candidatos_inaptos_uso_drogas_masculino,total_candidatos_inaptos_uso_drogas_feminino,total_candidatos_inaptos_uso_drogas_total,total_candidatos_inaptos_hepatite_masculino,total_candidatos_inaptos_hepatite_feminino,total_candidatos_inaptos_hepatite_total,total_candidatos_inaptos_doenca_chagas_masculino,total_candidatos_inaptos_doenca_chagas_feminino,total_candidatos_inaptos_doenca_chagas_total,total_candidatos_inaptos_malaria_masculino,total_candidatos_inaptos_malaria_feminino,total_candidatos_inaptos_malaria_total,total_candidatos_inaptos_outras_masculino,total_candidatos_inaptos_outras_feminino,total_candidatos_inaptos_outras_total,coleta_total_candidatos_desistentes,total_interrupcoes_coleta_dificuldade_puncao_venosa,total_interrupcoes_coleta_reacao_vagal,total_interrupcoes_coleta_outros_motivos,total_coletas_sangue_total,total_coletas_aferese,hemoprod_1_observacoes,exames_triagem_doenca_doenca_chagas_amostras_testadas,exames_triagem_doenca_doenca_chagas_amostras_reagentes,exames_triagem_doenca_hiv_amostras_testadas,exames_triagem_doenca_hiv_amostras_reagentes,exames_triagem_doenca_sifilis_amostras_testadas,`exames_triagem_doenca_sifilis_amostras_reagentes,exames_triagem_doenca_hepatite_b_hbs_ag_amostras_testadas,exames_triagem_doenca_hepatite_b_hbs_ag_amostras_reagentes,exames_triagem_doenca_hepatite_b_anti_hbc_amostras_testadas,exames_triagem_doenca_hepatite_b_anti_hbc_amostras_reagentes,exames_triagem_doenca_hepatite_c_amostras_testadas,exames_triagem_doenca_hepatite_c_amostras_reagentes,exames_triagem_doenca_htlv_i_ii_amostras_testadas,exames_triagem_doenca_htlv_i_ii_amostras_reagentes,exames_triagem_doenca_malaria_amostras_testadas,exames_triagem_doenca_malaria_amostras_reagentes,exames_triagem_doenca_hbv_teste_nat_amostras_testadas,exames_triagem_doenca_hbv_teste_nat_amostras_reagentes,exames_triagem_doenca_hcv_teste_nat_amostras_testadas,exames_triagem_doenca_hcv_teste_nat_amostras_reagentes,exames_triagem_doenca_hiv_teste_nat_amostras_testadas,exames_triagem_doenca_hiv_teste_nat_amostras_reagentes,imunohematologia_a_positivo_doador,imunohematologia_a_positivo_receptor,imunohe

In [3]:
# ASSUME QUE AS VARIÁVEIS 'hemoprod_al', 'dicionario', 'colunas_a_mais', 
# 'colunas_faltantes' E 'colunas_sql_desejadas' EXISTEM DA CÉLULA ANTERIOR.

print("--- 6. PADRONIZAÇÃO DO ESQUEMA (Remoção, Adição e Reordenação) ---")

# 6.1. REMOVE as Colunas a Mais
if colunas_a_mais:
    # Verificação de segurança: Só tenta dropar colunas que estão realmente no DF
    colunas_para_dropar = [col for col in colunas_a_mais if col in hemoprod_al.columns]
    hemoprod_al.drop(columns=colunas_para_dropar, inplace=True)
    print(f"✅ {len(colunas_para_dropar)} colunas a mais removidas.")
else:
    print("Nenhuma coluna a mais para remover.")

# 6.2. ADICIONA as Colunas Faltantes
if colunas_faltantes:
    
    # Prepara o sub-dicionário apenas com as colunas faltantes (para tipos)
    dicionario_faltante = dicionario[dicionario['nome_sql'].isin(colunas_faltantes)]
    
    for col_sql in colunas_faltantes:
        
        # Adiciona a coluna com valor nulo (NaN)
        hemoprod_al[col_sql] = np.nan 

        # --- Lógica Opcional de Conversão de Tipo (Baseada na coluna 'tipo_dados' no dicionário) ---
        # Tenta encontrar o tipo no dicionário (apenas se a coluna 'tipo_dados' existir e não estiver vazia)
        if 'tipo_dados' in dicionario.columns:
            tipo_desejado_row = dicionario_faltante[dicionario_faltante['nome_sql'] == col_sql]
            
            if not tipo_desejado_row.empty:
                tipo_desejado = tipo_desejado_row['tipo_dados'].iloc[0].lower()

                # Converte para o tipo, se for reconhecido
                if 'int' in tipo_desejado or 'float' in tipo_desejado:
                    # Usa 'float64' para números que podem ter NaNs, para evitar erros de Pandas
                    hemoprod_al[col_sql] = hemoprod_al[col_sql].astype('float64') 
                elif 'string' in tipo_desejado or 'object' in tipo_desejado:
                    # Converte para 'object' (string)
                    hemoprod_al[col_sql] = hemoprod_al[col_sql].astype('object')
                 
    print(f"✅ {len(colunas_faltantes)} colunas faltantes adicionadas e preenchidas com NaN.")
else:
    print("Nenhuma coluna faltante para adicionar.")


# 6.3. Reordena as colunas (Para garantir a ordem padronizada do dicionário)
colunas_finais = dicionario['nome_sql'].tolist()
# Garante que só as colunas que realmente existem no DF (após drop/adição) sejam usadas na reordenação
colunas_finais_presentes = [col for col in colunas_finais if col in hemoprod_al.columns]

# Esta é a linha que reordena o DataFrame
hemoprod_al = hemoprod_al[colunas_finais_presentes]
print("✅ Colunas reordenadas para seguir a ordem do dicionário.")


# --- 7. Verifique o resultado Final ---
print("\n--- Resultado Final ---")
print(f"Número de colunas final: {len(hemoprod_al.columns)}")
print("Informações do DataFrame padronizado:")
hemoprod_al.info()

print("\nAs 5 primeiras linhas com as colunas padronizadas:")
display(hemoprod_al.head())

--- 6. PADRONIZAÇÃO DO ESQUEMA (Remoção, Adição e Reordenação) ---
Nenhuma coluna a mais para remover.
Nenhuma coluna faltante para adicionar.
✅ Colunas reordenadas para seguir a ordem do dicionário.

--- Resultado Final ---
Número de colunas final: 269
Informações do DataFrame padronizado:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 315 entries, 0 to 314
Columns: 269 entries, id to hemoprod_3_observacoes
dtypes: Int64(235), float64(16), object(18)
memory usage: 734.4+ KB

As 5 primeiras linhas com as colunas padronizadas:


,id,data_envio,ultima_pagina,idioma_inicial,semente,codigo_acesso,data_inicio,data_ultima_acao,ip,identificacao_dado,tipo_envio,ano_referencia,periodo_referencia,identificacao_estabelecimento,municipio,razao_social_nome_fantasia,razao_social_nome_fantasia_outros,cnpj,tipo_estabelecimento,natureza_estabelecimento,dados_informados_referem_se,rede_estabelecimento,cnes,endereco,triagem_clinica_total_doacao_espontanea_aptos,triagem_clinica_total_doacao_espontanea_inaptos,triagem_clinica_total_doacao_reposicao_aptos,triagem_clinica_total_doacao_reposicao_inaptos,triagem_clinica_total_doacao_autologa_aptos,triagem_clinica_total_doacao_autologa_inaptos,total_doador_primeira_vez_aptos,total_doador_primeira_vez_inaptos,total_doador_repeticao_aptos,total_doador_repeticao_inaptos,total_doador_esporadico_aptos,total_doador_esporadico_inaptos,total_doador_masculino_aptos,total_doador_masculino_inaptos,total_doador_feminino_aptos,total_doador_feminino_inaptos,total_doador_menor_de_18_anos_aptos,total_doador_menor_de_18_anos_inaptos,total_doador_18_ate_29_anos_aptos,total_doador_18_ate_29_anos_inaptos,total_doador_acima_de_29_anos_aptos,total_doador_acima_de_29_anos_inaptos,total_candidatos_inaptos_anemia_masculino,total_candidatos_inaptos_anemia_feminino,total_candidatos_inaptos_anemia_total,total_candidatos_inaptos_hipertensao_masculino,total_candidatos_inaptos_hipertensao_feminino,total_candidatos_inaptos_hipertensao_total,total_candidatos_inaptos_hipotensao_masculino,total_candidatos_inaptos_hipotensao_feminino,total_candidatos_inaptos_hipotensao_total,total_candidatos_inaptos_alcoolismo_masculino,total_candidatos_inaptos_alcoolismo_feminino,total_candidatos_inaptos_alcoolismo_total,total_candidatos_inaptos_comportamento_risco_dst_masculino,total_candidatos_inaptos_comportamento_risco_dst_feminino,total_candidatos_inaptos_comportamento_risco_dst_total,total_candidatos_inaptos_uso_drogas_masculino,total_candidatos_inaptos_uso_drogas_feminino,total_candidatos_inaptos_uso_drogas_total,total_candidatos_inaptos_hepatite_masculino,total_candidatos_inaptos_hepatite_feminino,total_candidatos_inaptos_hepatite_total,total_candidatos_inaptos_doenca_chagas_masculino,total_candidatos_inaptos_doenca_chagas_feminino,total_candidatos_inaptos_doenca_chagas_total,total_candidatos_inaptos_malaria_masculino,total_candidatos_inaptos_malaria_feminino,total_candidatos_inaptos_malaria_total,total_candidatos_inaptos_outras_masculino,total_candidatos_inaptos_outras_feminino,total_candidatos_inaptos_outras_total,coleta_total_candidatos_desistentes,total_interrupcoes_coleta_dificuldade_puncao_venosa,total_interrupcoes_coleta_reacao_vagal,total_interrupcoes_coleta_outros_motivos,total_coletas_sangue_total,total_coletas_aferese,hemoprod_1_observacoes,exames_triagem_doenca_doenca_chagas_amostras_testadas,exames_triagem_doenca_doenca_chagas_amostras_reagentes,exames_triagem_doenca_hiv_amostras_testadas,exames_triagem_doenca_hiv_amostras_reagentes,exames_triagem_doenca_sifilis_amostras_testadas,`exames_triagem_doenca_sifilis_amostras_reagentes,exames_triagem_doenca_hepatite_b_hbs_ag_amostras_testadas,exames_triagem_doenca_hepatite_b_hbs_ag_amostras_reagentes,exames_triagem_doenca_hepatite_b_anti_hbc_amostras_testadas,exames_triagem_doenca_hepatite_b_anti_hbc_amostras_reagentes,exames_triagem_doenca_hepatite_c_amostras_testadas,exames_triagem_doenca_hepatite_c_amostras_reagentes,exames_triagem_doenca_htlv_i_ii_amostras_testadas,exames_triagem_doenca_htlv_i_ii_amostras_reagentes,exames_triagem_doenca_malaria_amostras_testadas,exames_triagem_doenca_malaria_amostras_reagentes,exames_triagem_doenca_hbv_teste_nat_amostras_testadas,exames_triagem_doenca_hbv_teste_nat_amostras_reagentes,exames_triagem_doenca_hcv_teste_nat_amostras_testadas,exames_triagem_doenca_hcv_teste_nat_amostras_reagentes,exames_triagem_doenca_hiv_teste_nat_amostras_testadas,exames_triagem_doenca_hiv_teste_nat_amostras_reagentes,imunohematologia_a_positivo_doador,imunohematologia_a_positivo_receptor,imunohe

In [4]:
# --- 2. Defina as colunas para a chave e para ordenação ---
registros_antes = len(hemoprod_al)
print(f"Total de registros ANTES da remoção de duplicatas: {registros_antes}")
    

colunas_chave = [
        'cnpj', 
        'ano_referencia', 
        'periodo_referencia', 
        'razao_social_nome_fantasia'
    ]

coluna_data = 'data_envio'

# --- 3. Verifique se as colunas necessárias existem ---
colunas_necessarias = colunas_chave + [coluna_data]

if not all(col in hemoprod_al.columns for col in colunas_necessarias):
    print("\n--- ERRO ---")
    print("Uma ou mais colunas necessárias para a deduplicação não foram encontradas.")
    colunas_faltantes = [col for col in colunas_necessarias if col not in hemoprod_al.columns]
    print(f"Colunas necessárias: {colunas_necessarias}")
    print(f"Colunas faltantes no DataFrame: {colunas_faltantes}")
else:
    # --- 4. Prepare a coluna de data e ordene os dados ---
    # Converte a coluna 'data_envio' para datetime para garantir a ordenação correta.
    # 'errors='coerce'' transformará datas inválidas em NaT (Not a Time), que são tratadas como nulas.
    hemoprod_al[coluna_data] = pd.to_datetime(hemoprod_al[coluna_data], errors='coerce')

    # Ordena o DataFrame. Os registros com data de envio mais recente ficarão por último.
    print(f"\nOrdenando os dados por '{coluna_data}'...")
    hemoprod_al_ordenado = hemoprod_al.sort_values(by=coluna_data, ascending=True)

    # --- 5. Identifique e separe os registros duplicados e únicos ---
    # Em vez de usar drop_duplicates() diretamente, vamos usar duplicated()
    # para criar uma máscara booleana.
    # 'keep='last'' marca todas as ocorrências de uma chave como True, EXCETO a última (a mais recente).
    print(f"Identificando duplicatas com base na chave: {colunas_chave}...")
    mascara_duplicatas = hemoprod_al_ordenado.duplicated(subset=colunas_chave, keep='last')

    # O DataFrame de removidos conterá todas as linhas marcadas como True
    hemoprod_removidos = hemoprod_al_ordenado[mascara_duplicatas]
    
    # O DataFrame deduplicado conterá o INVERSO (~) da máscara (linhas marcadas como False)
    hemoprod_al_deduplicado = hemoprod_al_ordenado[~mascara_duplicatas]
    
    # Agora podemos contar os registros diretamente dos novos DataFrames
    registros_depois = len(hemoprod_al_deduplicado)
    registros_removidos = len(hemoprod_removidos) 
    
    # --- 6. Exiba o resultado ---
    print("\n--- Processo Concluído ---")
    print(f"Registros removidos: {registros_removidos}")
    print(f"Total de registros DEPOIS da remoção de duplicatas: {registros_depois}")

    # (Opcional) Exibe a amostra dos removidos
    print("\nAmostra dos dados REMOVIDOS (os mais antigos/duplicados):")
    display(hemoprod_removidos.head(10))

    # Você pode continuar a usar o DataFrame 'hemoprod_al_deduplicado' para suas análises
    print("\nAmostra dos dados únicos (os mais recentes para cada chave):")
    display(hemoprod_al_deduplicado.head(10))
    
    # Agora você tem o DataFrame 'hemoprod_removidos' salvo

Total de registros ANTES da remoção de duplicatas: 315

Ordenando os dados por 'data_envio'...
Identificando duplicatas com base na chave: ['cnpj', 'ano_referencia', 'periodo_referencia', 'razao_social_nome_fantasia']...

--- Processo Concluído ---
Registros removidos: 8
Total de registros DEPOIS da remoção de duplicatas: 307

Amostra dos dados REMOVIDOS (os mais antigos/duplicados):


,id,data_envio,ultima_pagina,idioma_inicial,semente,codigo_acesso,data_inicio,data_ultima_acao,ip,identificacao_dado,tipo_envio,ano_referencia,periodo_referencia,identificacao_estabelecimento,municipio,razao_social_nome_fantasia,razao_social_nome_fantasia_outros,cnpj,tipo_estabelecimento,natureza_estabelecimento,dados_informados_referem_se,rede_estabelecimento,cnes,endereco,triagem_clinica_total_doacao_espontanea_aptos,triagem_clinica_total_doacao_espontanea_inaptos,triagem_clinica_total_doacao_reposicao_aptos,triagem_clinica_total_doacao_reposicao_inaptos,triagem_clinica_total_doacao_autologa_aptos,triagem_clinica_total_doacao_autologa_inaptos,total_doador_primeira_vez_aptos,total_doador_primeira_vez_inaptos,total_doador_repeticao_aptos,total_doador_repeticao_inaptos,total_doador_esporadico_aptos,total_doador_esporadico_inaptos,total_doador_masculino_aptos,total_doador_masculino_inaptos,total_doador_feminino_aptos,total_doador_feminino_inaptos,total_doador_menor_de_18_anos_aptos,total_doador_menor_de_18_anos_inaptos,total_doador_18_ate_29_anos_aptos,total_doador_18_ate_29_anos_inaptos,total_doador_acima_de_29_anos_aptos,total_doador_acima_de_29_anos_inaptos,total_candidatos_inaptos_anemia_masculino,total_candidatos_inaptos_anemia_feminino,total_candidatos_inaptos_anemia_total,total_candidatos_inaptos_hipertensao_masculino,total_candidatos_inaptos_hipertensao_feminino,total_candidatos_inaptos_hipertensao_total,total_candidatos_inaptos_hipotensao_masculino,total_candidatos_inaptos_hipotensao_feminino,total_candidatos_inaptos_hipotensao_total,total_candidatos_inaptos_alcoolismo_masculino,total_candidatos_inaptos_alcoolismo_feminino,total_candidatos_inaptos_alcoolismo_total,total_candidatos_inaptos_comportamento_risco_dst_masculino,total_candidatos_inaptos_comportamento_risco_dst_feminino,total_candidatos_inaptos_comportamento_risco_dst_total,total_candidatos_inaptos_uso_drogas_masculino,total_candidatos_inaptos_uso_drogas_feminino,total_candidatos_inaptos_uso_drogas_total,total_candidatos_inaptos_hepatite_masculino,total_candidatos_inaptos_hepatite_feminino,total_candidatos_inaptos_hepatite_total,total_candidatos_inaptos_doenca_chagas_masculino,total_candidatos_inaptos_doenca_chagas_feminino,total_candidatos_inaptos_doenca_chagas_total,total_candidatos_inaptos_malaria_masculino,total_candidatos_inaptos_malaria_feminino,total_candidatos_inaptos_malaria_total,total_candidatos_inaptos_outras_masculino,total_candidatos_inaptos_outras_feminino,total_candidatos_inaptos_outras_total,coleta_total_candidatos_desistentes,total_interrupcoes_coleta_dificuldade_puncao_venosa,total_interrupcoes_coleta_reacao_vagal,total_interrupcoes_coleta_outros_motivos,total_coletas_sangue_total,total_coletas_aferese,hemoprod_1_observacoes,exames_triagem_doenca_doenca_chagas_amostras_testadas,exames_triagem_doenca_doenca_chagas_amostras_reagentes,exames_triagem_doenca_hiv_amostras_testadas,exames_triagem_doenca_hiv_amostras_reagentes,exames_triagem_doenca_sifilis_amostras_testadas,`exames_triagem_doenca_sifilis_amostras_reagentes,exames_triagem_doenca_hepatite_b_hbs_ag_amostras_testadas,exames_triagem_doenca_hepatite_b_hbs_ag_amostras_reagentes,exames_triagem_doenca_hepatite_b_anti_hbc_amostras_testadas,exames_triagem_doenca_hepatite_b_anti_hbc_amostras_reagentes,exames_triagem_doenca_hepatite_c_amostras_testadas,exames_triagem_doenca_hepatite_c_amostras_reagentes,exames_triagem_doenca_htlv_i_ii_amostras_testadas,exames_triagem_doenca_htlv_i_ii_amostras_reagentes,exames_triagem_doenca_malaria_amostras_testadas,exames_triagem_doenca_malaria_amostras_reagentes,exames_triagem_doenca_hbv_teste_nat_amostras_testadas,exames_triagem_doenca_hbv_teste_nat_amostras_reagentes,exames_triagem_doenca_hcv_teste_nat_amostras_testadas,exames_triagem_doenca_hcv_teste_nat_amostras_reagentes,exames_triagem_doenca_hiv_teste_nat_amostras_testadas,exames_triagem_doenca_hiv_teste_nat_amostras_reagentes,imunohematologia_a_positivo_doador,imunohematologia_a_positivo_receptor,imunohe


Amostra dos dados únicos (os mais recentes para cada chave):


,id,data_envio,ultima_pagina,idioma_inicial,semente,codigo_acesso,data_inicio,data_ultima_acao,ip,identificacao_dado,tipo_envio,ano_referencia,periodo_referencia,identificacao_estabelecimento,municipio,razao_social_nome_fantasia,razao_social_nome_fantasia_outros,cnpj,tipo_estabelecimento,natureza_estabelecimento,dados_informados_referem_se,rede_estabelecimento,cnes,endereco,triagem_clinica_total_doacao_espontanea_aptos,triagem_clinica_total_doacao_espontanea_inaptos,triagem_clinica_total_doacao_reposicao_aptos,triagem_clinica_total_doacao_reposicao_inaptos,triagem_clinica_total_doacao_autologa_aptos,triagem_clinica_total_doacao_autologa_inaptos,total_doador_primeira_vez_aptos,total_doador_primeira_vez_inaptos,total_doador_repeticao_aptos,total_doador_repeticao_inaptos,total_doador_esporadico_aptos,total_doador_esporadico_inaptos,total_doador_masculino_aptos,total_doador_masculino_inaptos,total_doador_feminino_aptos,total_doador_feminino_inaptos,total_doador_menor_de_18_anos_aptos,total_doador_menor_de_18_anos_inaptos,total_doador_18_ate_29_anos_aptos,total_doador_18_ate_29_anos_inaptos,total_doador_acima_de_29_anos_aptos,total_doador_acima_de_29_anos_inaptos,total_candidatos_inaptos_anemia_masculino,total_candidatos_inaptos_anemia_feminino,total_candidatos_inaptos_anemia_total,total_candidatos_inaptos_hipertensao_masculino,total_candidatos_inaptos_hipertensao_feminino,total_candidatos_inaptos_hipertensao_total,total_candidatos_inaptos_hipotensao_masculino,total_candidatos_inaptos_hipotensao_feminino,total_candidatos_inaptos_hipotensao_total,total_candidatos_inaptos_alcoolismo_masculino,total_candidatos_inaptos_alcoolismo_feminino,total_candidatos_inaptos_alcoolismo_total,total_candidatos_inaptos_comportamento_risco_dst_masculino,total_candidatos_inaptos_comportamento_risco_dst_feminino,total_candidatos_inaptos_comportamento_risco_dst_total,total_candidatos_inaptos_uso_drogas_masculino,total_candidatos_inaptos_uso_drogas_feminino,total_candidatos_inaptos_uso_drogas_total,total_candidatos_inaptos_hepatite_masculino,total_candidatos_inaptos_hepatite_feminino,total_candidatos_inaptos_hepatite_total,total_candidatos_inaptos_doenca_chagas_masculino,total_candidatos_inaptos_doenca_chagas_feminino,total_candidatos_inaptos_doenca_chagas_total,total_candidatos_inaptos_malaria_masculino,total_candidatos_inaptos_malaria_feminino,total_candidatos_inaptos_malaria_total,total_candidatos_inaptos_outras_masculino,total_candidatos_inaptos_outras_feminino,total_candidatos_inaptos_outras_total,coleta_total_candidatos_desistentes,total_interrupcoes_coleta_dificuldade_puncao_venosa,total_interrupcoes_coleta_reacao_vagal,total_interrupcoes_coleta_outros_motivos,total_coletas_sangue_total,total_coletas_aferese,hemoprod_1_observacoes,exames_triagem_doenca_doenca_chagas_amostras_testadas,exames_triagem_doenca_doenca_chagas_amostras_reagentes,exames_triagem_doenca_hiv_amostras_testadas,exames_triagem_doenca_hiv_amostras_reagentes,exames_triagem_doenca_sifilis_amostras_testadas,`exames_triagem_doenca_sifilis_amostras_reagentes,exames_triagem_doenca_hepatite_b_hbs_ag_amostras_testadas,exames_triagem_doenca_hepatite_b_hbs_ag_amostras_reagentes,exames_triagem_doenca_hepatite_b_anti_hbc_amostras_testadas,exames_triagem_doenca_hepatite_b_anti_hbc_amostras_reagentes,exames_triagem_doenca_hepatite_c_amostras_testadas,exames_triagem_doenca_hepatite_c_amostras_reagentes,exames_triagem_doenca_htlv_i_ii_amostras_testadas,exames_triagem_doenca_htlv_i_ii_amostras_reagentes,exames_triagem_doenca_malaria_amostras_testadas,exames_triagem_doenca_malaria_amostras_reagentes,exames_triagem_doenca_hbv_teste_nat_amostras_testadas,exames_triagem_doenca_hbv_teste_nat_amostras_reagentes,exames_triagem_doenca_hcv_teste_nat_amostras_testadas,exames_triagem_doenca_hcv_teste_nat_amostras_reagentes,exames_triagem_doenca_hiv_teste_nat_amostras_testadas,exames_triagem_doenca_hiv_teste_nat_amostras_reagentes,imunohematologia_a_positivo_doador,imunohematologia_a_positivo_receptor,imunohe

In [5]:
hemoprod_al_deduplicado.to_excel('dados_processados/hemoprod_al.xlsx', index=False)

## Hemoprod Amazonas

In [20]:
import pandas as pd
import os

# --- 1. Defina os caminhos para seus arquivos ---
# Ajuste os caminhos conforme a estrutura do seu projeto no notebook
dados_brutos_path = 'dados_brutos'
dicionario_path = 'dicionario_colunas_269.xlsx'
arquivo_dados_path = os.path.join(dados_brutos_path, 'Hemoprod_AM.xlsx')
nome_planilha = 'HEMOPROD - AMAZONAS'

try:
    # Carrega o arquivo de dados
    hemoprod_am = pd.read_excel(arquivo_dados_path, sheet_name=nome_planilha)
    print("Arquivo de dados carregado com sucesso.")
    print(f"Número de colunas original: {len(hemoprod_am.columns)}")

    # Carrega o arquivo de dicionário
    dicionario = pd.read_excel(dicionario_path)
    print("Arquivo de dicionário carregado com sucesso.")

    # --- 3. Limpeza Inicial dos Nomes das Colunas (ADICIONADO) ---
    # Mapeia as colunas, limpando espaços em branco no início e fim
    # e substituindo o caractere \xa0 (Non-breaking space) por um espaço normal.
    mapa_limpeza = {
        col: col.strip().replace('\xa0', ' ') 
        for col in hemoprod_am.columns
    }

    # Aplica a renomeação para limpar os nomes
    hemoprod_am = hemoprod_am.rename(columns=mapa_limpeza)
    print("Nomes das colunas do DataFrame limpos e padronizados (strip/\\xa0).")

    # --- 4. Crie o dicionário de mapeamento ---
    # Cria um dicionário no formato {'nome_original': 'nome_sql'} para uso no .rename()
    # NOTA: O 'nome_original' no dicionário TAMBÉM deve estar limpo para combinar.
    # Se o dicionário não estiver limpo, adicione a mesma limpeza aqui:
    dicionario['nome_original'] = dicionario['nome_original'].astype(str).str.strip().str.replace('\xa0', ' ')
    
    mapa_renomeacao = pd.Series(dicionario['nome_sql'].values, index=dicionario['nome_original']).to_dict()
    
    # Lista dos nomes de coluna no DataFrame atual APÓS a limpeza inicial
    colunas_atuais = set(hemoprod_am.columns)
    
    # Lista dos nomes originais de coluna no dicionário APÓS a limpeza (se aplicada)
    colunas_originais_dicionario = set(dicionario['nome_original'].tolist())
    
    # Lista dos nomes SQL (os desejados) no dicionário
    colunas_sql_desejadas = set(dicionario['nome_sql'].tolist())
    
    # --- 5. Renomeie as colunas com base no mapeamento (nome_original -> nome_sql) ---
    print("\n--- Processo de Renomeação ---")
    hemoprod_am.rename(columns=mapa_renomeacao, inplace=True)
    print("Colunas renomeadas com sucesso (apenas as que existiam no dicionário foram modificadas).")


   # --- 4. Inferir Tipos e Criar Mapeamento de Tipos ---
    
    # --- Lógica de Inferência customizada para 'Int64' ---
    tipos_inferidos = {}
    for col in hemoprod_am.columns:
        
        # 4.1. Tenta converter a coluna para o tipo que melhor representa seus dados (incluindo Int64/string/datetime)
        # O errors='ignore' é crucial para não falhar se a coluna não puder ser convertida
        coluna_convertida = pd.to_numeric(hemoprod_am[col], errors='coerce')
        
        # Se a conversão for bem-sucedida (não é totalmente NaN, nem totalmente string)
        if not coluna_convertida.isna().all() and coluna_convertida.dtype.kind in 'fi': # 'f' para float, 'i' para int
            
            # Se a coluna parecer um número, tentamos forçar para Int64
            # O .astype('Int64') usa o tipo inteiro que suporta NaN
            try:
                # Se for possível converter para Int64, use 'Int64'
                hemoprod_am[col] = hemoprod_am[col].astype('Int64')
                tipos_inferidos[col] = 'Int64'
            except Exception:
                # Se for numérico mas não puder ser Int64 (ex: float com muitas casas decimais), use 'float64'
                tipos_inferidos[col] = 'float64'
        
        # Verifica se é data/hora
        elif pd.api.types.is_datetime64_any_dtype(hemoprod_am[col]):
             tipos_inferidos[col] = 'datetime64[ns]'
             
        # Caso contrário, assume-se que é um texto/string
        else:
            tipos_inferidos[col] = 'object' # O padrão para string no pandas

    
    print("\nTipos de dados inferidos (amostra):")
    for i, (col, dtype) in enumerate(tipos_inferidos.items()):
        if i < 5:
            print(f"  {col}: {dtype}")
        if i == 5:
            print("  ...")
    
    # --- 5. Atualizar o DataFrame Dicionário ---
    
    # 5.1. Cria a coluna 'tipo_dados' no dicionário e preenche com os tipos inferidos
    dicionario['tipo_dados'] = dicionario['nome_sql'].map(tipos_inferidos)
    
    # 5.2. Trata colunas não encontradas no DataFrame de dados
    dicionario['tipo_dados'] = dicionario['tipo_dados'].fillna('object')
    
    print("\n--- Dicionário Atualizado ---")
    print("Coluna 'tipo_dados' criada com sucesso, com numéricos definidos como 'Int64' ou 'float64'.")
    
    # --- 6. Salvar o Dicionário Atualizado ---

    # Sugestão de novo caminho para salvar
    # novo_dicionario_path = 'dicionario_colunas_269_COM_TIPOS_V3.xlsx'
    
    # dicionario.to_excel(novo_dicionario_path, index=False)
    
    print(f"\n✅ Dicionário salvo com a nova coluna 'tipo_dados' em: {novo_dicionario_path}")
    print("\nPrimeiras linhas do dicionário atualizado:")
    display(dicionario.head())

    # --- 6. Análise de Colunas (Qualidade dos Dados) ---

    # 6.1. Colunas que não puderam ser renomeadas (existem no DF, mas não no 'nome_original' do dicionário)
    # Aqui usamos as colunas atuais ANTES da renomeação para ver o que sobrou.
    colunas_nao_mapeadas = [
        col for col in colunas_atuais 
        if col not in colunas_originais_dicionario
    ]

    print("-" * 30)
    print("Análise de Colunas do DataFrame (hemoprod_am):")
    print(f"Número de colunas não mapeadas: {len(colunas_nao_mapeadas)}")
    print(f"Colunas não mapeadas (manterão nome original): {colunas_nao_mapeadas}")
    
    # 6.2. Análise de Colunas Faltantes/A Mais (Comparação com 'nome_sql' desejado)
    colunas_apos_renomeacao = set(hemoprod_am.columns)
    
    colunas_faltantes = list(colunas_sql_desejadas - colunas_apos_renomeacao)
    colunas_a_mais = list(colunas_apos_renomeacao - colunas_sql_desejadas)

    print("-" * 30)
    print("Análise de Colunas (Comparação com a lista SQL DESEJADA):")
    print(f"Número de colunas FALTANTES: {len(colunas_faltantes)}")
    print(f"Colunas FALTANTES (deveriam estar, mas não estão): {colunas_faltantes}")
    print("-" * 30)
    print(f"Número de colunas A MAIS: {len(colunas_a_mais)}")
    print(f"Colunas A MAIS (estão no DF, mas não na lista SQL desejada): {colunas_a_mais}")
    print("-" * 30)

    # --- 7. Verifique o resultado ---
    print("\nInformações do DataFrame com as novas colunas:")
    hemoprod_am.info()
    
    print("\nAs 5 primeiras linhas com as novas colunas:")
    display(hemoprod_am.head())

except FileNotFoundError as e:
    print(f"\nErro de arquivo não encontrado: {e}")
except KeyError as e:
    print(f"\nErro de coluna não encontrada: {e}. Verifique se as colunas 'nome_original' e 'nome_sql' existem no seu arquivo de dicionário.")
except Exception as e:
    print(f"\nOcorreu um erro inesperado: {e}")

Arquivo de dados carregado com sucesso.
Número de colunas original: 270
Arquivo de dicionário carregado com sucesso.
Nomes das colunas do DataFrame limpos e padronizados (strip/\xa0).

--- Processo de Renomeação ---
Colunas renomeadas com sucesso (apenas as que existiam no dicionário foram modificadas).

Tipos de dados inferidos (amostra):
  id: Int64
  data_envio: object
  ultima_pagina: Int64
  idioma_inicial: object
  semente: Int64
  ...

--- Dicionário Atualizado ---
Coluna 'tipo_dados' criada com sucesso, com numéricos definidos como 'Int64' ou 'float64'.

✅ Dicionário salvo com a nova coluna 'tipo_dados' em: dicionario_colunas_269_COM_TIPOS_V2.xlsx

Primeiras linhas do dicionário atualizado:


,nome_original,nome_sql,comentario,tipo_dados
0,ID da resposta,id,Identificador único e chave primária da submis...,Int64
1,Data de envio,data_envio,Data e hora exatas do envio final do formulário.,object
2,Última página,ultima_pagina,Título da última página do formulário acessada...,Int64
3,Idioma inicial,idioma_inicial,Idioma selecionado no início do preenchimento ...,object
4,Semente,semente,Valor interno de semente/aleatorização (uso té...,Int64


------------------------------
Análise de Colunas do DataFrame (hemoprod_am):
Número de colunas não mapeadas: 1
Colunas não mapeadas (manterão nome original): ['URL de referência']
------------------------------
Análise de Colunas (Comparação com a lista SQL DESEJADA):
Número de colunas FALTANTES: 0
Colunas FALTANTES (deveriam estar, mas não estão): []
------------------------------
Número de colunas A MAIS: 1
Colunas A MAIS (estão no DF, mas não na lista SQL desejada): ['URL de referência']
------------------------------

Informações do DataFrame com as novas colunas:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32 entries, 0 to 31
Columns: 270 entries, id to hemoprod_3_observacoes
dtypes: Int64(229), float64(27), object(14)
memory usage: 74.8+ KB

As 5 primeiras linhas com as novas colunas:


,id,data_envio,ultima_pagina,idioma_inicial,semente,codigo_acesso,data_inicio,data_ultima_acao,ip,URL de referência,identificacao_dado,tipo_envio,ano_referencia,periodo_referencia,identificacao_estabelecimento,municipio,razao_social_nome_fantasia,razao_social_nome_fantasia_outros,cnpj,tipo_estabelecimento,natureza_estabelecimento,dados_informados_referem_se,rede_estabelecimento,cnes,endereco,triagem_clinica_total_doacao_espontanea_aptos,triagem_clinica_total_doacao_espontanea_inaptos,triagem_clinica_total_doacao_reposicao_aptos,triagem_clinica_total_doacao_reposicao_inaptos,triagem_clinica_total_doacao_autologa_aptos,triagem_clinica_total_doacao_autologa_inaptos,total_doador_primeira_vez_aptos,total_doador_primeira_vez_inaptos,total_doador_repeticao_aptos,total_doador_repeticao_inaptos,total_doador_esporadico_aptos,total_doador_esporadico_inaptos,total_doador_masculino_aptos,total_doador_masculino_inaptos,total_doador_feminino_aptos,total_doador_feminino_inaptos,total_doador_menor_de_18_anos_aptos,total_doador_menor_de_18_anos_inaptos,total_doador_18_ate_29_anos_aptos,total_doador_18_ate_29_anos_inaptos,total_doador_acima_de_29_anos_aptos,total_doador_acima_de_29_anos_inaptos,total_candidatos_inaptos_anemia_masculino,total_candidatos_inaptos_anemia_feminino,total_candidatos_inaptos_anemia_total,total_candidatos_inaptos_hipertensao_masculino,total_candidatos_inaptos_hipertensao_feminino,total_candidatos_inaptos_hipertensao_total,total_candidatos_inaptos_hipotensao_masculino,total_candidatos_inaptos_hipotensao_feminino,total_candidatos_inaptos_hipotensao_total,total_candidatos_inaptos_alcoolismo_masculino,total_candidatos_inaptos_alcoolismo_feminino,total_candidatos_inaptos_alcoolismo_total,total_candidatos_inaptos_comportamento_risco_dst_masculino,total_candidatos_inaptos_comportamento_risco_dst_feminino,total_candidatos_inaptos_comportamento_risco_dst_total,total_candidatos_inaptos_uso_drogas_masculino,total_candidatos_inaptos_uso_drogas_feminino,total_candidatos_inaptos_uso_drogas_total,total_candidatos_inaptos_hepatite_masculino,total_candidatos_inaptos_hepatite_feminino,total_candidatos_inaptos_hepatite_total,total_candidatos_inaptos_doenca_chagas_masculino,total_candidatos_inaptos_doenca_chagas_feminino,total_candidatos_inaptos_doenca_chagas_total,total_candidatos_inaptos_malaria_masculino,total_candidatos_inaptos_malaria_feminino,total_candidatos_inaptos_malaria_total,total_candidatos_inaptos_outras_masculino,total_candidatos_inaptos_outras_feminino,total_candidatos_inaptos_outras_total,coleta_total_candidatos_desistentes,total_interrupcoes_coleta_dificuldade_puncao_venosa,total_interrupcoes_coleta_reacao_vagal,total_interrupcoes_coleta_outros_motivos,total_coletas_sangue_total,total_coletas_aferese,hemoprod_1_observacoes,exames_triagem_doenca_doenca_chagas_amostras_testadas,exames_triagem_doenca_doenca_chagas_amostras_reagentes,exames_triagem_doenca_hiv_amostras_testadas,exames_triagem_doenca_hiv_amostras_reagentes,exames_triagem_doenca_sifilis_amostras_testadas,`exames_triagem_doenca_sifilis_amostras_reagentes,exames_triagem_doenca_hepatite_b_hbs_ag_amostras_testadas,exames_triagem_doenca_hepatite_b_hbs_ag_amostras_reagentes,exames_triagem_doenca_hepatite_b_anti_hbc_amostras_testadas,exames_triagem_doenca_hepatite_b_anti_hbc_amostras_reagentes,exames_triagem_doenca_hepatite_c_amostras_testadas,exames_triagem_doenca_hepatite_c_amostras_reagentes,exames_triagem_doenca_htlv_i_ii_amostras_testadas,exames_triagem_doenca_htlv_i_ii_amostras_reagentes,exames_triagem_doenca_malaria_amostras_testadas,exames_triagem_doenca_malaria_amostras_reagentes,exames_triagem_doenca_hbv_teste_nat_amostras_testadas,exames_triagem_doenca_hbv_teste_nat_amostras_reagentes,exames_triagem_doenca_hcv_teste_nat_amostras_testadas,exames_triagem_doenca_hcv_teste_nat_amostras_reagentes,exames_triagem_doenca_hiv_teste_nat_amostras_testadas,exames_triagem_doenca_hiv_teste_nat_amostras_reagentes,imunohematologia_a_positivo_doador,imunohematologia_a_positiv

In [17]:
# ASSUME QUE AS VARIÁVEIS 'hemoprod_al', 'dicionario', 'colunas_a_mais', 
# 'colunas_faltantes' E 'colunas_sql_desejadas' EXISTEM DA CÉLULA ANTERIOR.

print("--- 6. PADRONIZAÇÃO DO ESQUEMA (Remoção, Adição e Reordenação) ---")

# 6.1. REMOVE as Colunas a Mais
if colunas_a_mais:
    # Verificação de segurança: Só tenta dropar colunas que estão realmente no DF
    colunas_para_dropar = [col for col in colunas_a_mais if col in hemoprod_al.columns]
    hemoprod_am.drop(columns=colunas_para_dropar, inplace=True)
    print(f"✅ {len(colunas_para_dropar)} colunas a mais removidas.")
else:
    print("Nenhuma coluna a mais para remover.")

# 6.2. ADICIONA as Colunas Faltantes
if colunas_faltantes:
    
    # Prepara o sub-dicionário apenas com as colunas faltantes (para tipos)
    dicionario_faltante = dicionario[dicionario['nome_sql'].isin(colunas_faltantes)]
    
    for col_sql in colunas_faltantes:
        
        # Adiciona a coluna com valor nulo (NaN)
        hemoprod_am[col_sql] = np.nan 

        # --- Lógica Opcional de Conversão de Tipo (Baseada na coluna 'tipo_dados' no dicionário) ---
        # Tenta encontrar o tipo no dicionário (apenas se a coluna 'tipo_dados' existir e não estiver vazia)
        if 'tipo_dados' in dicionario.columns:
            tipo_desejado_row = dicionario_faltante[dicionario_faltante['nome_sql'] == col_sql]
            
            if not tipo_desejado_row.empty:
                tipo_desejado = tipo_desejado_row['tipo_dados'].iloc[0].lower()

                # Converte para o tipo, se for reconhecido
                if 'int' in tipo_desejado or 'float' in tipo_desejado:
                    # Usa 'float64' para números que podem ter NaNs, para evitar erros de Pandas
                    hemoprod_am[col_sql] = hemoprod_am[col_sql].astype('float64') 
                elif 'string' in tipo_desejado or 'object' in tipo_desejado:
                    # Converte para 'object' (string)
                    hemoprod_am[col_sql] = hemoprod_am[col_sql].astype('object')
                 
    print(f"✅ {len(colunas_faltantes)} colunas faltantes adicionadas e preenchidas com NaN.")
else:
    print("Nenhuma coluna faltante para adicionar.")


# 6.3. Reordena as colunas (Para garantir a ordem padronizada do dicionário)
colunas_finais = dicionario['nome_sql'].tolist()
# Garante que só as colunas que realmente existem no DF (após drop/adição) sejam usadas na reordenação
colunas_finais_presentes = [col for col in colunas_finais if col in hemoprod_am.columns]

# Esta é a linha que reordena o DataFrame
hemoprod_am = hemoprod_am[colunas_finais_presentes]
print("✅ Colunas reordenadas para seguir a ordem do dicionário.")


# --- 7. Verifique o resultado Final ---
print("\n--- Resultado Final ---")
print(f"Número de colunas final: {len(hemoprod_am.columns)}")
print("Informações do DataFrame padronizado:")
hemoprod_am.info()

print("\nAs 5 primeiras linhas com as colunas padronizadas:")
display(hemoprod_am.head())

--- 6. PADRONIZAÇÃO DO ESQUEMA (Remoção, Adição e Reordenação) ---
✅ 0 colunas a mais removidas.
Nenhuma coluna faltante para adicionar.
✅ Colunas reordenadas para seguir a ordem do dicionário.

--- Resultado Final ---
Número de colunas final: 269
Informações do DataFrame padronizado:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32 entries, 0 to 31
Columns: 269 entries, id to hemoprod_3_observacoes
dtypes: Int64(229), float64(27), object(13)
memory usage: 74.5+ KB

As 5 primeiras linhas com as colunas padronizadas:


,id,data_envio,ultima_pagina,idioma_inicial,semente,codigo_acesso,data_inicio,data_ultima_acao,ip,identificacao_dado,tipo_envio,ano_referencia,periodo_referencia,identificacao_estabelecimento,municipio,razao_social_nome_fantasia,razao_social_nome_fantasia_outros,cnpj,tipo_estabelecimento,natureza_estabelecimento,dados_informados_referem_se,rede_estabelecimento,cnes,endereco,triagem_clinica_total_doacao_espontanea_aptos,triagem_clinica_total_doacao_espontanea_inaptos,triagem_clinica_total_doacao_reposicao_aptos,triagem_clinica_total_doacao_reposicao_inaptos,triagem_clinica_total_doacao_autologa_aptos,triagem_clinica_total_doacao_autologa_inaptos,total_doador_primeira_vez_aptos,total_doador_primeira_vez_inaptos,total_doador_repeticao_aptos,total_doador_repeticao_inaptos,total_doador_esporadico_aptos,total_doador_esporadico_inaptos,total_doador_masculino_aptos,total_doador_masculino_inaptos,total_doador_feminino_aptos,total_doador_feminino_inaptos,total_doador_menor_de_18_anos_aptos,total_doador_menor_de_18_anos_inaptos,total_doador_18_ate_29_anos_aptos,total_doador_18_ate_29_anos_inaptos,total_doador_acima_de_29_anos_aptos,total_doador_acima_de_29_anos_inaptos,total_candidatos_inaptos_anemia_masculino,total_candidatos_inaptos_anemia_feminino,total_candidatos_inaptos_anemia_total,total_candidatos_inaptos_hipertensao_masculino,total_candidatos_inaptos_hipertensao_feminino,total_candidatos_inaptos_hipertensao_total,total_candidatos_inaptos_hipotensao_masculino,total_candidatos_inaptos_hipotensao_feminino,total_candidatos_inaptos_hipotensao_total,total_candidatos_inaptos_alcoolismo_masculino,total_candidatos_inaptos_alcoolismo_feminino,total_candidatos_inaptos_alcoolismo_total,total_candidatos_inaptos_comportamento_risco_dst_masculino,total_candidatos_inaptos_comportamento_risco_dst_feminino,total_candidatos_inaptos_comportamento_risco_dst_total,total_candidatos_inaptos_uso_drogas_masculino,total_candidatos_inaptos_uso_drogas_feminino,total_candidatos_inaptos_uso_drogas_total,total_candidatos_inaptos_hepatite_masculino,total_candidatos_inaptos_hepatite_feminino,total_candidatos_inaptos_hepatite_total,total_candidatos_inaptos_doenca_chagas_masculino,total_candidatos_inaptos_doenca_chagas_feminino,total_candidatos_inaptos_doenca_chagas_total,total_candidatos_inaptos_malaria_masculino,total_candidatos_inaptos_malaria_feminino,total_candidatos_inaptos_malaria_total,total_candidatos_inaptos_outras_masculino,total_candidatos_inaptos_outras_feminino,total_candidatos_inaptos_outras_total,coleta_total_candidatos_desistentes,total_interrupcoes_coleta_dificuldade_puncao_venosa,total_interrupcoes_coleta_reacao_vagal,total_interrupcoes_coleta_outros_motivos,total_coletas_sangue_total,total_coletas_aferese,hemoprod_1_observacoes,exames_triagem_doenca_doenca_chagas_amostras_testadas,exames_triagem_doenca_doenca_chagas_amostras_reagentes,exames_triagem_doenca_hiv_amostras_testadas,exames_triagem_doenca_hiv_amostras_reagentes,exames_triagem_doenca_sifilis_amostras_testadas,`exames_triagem_doenca_sifilis_amostras_reagentes,exames_triagem_doenca_hepatite_b_hbs_ag_amostras_testadas,exames_triagem_doenca_hepatite_b_hbs_ag_amostras_reagentes,exames_triagem_doenca_hepatite_b_anti_hbc_amostras_testadas,exames_triagem_doenca_hepatite_b_anti_hbc_amostras_reagentes,exames_triagem_doenca_hepatite_c_amostras_testadas,exames_triagem_doenca_hepatite_c_amostras_reagentes,exames_triagem_doenca_htlv_i_ii_amostras_testadas,exames_triagem_doenca_htlv_i_ii_amostras_reagentes,exames_triagem_doenca_malaria_amostras_testadas,exames_triagem_doenca_malaria_amostras_reagentes,exames_triagem_doenca_hbv_teste_nat_amostras_testadas,exames_triagem_doenca_hbv_teste_nat_amostras_reagentes,exames_triagem_doenca_hcv_teste_nat_amostras_testadas,exames_triagem_doenca_hcv_teste_nat_amostras_reagentes,exames_triagem_doenca_hiv_teste_nat_amostras_testadas,exames_triagem_doenca_hiv_teste_nat_amostras_reagentes,imunohematologia_a_positivo_doador,imunohematologia_a_positivo_receptor,imunohe

In [9]:
#remover o url
hemoprod_am = hemoprod_am.drop(columns=['url'])

In [11]:
# --- 2. Defina as colunas para a chave e para ordenação ---
registros_antes = len(hemoprod_am)
print(f"Total de registros ANTES da remoção de duplicatas: {registros_antes}")
    

colunas_chave = [
        'cnpj', 
        'ano_referencia', 
        'periodo_referencia', 
        'razao_social_nome_fantasia'
    ]

coluna_data = 'data_envio'

# --- 3. Verifique se as colunas necessárias existem ---
colunas_necessarias = colunas_chave + [coluna_data]

if not all(col in hemoprod_am.columns for col in colunas_necessarias):
    print("\n--- ERRO ---")
    print("Uma ou mais colunas necessárias para a deduplicação não foram encontradas.")
    colunas_faltantes = [col for col in colunas_necessarias if col not in hemoprod_am.columns]
    print(f"Colunas necessárias: {colunas_necessarias}")
    print(f"Colunas faltantes no DataFrame: {colunas_faltantes}")
else:
    # --- 4. Prepare a coluna de data e ordene os dados ---
    # Converte a coluna 'data_envio' para datetime para garantir a ordenação correta.
    # 'errors='coerce'' transformará datas inválidas em NaT (Not a Time), que são tratadas como nulas.
    hemoprod_am[coluna_data] = pd.to_datetime(hemoprod_am[coluna_data], errors='coerce')

    # Ordena o DataFrame. Os registros com data de envio mais recente ficarão por último.
    print(f"\nOrdenando os dados por '{coluna_data}'...")
    hemoprod_am_ordenado = hemoprod_am.sort_values(by=coluna_data, ascending=True)

    # --- 5. Identifique e separe os registros duplicados e únicos ---
    # Em vez de usar drop_duplicates() diretamente, vamos usar duplicated()
    # para criar uma máscara booleana.
    # 'keep='last'' marca todas as ocorrências de uma chave como True, EXCETO a última (a mais recente).
    print(f"Identificando duplicatas com base na chave: {colunas_chave}...")
    mascara_duplicatas = hemoprod_am_ordenado.duplicated(subset=colunas_chave, keep='last')

    # O DataFrame de removidos conterá todas as linhas marcadas como True
    hemoprod_removidos = hemoprod_am_ordenado[mascara_duplicatas]
    
    # O DataFrame deduplicado conterá o INVERSO (~) da máscara (linhas marcadas como False)
    hemoprod_am_deduplicado = hemoprod_am_ordenado[~mascara_duplicatas]
    
    # Agora podemos contar os registros diretamente dos novos DataFrames
    registros_depois = len(hemoprod_am_deduplicado)
    registros_removidos = len(hemoprod_removidos) 
    
    # --- 6. Exiba o resultado ---
    print("\n--- Processo Concluído ---")
    print(f"Registros removidos: {registros_removidos}")
    print(f"Total de registros DEPOIS da remoção de duplicatas: {registros_depois}")

    # (Opcional) Exibe a amostra dos removidos
    print("\nAmostra dos dados REMOVIDOS (os mais antigos/duplicados):")
    display(hemoprod_removidos.head(10))

    # Você pode continuar a usar o DataFrame 'hemoprod_am_deduplicado' para suas análises
    print("\nAmostra dos dados únicos (os mais recentes para cada chave):")
    display(hemoprod_am_deduplicado.head(10))

    hemoprod_am.describe()
    
    # Agora você tem o DataFrame 'hemoprod_removidos' salvo

Total de registros ANTES da remoção de duplicatas: 32

Ordenando os dados por 'data_envio'...
Identificando duplicatas com base na chave: ['cnpj', 'ano_referencia', 'periodo_referencia', 'razao_social_nome_fantasia']...

--- Processo Concluído ---
Registros removidos: 0
Total de registros DEPOIS da remoção de duplicatas: 32

Amostra dos dados REMOVIDOS (os mais antigos/duplicados):


,id,data_envio,ultima_pagina,idioma_inicial,semente,codigo_acesso,data_inicio,data_ultima_acao,ip,identificacao_dado,tipo_envio,ano_referencia,periodo_referencia,identificacao_estabelecimento,municipio,razao_social_nome_fantasia,razao_social_nome_fantasia_outros,cnpj,tipo_estabelecimento,natureza_estabelecimento,dados_informados_referem_se,rede_estabelecimento,cnes,endereco,triagem_clinica_total_doacao_espontanea_aptos,triagem_clinica_total_doacao_espontanea_inaptos,triagem_clinica_total_doacao_reposicao_aptos,triagem_clinica_total_doacao_reposicao_inaptos,triagem_clinica_total_doacao_autologa_aptos,triagem_clinica_total_doacao_autologa_inaptos,total_doador_primeira_vez_aptos,total_doador_primeira_vez_inaptos,total_doador_repeticao_aptos,total_doador_repeticao_inaptos,total_doador_esporadico_aptos,total_doador_esporadico_inaptos,total_doador_masculino_aptos,total_doador_masculino_inaptos,total_doador_feminino_aptos,total_doador_feminino_inaptos,total_doador_menor_de_18_anos_aptos,total_doador_menor_de_18_anos_inaptos,total_doador_18_ate_29_anos_aptos,total_doador_18_ate_29_anos_inaptos,total_doador_acima_de_29_anos_aptos,total_doador_acima_de_29_anos_inaptos,total_candidatos_inaptos_anemia_masculino,total_candidatos_inaptos_anemia_feminino,total_candidatos_inaptos_anemia_total,total_candidatos_inaptos_hipertensao_masculino,total_candidatos_inaptos_hipertensao_feminino,total_candidatos_inaptos_hipertensao_total,total_candidatos_inaptos_hipotensao_masculino,total_candidatos_inaptos_hipotensao_feminino,total_candidatos_inaptos_hipotensao_total,total_candidatos_inaptos_alcoolismo_masculino,total_candidatos_inaptos_alcoolismo_feminino,total_candidatos_inaptos_alcoolismo_total,total_candidatos_inaptos_comportamento_risco_dst_masculino,total_candidatos_inaptos_comportamento_risco_dst_feminino,total_candidatos_inaptos_comportamento_risco_dst_total,total_candidatos_inaptos_uso_drogas_masculino,total_candidatos_inaptos_uso_drogas_feminino,total_candidatos_inaptos_uso_drogas_total,total_candidatos_inaptos_hepatite_masculino,total_candidatos_inaptos_hepatite_feminino,total_candidatos_inaptos_hepatite_total,total_candidatos_inaptos_doenca_chagas_masculino,total_candidatos_inaptos_doenca_chagas_feminino,total_candidatos_inaptos_doenca_chagas_total,total_candidatos_inaptos_malaria_masculino,total_candidatos_inaptos_malaria_feminino,total_candidatos_inaptos_malaria_total,total_candidatos_inaptos_outras_masculino,total_candidatos_inaptos_outras_feminino,total_candidatos_inaptos_outras_total,coleta_total_candidatos_desistentes,total_interrupcoes_coleta_dificuldade_puncao_venosa,total_interrupcoes_coleta_reacao_vagal,total_interrupcoes_coleta_outros_motivos,total_coletas_sangue_total,total_coletas_aferese,hemoprod_1_observacoes,exames_triagem_doenca_doenca_chagas_amostras_testadas,exames_triagem_doenca_doenca_chagas_amostras_reagentes,exames_triagem_doenca_hiv_amostras_testadas,exames_triagem_doenca_hiv_amostras_reagentes,exames_triagem_doenca_sifilis_amostras_testadas,`exames_triagem_doenca_sifilis_amostras_reagentes,exames_triagem_doenca_hepatite_b_hbs_ag_amostras_testadas,exames_triagem_doenca_hepatite_b_hbs_ag_amostras_reagentes,exames_triagem_doenca_hepatite_b_anti_hbc_amostras_testadas,exames_triagem_doenca_hepatite_b_anti_hbc_amostras_reagentes,exames_triagem_doenca_hepatite_c_amostras_testadas,exames_triagem_doenca_hepatite_c_amostras_reagentes,exames_triagem_doenca_htlv_i_ii_amostras_testadas,exames_triagem_doenca_htlv_i_ii_amostras_reagentes,exames_triagem_doenca_malaria_amostras_testadas,exames_triagem_doenca_malaria_amostras_reagentes,exames_triagem_doenca_hbv_teste_nat_amostras_testadas,exames_triagem_doenca_hbv_teste_nat_amostras_reagentes,exames_triagem_doenca_hcv_teste_nat_amostras_testadas,exames_triagem_doenca_hcv_teste_nat_amostras_reagentes,exames_triagem_doenca_hiv_teste_nat_amostras_testadas,exames_triagem_doenca_hiv_teste_nat_amostras_reagentes,imunohematologia_a_positivo_doador,imunohematologia_a_positivo_receptor,imunohe


Amostra dos dados únicos (os mais recentes para cada chave):


,id,data_envio,ultima_pagina,idioma_inicial,semente,codigo_acesso,data_inicio,data_ultima_acao,ip,identificacao_dado,tipo_envio,ano_referencia,periodo_referencia,identificacao_estabelecimento,municipio,razao_social_nome_fantasia,razao_social_nome_fantasia_outros,cnpj,tipo_estabelecimento,natureza_estabelecimento,dados_informados_referem_se,rede_estabelecimento,cnes,endereco,triagem_clinica_total_doacao_espontanea_aptos,triagem_clinica_total_doacao_espontanea_inaptos,triagem_clinica_total_doacao_reposicao_aptos,triagem_clinica_total_doacao_reposicao_inaptos,triagem_clinica_total_doacao_autologa_aptos,triagem_clinica_total_doacao_autologa_inaptos,total_doador_primeira_vez_aptos,total_doador_primeira_vez_inaptos,total_doador_repeticao_aptos,total_doador_repeticao_inaptos,total_doador_esporadico_aptos,total_doador_esporadico_inaptos,total_doador_masculino_aptos,total_doador_masculino_inaptos,total_doador_feminino_aptos,total_doador_feminino_inaptos,total_doador_menor_de_18_anos_aptos,total_doador_menor_de_18_anos_inaptos,total_doador_18_ate_29_anos_aptos,total_doador_18_ate_29_anos_inaptos,total_doador_acima_de_29_anos_aptos,total_doador_acima_de_29_anos_inaptos,total_candidatos_inaptos_anemia_masculino,total_candidatos_inaptos_anemia_feminino,total_candidatos_inaptos_anemia_total,total_candidatos_inaptos_hipertensao_masculino,total_candidatos_inaptos_hipertensao_feminino,total_candidatos_inaptos_hipertensao_total,total_candidatos_inaptos_hipotensao_masculino,total_candidatos_inaptos_hipotensao_feminino,total_candidatos_inaptos_hipotensao_total,total_candidatos_inaptos_alcoolismo_masculino,total_candidatos_inaptos_alcoolismo_feminino,total_candidatos_inaptos_alcoolismo_total,total_candidatos_inaptos_comportamento_risco_dst_masculino,total_candidatos_inaptos_comportamento_risco_dst_feminino,total_candidatos_inaptos_comportamento_risco_dst_total,total_candidatos_inaptos_uso_drogas_masculino,total_candidatos_inaptos_uso_drogas_feminino,total_candidatos_inaptos_uso_drogas_total,total_candidatos_inaptos_hepatite_masculino,total_candidatos_inaptos_hepatite_feminino,total_candidatos_inaptos_hepatite_total,total_candidatos_inaptos_doenca_chagas_masculino,total_candidatos_inaptos_doenca_chagas_feminino,total_candidatos_inaptos_doenca_chagas_total,total_candidatos_inaptos_malaria_masculino,total_candidatos_inaptos_malaria_feminino,total_candidatos_inaptos_malaria_total,total_candidatos_inaptos_outras_masculino,total_candidatos_inaptos_outras_feminino,total_candidatos_inaptos_outras_total,coleta_total_candidatos_desistentes,total_interrupcoes_coleta_dificuldade_puncao_venosa,total_interrupcoes_coleta_reacao_vagal,total_interrupcoes_coleta_outros_motivos,total_coletas_sangue_total,total_coletas_aferese,hemoprod_1_observacoes,exames_triagem_doenca_doenca_chagas_amostras_testadas,exames_triagem_doenca_doenca_chagas_amostras_reagentes,exames_triagem_doenca_hiv_amostras_testadas,exames_triagem_doenca_hiv_amostras_reagentes,exames_triagem_doenca_sifilis_amostras_testadas,`exames_triagem_doenca_sifilis_amostras_reagentes,exames_triagem_doenca_hepatite_b_hbs_ag_amostras_testadas,exames_triagem_doenca_hepatite_b_hbs_ag_amostras_reagentes,exames_triagem_doenca_hepatite_b_anti_hbc_amostras_testadas,exames_triagem_doenca_hepatite_b_anti_hbc_amostras_reagentes,exames_triagem_doenca_hepatite_c_amostras_testadas,exames_triagem_doenca_hepatite_c_amostras_reagentes,exames_triagem_doenca_htlv_i_ii_amostras_testadas,exames_triagem_doenca_htlv_i_ii_amostras_reagentes,exames_triagem_doenca_malaria_amostras_testadas,exames_triagem_doenca_malaria_amostras_reagentes,exames_triagem_doenca_hbv_teste_nat_amostras_testadas,exames_triagem_doenca_hbv_teste_nat_amostras_reagentes,exames_triagem_doenca_hcv_teste_nat_amostras_testadas,exames_triagem_doenca_hcv_teste_nat_amostras_reagentes,exames_triagem_doenca_hiv_teste_nat_amostras_testadas,exames_triagem_doenca_hiv_teste_nat_amostras_reagentes,imunohematologia_a_positivo_doador,imunohematologia_a_positivo_receptor,imunohe

In [13]:
hemoprod_am.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32 entries, 0 to 31
Columns: 269 entries, id to hemoprod_3_observacoes
dtypes: Int64(229), datetime64[ns](1), float64(27), object(12)
memory usage: 74.5+ KB


In [14]:
hemoprod_am_deduplicado.to_excel('dados_processados/hemoprod_am.xlsx', index=False)

## Hemoprod Amapa

In [19]:
import pandas as pd
import os

# --- 1. Defina os caminhos para seus arquivos ---
# Ajuste os caminhos conforme a estrutura do seu projeto no notebook
dados_brutos_path = 'dados_brutos'
dicionario_path = 'dicionario_colunas_269.xlsx'
arquivo_dados_path = os.path.join(dados_brutos_path, 'Hemoprod_AP.xlsx')
nome_planilha = 'HEMOPROD - AMAPA'

try:
    # Carrega o arquivo de dados
    hemoprod_ap = pd.read_excel(arquivo_dados_path, sheet_name=nome_planilha)
    print("Arquivo de dados carregado com sucesso.")
    print(f"Número de colunas original: {len(hemoprod_ap.columns)}")

    # Carrega o arquivo de dicionário
    dicionario = pd.read_excel(dicionario_path)
    print("Arquivo de dicionário carregado com sucesso.")

    # --- 3. Limpeza Inicial dos Nomes das Colunas (ADICIONADO) ---
    # Mapeia as colunas, limpando espaços em branco no início e fim
    # e substituindo o caractere \xa0 (Non-breaking space) por um espaço normal.
    mapa_limpeza = {
        col: col.strip().replace('\xa0', ' ') 
        for col in hemoprod_ap.columns
    }

    # Aplica a renomeação para limpar os nomes
    hemoprod_ap = hemoprod_ap.rename(columns=mapa_limpeza)
    print("Nomes das colunas do DataFrame limpos e padronizados (strip/\\xa0).")

    # --- 4. Crie o dicionário de mapeamento ---
    # Cria um dicionário no formato {'nome_original': 'nome_sql'} para uso no .rename()
    # NOTA: O 'nome_original' no dicionário TAMBÉM deve estar limpo para combinar.
    # Se o dicionário não estiver limpo, adicione a mesma limpeza aqui:
    dicionario['nome_original'] = dicionario['nome_original'].astype(str).str.strip().str.replace('\xa0', ' ')
    
    mapa_renomeacao = pd.Series(dicionario['nome_sql'].values, index=dicionario['nome_original']).to_dict()
    
    # Lista dos nomes de coluna no DataFrame atual APÓS a limpeza inicial
    colunas_atuais = set(hemoprod_ap.columns)
    
    # Lista dos nomes originais de coluna no dicionário APÓS a limpeza (se aplicada)
    colunas_originais_dicionario = set(dicionario['nome_original'].tolist())
    
    # Lista dos nomes SQL (os desejados) no dicionário
    colunas_sql_desejadas = set(dicionario['nome_sql'].tolist())
    
    # --- 5. Renomeie as colunas com base no mapeamento (nome_original -> nome_sql) ---
    print("\n--- Processo de Renomeação ---")
    hemoprod_ap.rename(columns=mapa_renomeacao, inplace=True)
    print("Colunas renomeadas com sucesso (apenas as que existiam no dicionário foram modificadas).")


   # --- 4. Inferir Tipos e Criar Mapeamento de Tipos ---
    
    # --- Lógica de Inferência customizada para 'Int64' ---
    tipos_inferidos = {}
    for col in hemoprod_ap.columns:
        
        # 4.1. Tenta converter a coluna para o tipo que melhor representa seus dados (incluindo Int64/string/datetime)
        # O errors='ignore' é crucial para não falhar se a coluna não puder ser convertida
        coluna_convertida = pd.to_numeric(hemoprod_ap[col], errors='coerce')
        
        # Se a conversão for bem-sucedida (não é totalmente NaN, nem totalmente string)
        if not coluna_convertida.isna().all() and coluna_convertida.dtype.kind in 'fi': # 'f' para float, 'i' para int
            
            # Se a coluna parecer um número, tentamos forçar para Int64
            # O .astype('Int64') usa o tipo inteiro que suporta NaN
            try:
                # Se for possível converter para Int64, use 'Int64'
                hemoprod_ap[col] = hemoprod_ap[col].astype('Int64')
                tipos_inferidos[col] = 'Int64'
            except Exception:
                # Se for numérico mas não puder ser Int64 (ex: float com muitas casas decimais), use 'float64'
                tipos_inferidos[col] = 'float64'
        
        # Verifica se é data/hora
        elif pd.api.types.is_datetime64_any_dtype(hemoprod_ap[col]):
             tipos_inferidos[col] = 'datetime64[ns]'
             
        # Caso contrário, assume-se que é um texto/string
        else:
            tipos_inferidos[col] = 'object' # O padrão para string no pandas

    
    print("\nTipos de dados inferidos (amostra):")
    for i, (col, dtype) in enumerate(tipos_inferidos.items()):
        if i < 5:
            print(f"  {col}: {dtype}")
        if i == 5:
            print("  ...")
    
    # --- 5. Atualizar o DataFrame Dicionário ---
    
    # 5.1. Cria a coluna 'tipo_dados' no dicionário e preenche com os tipos inferidos
    dicionario['tipo_dados'] = dicionario['nome_sql'].map(tipos_inferidos)
    
    # 5.2. Trata colunas não encontradas no DataFrame de dados
    dicionario['tipo_dados'] = dicionario['tipo_dados'].fillna('object')
    
    print("\n--- Dicionário Atualizado ---")
    print("Coluna 'tipo_dados' criada com sucesso, com numéricos definidos como 'Int64' ou 'float64'.")
    
    # --- 6. Salvar o Dicionário Atualizado ---

    # Sugestão de novo caminho para salvar
    # novo_dicionario_path = 'dicionario_colunas_269_COM_TIPOS_V3.xlsx'
    
    # dicionario.to_excel(novo_dicionario_path, index=False)
    
    print(f"\n✅ Dicionário salvo com a nova coluna 'tipo_dados' em: {novo_dicionario_path}")
    print("\nPrimeiras linhas do dicionário atualizado:")
    display(dicionario.head())

    # --- 6. Análise de Colunas (Qualidade dos Dados) ---

    # 6.1. Colunas que não puderam ser renomeadas (existem no DF, mas não no 'nome_original' do dicionário)
    # Aqui usamos as colunas atuais ANTES da renomeação para ver o que sobrou.
    colunas_nao_mapeadas = [
        col for col in colunas_atuais 
        if col not in colunas_originais_dicionario
    ]

    print("-" * 30)
    print("Análise de Colunas do DataFrame (hemoprod_ap):")
    print(f"Número de colunas não mapeadas: {len(colunas_nao_mapeadas)}")
    print(f"Colunas não mapeadas (manterão nome original): {colunas_nao_mapeadas}")
    
    # 6.2. Análise de Colunas Faltantes/A Mais (Comparação com 'nome_sql' desejado)
    colunas_apos_renomeacao = set(hemoprod_ap.columns)
    
    colunas_faltantes = list(colunas_sql_desejadas - colunas_apos_renomeacao)
    colunas_a_mais = list(colunas_apos_renomeacao - colunas_sql_desejadas)

    print("-" * 30)
    print("Análise de Colunas (Comparação com a lista SQL DESEJADA):")
    print(f"Número de colunas FALTANTES: {len(colunas_faltantes)}")
    print(f"Colunas FALTANTES (deveriam estar, mas não estão): {colunas_faltantes}")
    print("-" * 30)
    print(f"Número de colunas A MAIS: {len(colunas_a_mais)}")
    print(f"Colunas A MAIS (estão no DF, mas não na lista SQL desejada): {colunas_a_mais}")
    print("-" * 30)

    # --- 7. Verifique o resultado ---
    print("\nInformações do DataFrame com as novas colunas:")
    hemoprod_ap.info()
    
    print("\nAs 5 primeiras linhas com as novas colunas:")
    display(hemoprod_ap.head())

except FileNotFoundError as e:
    print(f"\nErro de arquivo não encontrado: {e}")
except KeyError as e:
    print(f"\nErro de coluna não encontrada: {e}. Verifique se as colunas 'nome_original' e 'nome_sql' existem no seu arquivo de dicionário.")
except Exception as e:
    print(f"\nOcorreu um erro inesperado: {e}")

Arquivo de dados carregado com sucesso.
Número de colunas original: 269
Arquivo de dicionário carregado com sucesso.
Nomes das colunas do DataFrame limpos e padronizados (strip/\xa0).

--- Processo de Renomeação ---
Colunas renomeadas com sucesso (apenas as que existiam no dicionário foram modificadas).

Tipos de dados inferidos (amostra):
  id: Int64
  data_envio: object
  ultima_pagina: Int64
  idioma_inicial: object
  semente: Int64
  ...

--- Dicionário Atualizado ---
Coluna 'tipo_dados' criada com sucesso, com numéricos definidos como 'Int64' ou 'float64'.

✅ Dicionário salvo com a nova coluna 'tipo_dados' em: dicionario_colunas_269_COM_TIPOS_V2.xlsx

Primeiras linhas do dicionário atualizado:


,nome_original,nome_sql,comentario,tipo_dados
0,ID da resposta,id,Identificador único e chave primária da submis...,Int64
1,Data de envio,data_envio,Data e hora exatas do envio final do formulário.,object
2,Última página,ultima_pagina,Título da última página do formulário acessada...,Int64
3,Idioma inicial,idioma_inicial,Idioma selecionado no início do preenchimento ...,object
4,Semente,semente,Valor interno de semente/aleatorização (uso té...,Int64


------------------------------
Análise de Colunas do DataFrame (hemoprod_ap):
Número de colunas não mapeadas: 0
Colunas não mapeadas (manterão nome original): []
------------------------------
Análise de Colunas (Comparação com a lista SQL DESEJADA):
Número de colunas FALTANTES: 0
Colunas FALTANTES (deveriam estar, mas não estão): []
------------------------------
Número de colunas A MAIS: 0
Colunas A MAIS (estão no DF, mas não na lista SQL desejada): []
------------------------------

Informações do DataFrame com as novas colunas:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1 entries, 0 to 0
Columns: 269 entries, id to hemoprod_3_observacoes
dtypes: Int64(247), float64(8), object(14)
memory usage: 2.5+ KB

As 5 primeiras linhas com as novas colunas:


,id,data_envio,ultima_pagina,idioma_inicial,semente,codigo_acesso,data_inicio,data_ultima_acao,ip,identificacao_dado,tipo_envio,ano_referencia,periodo_referencia,identificacao_estabelecimento,municipio,razao_social_nome_fantasia,razao_social_nome_fantasia_outros,cnpj,tipo_estabelecimento,natureza_estabelecimento,dados_informados_referem_se,rede_estabelecimento,cnes,endereco,triagem_clinica_total_doacao_espontanea_aptos,triagem_clinica_total_doacao_espontanea_inaptos,triagem_clinica_total_doacao_reposicao_aptos,triagem_clinica_total_doacao_reposicao_inaptos,triagem_clinica_total_doacao_autologa_aptos,triagem_clinica_total_doacao_autologa_inaptos,total_doador_primeira_vez_aptos,total_doador_primeira_vez_inaptos,total_doador_repeticao_aptos,total_doador_repeticao_inaptos,total_doador_esporadico_aptos,total_doador_esporadico_inaptos,total_doador_masculino_aptos,total_doador_masculino_inaptos,total_doador_feminino_aptos,total_doador_feminino_inaptos,total_doador_menor_de_18_anos_aptos,total_doador_menor_de_18_anos_inaptos,total_doador_18_ate_29_anos_aptos,total_doador_18_ate_29_anos_inaptos,total_doador_acima_de_29_anos_aptos,total_doador_acima_de_29_anos_inaptos,total_candidatos_inaptos_anemia_masculino,total_candidatos_inaptos_anemia_feminino,total_candidatos_inaptos_anemia_total,total_candidatos_inaptos_hipertensao_masculino,total_candidatos_inaptos_hipertensao_feminino,total_candidatos_inaptos_hipertensao_total,total_candidatos_inaptos_hipotensao_masculino,total_candidatos_inaptos_hipotensao_feminino,total_candidatos_inaptos_hipotensao_total,total_candidatos_inaptos_alcoolismo_masculino,total_candidatos_inaptos_alcoolismo_feminino,total_candidatos_inaptos_alcoolismo_total,total_candidatos_inaptos_comportamento_risco_dst_masculino,total_candidatos_inaptos_comportamento_risco_dst_feminino,total_candidatos_inaptos_comportamento_risco_dst_total,total_candidatos_inaptos_uso_drogas_masculino,total_candidatos_inaptos_uso_drogas_feminino,total_candidatos_inaptos_uso_drogas_total,total_candidatos_inaptos_hepatite_masculino,total_candidatos_inaptos_hepatite_feminino,total_candidatos_inaptos_hepatite_total,total_candidatos_inaptos_doenca_chagas_masculino,total_candidatos_inaptos_doenca_chagas_feminino,total_candidatos_inaptos_doenca_chagas_total,total_candidatos_inaptos_malaria_masculino,total_candidatos_inaptos_malaria_feminino,total_candidatos_inaptos_malaria_total,total_candidatos_inaptos_outras_masculino,total_candidatos_inaptos_outras_feminino,total_candidatos_inaptos_outras_total,coleta_total_candidatos_desistentes,total_interrupcoes_coleta_dificuldade_puncao_venosa,total_interrupcoes_coleta_reacao_vagal,total_interrupcoes_coleta_outros_motivos,total_coletas_sangue_total,total_coletas_aferese,hemoprod_1_observacoes,exames_triagem_doenca_doenca_chagas_amostras_testadas,exames_triagem_doenca_doenca_chagas_amostras_reagentes,exames_triagem_doenca_hiv_amostras_testadas,exames_triagem_doenca_hiv_amostras_reagentes,exames_triagem_doenca_sifilis_amostras_testadas,`exames_triagem_doenca_sifilis_amostras_reagentes,exames_triagem_doenca_hepatite_b_hbs_ag_amostras_testadas,exames_triagem_doenca_hepatite_b_hbs_ag_amostras_reagentes,exames_triagem_doenca_hepatite_b_anti_hbc_amostras_testadas,exames_triagem_doenca_hepatite_b_anti_hbc_amostras_reagentes,exames_triagem_doenca_hepatite_c_amostras_testadas,exames_triagem_doenca_hepatite_c_amostras_reagentes,exames_triagem_doenca_htlv_i_ii_amostras_testadas,exames_triagem_doenca_htlv_i_ii_amostras_reagentes,exames_triagem_doenca_malaria_amostras_testadas,exames_triagem_doenca_malaria_amostras_reagentes,exames_triagem_doenca_hbv_teste_nat_amostras_testadas,exames_triagem_doenca_hbv_teste_nat_amostras_reagentes,exames_triagem_doenca_hcv_teste_nat_amostras_testadas,exames_triagem_doenca_hcv_teste_nat_amostras_reagentes,exames_triagem_doenca_hiv_teste_nat_amostras_testadas,exames_triagem_doenca_hiv_teste_nat_amostras_reagentes,imunohematologia_a_positivo_doador,imunohematologia_a_positivo_receptor,imunohe

In [ ]:
# hemoprod_ap.head()

,ID da resposta,Data de envio,Última página,Idioma inicial,Semente,Código de acesso,Data de início,Data da última ação,Endereço IP,IDENTIFICAÇÃO DO DADO,"Tipo de Informação Antes de responder ao formulário, declare o tipo de informação que será inserida.",Ano de referência,Período de referência,IDENTIFICAÇÃO DO ESTABELECIMENTO,Município,Razão Social - Nome Fantasia,Razão Social - Nome Fantasia [Outros],CNPJ,Tipo de estabelecimento,Natureza do estabelecimento,Os dados informados referem-se à um(a):,"Cite os estabelecimentos que compõem a rede Informe o Tipo de Estabelecimento, o Nome Fantasia e o Município de localização de cada um.",CNES - Cadastro Nacional de Estabelecimentos de Saúde,Endereço,2. Triagem Clínica 2.1 Total de candidatos quanto ao tipo de doação [Espontânea][Aptos],2. Triagem Clínica 2.1 Total de candidatos quanto ao tipo de doação [Espontânea][Inaptos],2. Triagem Clínica 2.1 Total de candidatos quanto ao tipo de doação [Reposição][Aptos],2. Triagem Clínica 2.1 Total de candidatos quanto ao tipo de doação [Reposição][Inaptos],2. Triagem Clínica 2.1 Total de candidatos quanto ao tipo de doação [Autóloga][Aptos],2. Triagem Clínica 2.1 Total de candidatos quanto ao tipo de doação [Autóloga][Inaptos],2.2 Total de candidatos quanto ao tipo de doador [Primeira vez][Aptos],2.2 Total de candidatos quanto ao tipo de doador [Primeira vez][Inaptos],2.2 Total de candidatos quanto ao tipo de doador [Repetição][Aptos],2.2 Total de candidatos quanto ao tipo de doador [Repetição][Inaptos],2.2 Total de candidatos quanto ao tipo de doador [Esporádico][Aptos],2.2 Total de candidatos quanto ao tipo de doador [Esporádico][Inaptos],2.3 Total de candidatos quanto ao gênero do doador [Masculino][Aptos],2.3 Total de candidatos quanto ao gênero do doador [Masculino][Inaptos],2.3 Total de candidatos quanto ao gênero do doador [Feminino][Aptos],2.3 Total de candidatos quanto ao gênero do doador [Feminino][Inaptos],2.4 Total de candidatos quanto a idade do doador [Menor de 18 anos][Aptos],2.4 Total de candidatos quanto a idade do doador [Menor de 18 anos][Inaptos],2.4 Total de candidatos quanto a idade do doador [18 até 29 anos][Aptos],2.4 Total de candidatos quanto a idade do doador [18 até 29 anos][Inaptos],2.4 Total de candidatos quanto a idade do doador [Acima de 29 anos][Aptos],2.4 Total de candidatos quanto a idade do doador [Acima de 29 anos][Inaptos],2.5 Total de candidatos inaptos por motivo de inaptidão e por gênero [Anemia][Masculino],2.5 Total de candidatos inaptos por motivo de inaptidão e por gênero [Anemia][Feminino],2.5 Total de candidatos inaptos por motivo de inaptidão e por gênero [Anemia][Total],2.5 Total de candidatos inaptos por motivo de inaptidão e por gênero [Hipertensão][Masculino],2.5 Total de candidatos inaptos por motivo de inaptidão e por gênero [Hipertensão][Feminino],2.5 Total de candidatos inaptos por motivo de inaptidão e por gênero [Hipertensão][Total],2.5 Total de candidatos inaptos por motivo de inaptidão e por gênero [Hipotensão][Masculino],2.5 Total de candidatos inaptos por motivo de inaptidão e por gênero [Hipotensão][Feminino],2.5 Total de candidatos inaptos por motivo de inaptidão e por gênero [Hipotensão][Total],2.5 Total de candidatos inaptos por motivo de inaptidão e por gênero [Alcoolismo][Masculino],2.5 Total de candidatos inaptos por motivo de inaptidão e por gênero [Alcoolismo][Feminino],2.5 Total de candidatos inaptos por motivo de inaptidão e por gênero [Alcoolismo][Total],2.5 Total de candidatos inaptos por motivo de inaptidão e por gênero [Comportamento de risco para DST][Masculino],2.5 Total de candidatos inaptos por motivo de inaptidão e por gênero [Comportamento de risco para DST][Feminino],2.5 Total de candidatos inaptos por motivo de inaptidão e por gênero [Comportamento de risco para DST][Total],2.5 Total de candidatos inaptos por motivo de inaptidão e por gênero [Uso de drogas][Masculino],2.5 Total de candidatos inaptos por motivo de inaptidão e por gênero [Uso de drogas][Feminin

In [ ]:
# print("--- Nomes das Colunas (um por linha) ---")
# for coluna in hemoprod_ap.columns:
#     print(coluna + ",") 

--- Nomes das Colunas (um por linha) ---
ID da resposta,
Data de envio,
Última página,
Idioma inicial,
Semente,
Código de acesso,
Data de início,
Data da última ação,
Endereço IP,
IDENTIFICAÇÃO DO DADO ,
Tipo de Informação  Antes de responder ao formulário, declare o tipo de informação que será inserida.  ,
Ano de referência ,
Período de referência,
IDENTIFICAÇÃO DO ESTABELECIMENTO ,
Município,
Razão Social - Nome Fantasia ,
Razão Social - Nome Fantasia  [Outros],
CNPJ,
Tipo de estabelecimento,
Natureza do estabelecimento,
Os dados informados referem-se à um(a): ,
Cite os estabelecimentos que compõem a rede  Informe o Tipo de Estabelecimento, o Nome Fantasia e o Município de localização de cada um. ,
CNES - Cadastro Nacional de Estabelecimentos de Saúde ,
Endereço,
2. Triagem Clínica  2.1 Total de candidatos quanto ao tipo de doação  [Espontânea][Aptos],
2. Triagem Clínica  2.1 Total de candidatos quanto ao tipo de doação  [Espontânea][Inaptos],
2. Triagem Clínica  2.1 Total de candida

In [18]:
import os
import pandas as pd

dados_brutos_path = 'dados_brutos'

arquivo_dados_path = os.path.join(dados_brutos_path, 'Hemoprod_AP.xlsx')
nome_planilha = 'HEMOPROD - AMAPA'

dicionario_path_ap = ('./dicionario_colunas_269.xlsx')
# dicionario_path_ap = ('./dicionario_colunas_270.xlsx')

dicionario_ap = pd.read_excel(dicionario_path_ap, sheet_name='Sheet1')

# --- 2. Carregue os dados e o dicionário ---
try:
    # Carrega o arquivo de dados
    hemoprod_ap = pd.read_excel(arquivo_dados_path, sheet_name=nome_planilha)
    print("Arquivo de dados carregado com sucesso.")
    print(f"Número de colunas original: {len(hemoprod_ap.columns)}")

    # Carrega o arquivo de dicionário
    dicionario = pd.read_excel(dicionario_path_ap)
    print("Arquivo de dicionário carregado com sucesso.")

    # --- 3. Extraia a lista de novos nomes ---
    # Pega os valores da coluna 'nome_sql' e converte para uma lista
    novos_nomes = dicionario['nome_sql'].tolist()
    print(f"Número de novos nomes no dicionário: {len(novos_nomes)}")

    # --- 4. Verificação de segurança (MUITO IMPORTANTE) ---
    # Garante que o número de colunas é o mesmo antes de renomear
    if len(hemoprod_ap.columns) == len(novos_nomes):
        print("\nO número de colunas corresponde. Renomeando...")
        
        # --- 5. Substitua os nomes das colunas ---
        # Esta é a linha principal que faz a substituição direta
        hemoprod_ap.columns = novos_nomes
        
        print("Colunas renomeadas com sucesso!")
        
        # --- 6. Verifique o resultado ---
        print("\nInformações do DataFrame com as novas colunas:")
        hemoprod_ap.info()
        
        print("\nAs 5 primeiras linhas com as novas colunas:")
        display(hemoprod_ap.head())

    else:
        # Mensagem de erro se o número de colunas for diferente
        print("\n--- ERRO ---")
        print("A renomeação foi cancelada. O número de colunas no arquivo de dados não é igual ao número de nomes no dicionário.")
        print(f"Colunas no arquivo de dados: {len(hemoprod_ap.columns)}")
        print(f"Nomes no dicionário: {len(novos_nomes)}")

except FileNotFoundError as e:
    print(f"\nErro de arquivo não encontrado: {e}")
except KeyError as e:
    print(f"\nErro de coluna não encontrada: {e}. Verifique se a coluna 'nome_sql' existe no seu arquivo de dicionário.")
except Exception as e:
    print(f"\nOcorreu um erro inesperado: {e}")



Arquivo de dados carregado com sucesso.
Número de colunas original: 269
Arquivo de dicionário carregado com sucesso.
Número de novos nomes no dicionário: 269

O número de colunas corresponde. Renomeando...
Colunas renomeadas com sucesso!

Informações do DataFrame com as novas colunas:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1 entries, 0 to 0
Columns: 269 entries, id to hemoprod_3_observacoes
dtypes: float64(8), int64(247), object(14)
memory usage: 2.2+ KB

As 5 primeiras linhas com as novas colunas:


,id,data_envio,ultima_pagina,idioma_inicial,semente,codigo_acesso,data_inicio,data_ultima_acao,ip,identificacao_dado,tipo_envio,ano_referencia,periodo_referencia,identificacao_estabelecimento,municipio,razao_social_nome_fantasia,razao_social_nome_fantasia_outros,cnpj,tipo_estabelecimento,natureza_estabelecimento,dados_informados_referem_se,rede_estabelecimento,cnes,endereco,triagem_clinica_total_doacao_espontanea_aptos,triagem_clinica_total_doacao_espontanea_inaptos,triagem_clinica_total_doacao_reposicao_aptos,triagem_clinica_total_doacao_reposicao_inaptos,triagem_clinica_total_doacao_autologa_aptos,triagem_clinica_total_doacao_autologa_inaptos,total_doador_primeira_vez_aptos,total_doador_primeira_vez_inaptos,total_doador_repeticao_aptos,total_doador_repeticao_inaptos,total_doador_esporadico_aptos,total_doador_esporadico_inaptos,total_doador_masculino_aptos,total_doador_masculino_inaptos,total_doador_feminino_aptos,total_doador_feminino_inaptos,total_doador_menor_de_18_anos_aptos,total_doador_menor_de_18_anos_inaptos,total_doador_18_ate_29_anos_aptos,total_doador_18_ate_29_anos_inaptos,total_doador_acima_de_29_anos_aptos,total_doador_acima_de_29_anos_inaptos,total_candidatos_inaptos_anemia_masculino,total_candidatos_inaptos_anemia_feminino,total_candidatos_inaptos_anemia_total,total_candidatos_inaptos_hipertensao_masculino,total_candidatos_inaptos_hipertensao_feminino,total_candidatos_inaptos_hipertensao_total,total_candidatos_inaptos_hipotensao_masculino,total_candidatos_inaptos_hipotensao_feminino,total_candidatos_inaptos_hipotensao_total,total_candidatos_inaptos_alcoolismo_masculino,total_candidatos_inaptos_alcoolismo_feminino,total_candidatos_inaptos_alcoolismo_total,total_candidatos_inaptos_comportamento_risco_dst_masculino,total_candidatos_inaptos_comportamento_risco_dst_feminino,total_candidatos_inaptos_comportamento_risco_dst_total,total_candidatos_inaptos_uso_drogas_masculino,total_candidatos_inaptos_uso_drogas_feminino,total_candidatos_inaptos_uso_drogas_total,total_candidatos_inaptos_hepatite_masculino,total_candidatos_inaptos_hepatite_feminino,total_candidatos_inaptos_hepatite_total,total_candidatos_inaptos_doenca_chagas_masculino,total_candidatos_inaptos_doenca_chagas_feminino,total_candidatos_inaptos_doenca_chagas_total,total_candidatos_inaptos_malaria_masculino,total_candidatos_inaptos_malaria_feminino,total_candidatos_inaptos_malaria_total,total_candidatos_inaptos_outras_masculino,total_candidatos_inaptos_outras_feminino,total_candidatos_inaptos_outras_total,coleta_total_candidatos_desistentes,total_interrupcoes_coleta_dificuldade_puncao_venosa,total_interrupcoes_coleta_reacao_vagal,total_interrupcoes_coleta_outros_motivos,total_coletas_sangue_total,total_coletas_aferese,hemoprod_1_observacoes,exames_triagem_doenca_doenca_chagas_amostras_testadas,exames_triagem_doenca_doenca_chagas_amostras_reagentes,exames_triagem_doenca_hiv_amostras_testadas,exames_triagem_doenca_hiv_amostras_reagentes,exames_triagem_doenca_sifilis_amostras_testadas,`exames_triagem_doenca_sifilis_amostras_reagentes,exames_triagem_doenca_hepatite_b_hbs_ag_amostras_testadas,exames_triagem_doenca_hepatite_b_hbs_ag_amostras_reagentes,exames_triagem_doenca_hepatite_b_anti_hbc_amostras_testadas,exames_triagem_doenca_hepatite_b_anti_hbc_amostras_reagentes,exames_triagem_doenca_hepatite_c_amostras_testadas,exames_triagem_doenca_hepatite_c_amostras_reagentes,exames_triagem_doenca_htlv_i_ii_amostras_testadas,exames_triagem_doenca_htlv_i_ii_amostras_reagentes,exames_triagem_doenca_malaria_amostras_testadas,exames_triagem_doenca_malaria_amostras_reagentes,exames_triagem_doenca_hbv_teste_nat_amostras_testadas,exames_triagem_doenca_hbv_teste_nat_amostras_reagentes,exames_triagem_doenca_hcv_teste_nat_amostras_testadas,exames_triagem_doenca_hcv_teste_nat_amostras_reagentes,exames_triagem_doenca_hiv_teste_nat_amostras_testadas,exames_triagem_doenca_hiv_teste_nat_amostras_reagentes,imunohematologia_a_positivo_doador,imunohematologia_a_positivo_receptor,imunohe

In [26]:
# --- 2. Defina as colunas para a chave e para ordenação ---
registros_antes = len(hemoprod_ap)
print(f"Total de registros ANTES da remoção de duplicatas: {registros_antes}")
    

colunas_chave = [
        'cnpj', 
        'ano_referencia', 
        'periodo_referencia', 
        'razao_social_nome_fantasia'
    ]

coluna_data = 'data_envio'

# --- 3. Verifique se as colunas necessárias existem ---
colunas_necessarias = colunas_chave + [coluna_data]

if not all(col in hemoprod_ap.columns for col in colunas_necessarias):
    print("\n--- ERRO ---")
    print("Uma ou mais colunas necessárias para a deduplicação não foram encontradas.")
    colunas_faltantes = [col for col in colunas_necessarias if col not in hemoprod_ap.columns]
    print(f"Colunas necessárias: {colunas_necessarias}")
    print(f"Colunas faltantes no DataFrame: {colunas_faltantes}")
else:
    # --- 4. Prepare a coluna de data e ordene os dados ---
    # Converte a coluna 'data_envio' para datetime para garantir a ordenação correta.
    # 'errors='coerce'' transformará datas inválidas em NaT (Not a Time), que são tratadas como nulas.
    hemoprod_ap[coluna_data] = pd.to_datetime(hemoprod_ap[coluna_data], errors='coerce')

    # Ordena o DataFrame. Os registros com data de envio mais recente ficarão por último.
    print(f"\nOrdenando os dados por '{coluna_data}'...")
    hemoprod_ap_ordenado = hemoprod_ap.sort_values(by=coluna_data, ascending=True)

    # --- 5. Identifique e separe os registros duplicados e únicos ---
    # Em vez de usar drop_duplicates() diretamente, vamos usar duplicated()
    # para criar uma máscara booleana.
    # 'keep='last'' marca todas as ocorrências de uma chave como True, EXCETO a última (a mais recente).
    print(f"Identificando duplicatas com base na chave: {colunas_chave}...")
    mascara_duplicatas = hemoprod_ap_ordenado.duplicated(subset=colunas_chave, keep='last')

    # O DataFrame de removidos conterá todas as linhas marcadas como True
    hemoprod_removidos = hemoprod_ap_ordenado[mascara_duplicatas]
    
    # O DataFrame deduplicado conterá o INVERSO (~) da máscara (linhas marcadas como False)
    hemoprod_ap_deduplicado = hemoprod_ap_ordenado[~mascara_duplicatas]
    
    # Agora podemos contar os registros diretamente dos novos DataFrames
    registros_depois = len(hemoprod_ap_deduplicado)
    registros_removidos = len(hemoprod_removidos) 
    
    # --- 6. Exiba o resultado ---
    print("\n--- Processo Concluído ---")
    print(f"Registros removidos: {registros_removidos}")
    print(f"Total de registros DEPOIS da remoção de duplicatas: {registros_depois}")

    # (Opcional) Exibe a amostra dos removidos
    print("\nAmostra dos dados REMOVIDOS (os mais antigos/duplicados):")
    display(hemoprod_removidos.head(10))

    # Você pode continuar a usar o DataFrame 'hemoprod_ap_deduplicado' para suas análises
    print("\nAmostra dos dados únicos (os mais recentes para cada chave):")
    display(hemoprod_ap_deduplicado.head(10))
    
    # Agora você tem o DataFrame 'hemoprod_removidos' salvo

Total de registros ANTES da remoção de duplicatas: 1

Ordenando os dados por 'data_envio'...
Identificando duplicatas com base na chave: ['cnpj', 'ano_referencia', 'periodo_referencia', 'razao_social_nome_fantasia']...

--- Processo Concluído ---
Registros removidos: 0
Total de registros DEPOIS da remoção de duplicatas: 1

Amostra dos dados REMOVIDOS (os mais antigos/duplicados):


,id,data_envio,ultima_pagina,idioma_inicial,semente,codigo_acesso,data_inicio,data_ultima_acao,ip,identificacao_dado,tipo_envio,ano_referencia,periodo_referencia,identificacao_estabelecimento,municipio,razao_social_nome_fantasia,razao_social_nome_fantasia_outros,cnpj,tipo_estabelecimento,natureza_estabelecimento,dados_informados_referem_se,rede_estabelecimento,cnes,endereco,triagem_clinica_total_doacao_espontanea_aptos,triagem_clinica_total_doacao_espontanea_inaptos,triagem_clinica_total_doacao_reposicao_aptos,triagem_clinica_total_doacao_reposicao_inaptos,triagem_clinica_total_doacao_autologa_aptos,triagem_clinica_total_doacao_autologa_inaptos,total_doador_primeira_vez_aptos,total_doador_primeira_vez_inaptos,total_doador_repeticao_aptos,total_doador_repeticao_inaptos,total_doador_esporadico_aptos,total_doador_esporadico_inaptos,total_doador_masculino_aptos,total_doador_masculino_inaptos,total_doador_feminino_aptos,total_doador_feminino_inaptos,total_doador_menor_de_18_anos_aptos,total_doador_menor_de_18_anos_inaptos,total_doador_18_ate_29_anos_aptos,total_doador_18_ate_29_anos_inaptos,total_doador_acima_de_29_anos_aptos,total_doador_acima_de_29_anos_inaptos,total_candidatos_inaptos_anemia_masculino,total_candidatos_inaptos_anemia_feminino,total_candidatos_inaptos_anemia_total,total_candidatos_inaptos_hipertensao_masculino,total_candidatos_inaptos_hipertensao_feminino,total_candidatos_inaptos_hipertensao_total,total_candidatos_inaptos_hipotensao_masculino,total_candidatos_inaptos_hipotensao_feminino,total_candidatos_inaptos_hipotensao_total,total_candidatos_inaptos_alcoolismo_masculino,total_candidatos_inaptos_alcoolismo_feminino,total_candidatos_inaptos_alcoolismo_total,total_candidatos_inaptos_comportamento_risco_dst_masculino,total_candidatos_inaptos_comportamento_risco_dst_feminino,total_candidatos_inaptos_comportamento_risco_dst_total,total_candidatos_inaptos_uso_drogas_masculino,total_candidatos_inaptos_uso_drogas_feminino,total_candidatos_inaptos_uso_drogas_total,total_candidatos_inaptos_hepatite_masculino,total_candidatos_inaptos_hepatite_feminino,total_candidatos_inaptos_hepatite_total,total_candidatos_inaptos_doenca_chagas_masculino,total_candidatos_inaptos_doenca_chagas_feminino,total_candidatos_inaptos_doenca_chagas_total,total_candidatos_inaptos_malaria_masculino,total_candidatos_inaptos_malaria_feminino,total_candidatos_inaptos_malaria_total,total_candidatos_inaptos_outras_masculino,total_candidatos_inaptos_outras_feminino,total_candidatos_inaptos_outras_total,coleta_total_candidatos_desistentes,total_interrupcoes_coleta_dificuldade_puncao_venosa,total_interrupcoes_coleta_reacao_vagal,total_interrupcoes_coleta_outros_motivos,total_coletas_sangue_total,total_coletas_aferese,hemoprod_1_observacoes,exames_triagem_doenca_doenca_chagas_amostras_testadas,exames_triagem_doenca_doenca_chagas_amostras_reagentes,exames_triagem_doenca_hiv_amostras_testadas,exames_triagem_doenca_hiv_amostras_reagentes,exames_triagem_doenca_sifilis_amostras_testadas,`exames_triagem_doenca_sifilis_amostras_reagentes,exames_triagem_doenca_hepatite_b_hbs_ag_amostras_testadas,exames_triagem_doenca_hepatite_b_hbs_ag_amostras_reagentes,exames_triagem_doenca_hepatite_b_anti_hbc_amostras_testadas,exames_triagem_doenca_hepatite_b_anti_hbc_amostras_reagentes,exames_triagem_doenca_hepatite_c_amostras_testadas,exames_triagem_doenca_hepatite_c_amostras_reagentes,exames_triagem_doenca_htlv_i_ii_amostras_testadas,exames_triagem_doenca_htlv_i_ii_amostras_reagentes,exames_triagem_doenca_malaria_amostras_testadas,exames_triagem_doenca_malaria_amostras_reagentes,exames_triagem_doenca_hbv_teste_nat_amostras_testadas,exames_triagem_doenca_hbv_teste_nat_amostras_reagentes,exames_triagem_doenca_hcv_teste_nat_amostras_testadas,exames_triagem_doenca_hcv_teste_nat_amostras_reagentes,exames_triagem_doenca_hiv_teste_nat_amostras_testadas,exames_triagem_doenca_hiv_teste_nat_amostras_reagentes,imunohematologia_a_positivo_doador,imunohematologia_a_positivo_receptor,imunohe


Amostra dos dados únicos (os mais recentes para cada chave):


,id,data_envio,ultima_pagina,idioma_inicial,semente,codigo_acesso,data_inicio,data_ultima_acao,ip,identificacao_dado,tipo_envio,ano_referencia,periodo_referencia,identificacao_estabelecimento,municipio,razao_social_nome_fantasia,razao_social_nome_fantasia_outros,cnpj,tipo_estabelecimento,natureza_estabelecimento,dados_informados_referem_se,rede_estabelecimento,cnes,endereco,triagem_clinica_total_doacao_espontanea_aptos,triagem_clinica_total_doacao_espontanea_inaptos,triagem_clinica_total_doacao_reposicao_aptos,triagem_clinica_total_doacao_reposicao_inaptos,triagem_clinica_total_doacao_autologa_aptos,triagem_clinica_total_doacao_autologa_inaptos,total_doador_primeira_vez_aptos,total_doador_primeira_vez_inaptos,total_doador_repeticao_aptos,total_doador_repeticao_inaptos,total_doador_esporadico_aptos,total_doador_esporadico_inaptos,total_doador_masculino_aptos,total_doador_masculino_inaptos,total_doador_feminino_aptos,total_doador_feminino_inaptos,total_doador_menor_de_18_anos_aptos,total_doador_menor_de_18_anos_inaptos,total_doador_18_ate_29_anos_aptos,total_doador_18_ate_29_anos_inaptos,total_doador_acima_de_29_anos_aptos,total_doador_acima_de_29_anos_inaptos,total_candidatos_inaptos_anemia_masculino,total_candidatos_inaptos_anemia_feminino,total_candidatos_inaptos_anemia_total,total_candidatos_inaptos_hipertensao_masculino,total_candidatos_inaptos_hipertensao_feminino,total_candidatos_inaptos_hipertensao_total,total_candidatos_inaptos_hipotensao_masculino,total_candidatos_inaptos_hipotensao_feminino,total_candidatos_inaptos_hipotensao_total,total_candidatos_inaptos_alcoolismo_masculino,total_candidatos_inaptos_alcoolismo_feminino,total_candidatos_inaptos_alcoolismo_total,total_candidatos_inaptos_comportamento_risco_dst_masculino,total_candidatos_inaptos_comportamento_risco_dst_feminino,total_candidatos_inaptos_comportamento_risco_dst_total,total_candidatos_inaptos_uso_drogas_masculino,total_candidatos_inaptos_uso_drogas_feminino,total_candidatos_inaptos_uso_drogas_total,total_candidatos_inaptos_hepatite_masculino,total_candidatos_inaptos_hepatite_feminino,total_candidatos_inaptos_hepatite_total,total_candidatos_inaptos_doenca_chagas_masculino,total_candidatos_inaptos_doenca_chagas_feminino,total_candidatos_inaptos_doenca_chagas_total,total_candidatos_inaptos_malaria_masculino,total_candidatos_inaptos_malaria_feminino,total_candidatos_inaptos_malaria_total,total_candidatos_inaptos_outras_masculino,total_candidatos_inaptos_outras_feminino,total_candidatos_inaptos_outras_total,coleta_total_candidatos_desistentes,total_interrupcoes_coleta_dificuldade_puncao_venosa,total_interrupcoes_coleta_reacao_vagal,total_interrupcoes_coleta_outros_motivos,total_coletas_sangue_total,total_coletas_aferese,hemoprod_1_observacoes,exames_triagem_doenca_doenca_chagas_amostras_testadas,exames_triagem_doenca_doenca_chagas_amostras_reagentes,exames_triagem_doenca_hiv_amostras_testadas,exames_triagem_doenca_hiv_amostras_reagentes,exames_triagem_doenca_sifilis_amostras_testadas,`exames_triagem_doenca_sifilis_amostras_reagentes,exames_triagem_doenca_hepatite_b_hbs_ag_amostras_testadas,exames_triagem_doenca_hepatite_b_hbs_ag_amostras_reagentes,exames_triagem_doenca_hepatite_b_anti_hbc_amostras_testadas,exames_triagem_doenca_hepatite_b_anti_hbc_amostras_reagentes,exames_triagem_doenca_hepatite_c_amostras_testadas,exames_triagem_doenca_hepatite_c_amostras_reagentes,exames_triagem_doenca_htlv_i_ii_amostras_testadas,exames_triagem_doenca_htlv_i_ii_amostras_reagentes,exames_triagem_doenca_malaria_amostras_testadas,exames_triagem_doenca_malaria_amostras_reagentes,exames_triagem_doenca_hbv_teste_nat_amostras_testadas,exames_triagem_doenca_hbv_teste_nat_amostras_reagentes,exames_triagem_doenca_hcv_teste_nat_amostras_testadas,exames_triagem_doenca_hcv_teste_nat_amostras_reagentes,exames_triagem_doenca_hiv_teste_nat_amostras_testadas,exames_triagem_doenca_hiv_teste_nat_amostras_reagentes,imunohematologia_a_positivo_doador,imunohematologia_a_positivo_receptor,imunohe

In [29]:
hemoprod_ap_deduplicado.to_excel('dados_processados/hemoprod_ap.xlsx', index=False)

## Hemoprod Bahia

In [14]:
arquivo_dados_path = os.path.join(dados_brutos_path, 'Hemoprod_BA.xlsx')
nome_planilha = 'HEMOPROD - BAHIA'
hemoprod_ba = pd.read_excel(arquivo_dados_path, sheet_name=nome_planilha)
print("Arquivo de dados carregado com sucesso.")
print(f"Número de colunas original: {len(hemoprod_ba.columns)}")

try:
    # Lê apenas o cabeçalho do arquivo original para obter os nomes das colunas
    # df_original = pd.read_excel(arquivo_dados_path, nrows=0)
    colunas_originais = hemoprod_ba.columns.tolist()

    # Cria um novo DataFrame para o dicionário
    dicionario_df = pd.DataFrame({
        'nome_original': colunas_originais,
        'nome_sql': ''  # Adiciona uma coluna vazia para os novos nomes
    })

    # Salva o DataFrame do dicionário em um novo arquivo Excel
    dicionario_df.to_excel('dicionario_ba.xlsx', index=False)

    print(f"Arquivo de dicionário dicionario.xlsx gerado com sucesso!")
    print(f"O arquivo contém {len(colunas_originais)} colunas originais.")
    print("Agora você pode preencher a coluna 'nome_sql' com os nomes desejados.")

except FileNotFoundError:
    print(f"Erro: O arquivo de dados '{arquivo_dados_path}' não foi encontrado.")
except Exception as e:
    print(f"Ocorreu um erro inesperado: {e}")

Arquivo de dados carregado com sucesso.
Número de colunas original: 269
Arquivo de dicionário dicionario.xlsx gerado com sucesso!
O arquivo contém 269 colunas originais.
Agora você pode preencher a coluna 'nome_sql' com os nomes desejados.


In [30]:
import os
import pandas as pd

dados_brutos_path = 'dados_brutos'

arquivo_dados_path = os.path.join(dados_brutos_path, 'Hemoprod_BA.xlsx')
nome_planilha = 'HEMOPROD - BAHIA'

dicionario_path_ba = ('./dicionario_colunas_269.xlsx')
# dicionario_path_ap = ('./dicionario_colunas_270.xlsx')

dicionario_ba = pd.read_excel(dicionario_path_ba, sheet_name='Sheet1')

# --- 2. Carregue os dados e o dicionário ---
try:
    # Carrega o arquivo de dados
    hemoprod_ba = pd.read_excel(arquivo_dados_path, sheet_name=nome_planilha)
    print("Arquivo de dados carregado com sucesso.")
    print(f"Número de colunas original: {len(hemoprod_ba.columns)}")

    # Carrega o arquivo de dicionário
    dicionario = pd.read_excel(dicionario_path_ba)
    print("Arquivo de dicionário carregado com sucesso.")

    # --- 3. Extraia a lista de novos nomes ---
    # Pega os valores da coluna 'nome_sql' e converte para uma lista
    novos_nomes = dicionario['nome_sql'].tolist()
    print(f"Número de novos nomes no dicionário: {len(novos_nomes)}")

    # --- 4. Verificação de segurança (MUITO IMPORTANTE) ---
    # Garante que o número de colunas é o mesmo antes de renomear
    if len(hemoprod_ba.columns) == len(novos_nomes):
        print("\nO número de colunas corresponde. Renomeando...")
        
        # --- 5. Substitua os nomes das colunas ---
        # Esta é a linha principal que faz a substituição direta
        hemoprod_ba.columns = novos_nomes
        
        print("Colunas renomeadas com sucesso!")
        
        # --- 6. Verifique o resultado ---
        print("\nInformações do DataFrame com as novas colunas:")
        hemoprod_ba.info()
        
        print("\nAs 5 primeiras linhas com as novas colunas:")
        display(hemoprod_ba.head())

    else:
        # Mensagem de erro se o número de colunas for diferente
        print("\n--- ERRO ---")
        print("A renomeação foi cancelada. O número de colunas no arquivo de dados não é igual ao número de nomes no dicionário.")
        print(f"Colunas no arquivo de dados: {len(hemoprod_ba.columns)}")
        print(f"Nomes no dicionário: {len(novos_nomes)}")

except FileNotFoundError as e:
    print(f"\nErro de arquivo não encontrado: {e}")
except KeyError as e:
    print(f"\nErro de coluna não encontrada: {e}. Verifique se a coluna 'nome_sql' existe no seu arquivo de dicionário.")
except Exception as e:
    print(f"\nOcorreu um erro inesperado: {e}")



Arquivo de dados carregado com sucesso.
Número de colunas original: 269
Arquivo de dicionário carregado com sucesso.
Número de novos nomes no dicionário: 269

O número de colunas corresponde. Renomeando...
Colunas renomeadas com sucesso!

Informações do DataFrame com as novas colunas:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 975 entries, 0 to 974
Columns: 269 entries, id to hemoprod_3_observacoes
dtypes: float64(247), int64(4), object(18)
memory usage: 2.0+ MB

As 5 primeiras linhas com as novas colunas:


,id,data_envio,ultima_pagina,idioma_inicial,semente,codigo_acesso,data_inicio,data_ultima_acao,ip,identificacao_dado,tipo_envio,ano_referencia,periodo_referencia,identificacao_estabelecimento,municipio,razao_social_nome_fantasia,razao_social_nome_fantasia_outros,cnpj,tipo_estabelecimento,natureza_estabelecimento,dados_informados_referem_se,rede_estabelecimento,cnes,endereco,triagem_clinica_total_doacao_espontanea_aptos,triagem_clinica_total_doacao_espontanea_inaptos,triagem_clinica_total_doacao_reposicao_aptos,triagem_clinica_total_doacao_reposicao_inaptos,triagem_clinica_total_doacao_autologa_aptos,triagem_clinica_total_doacao_autologa_inaptos,total_doador_primeira_vez_aptos,total_doador_primeira_vez_inaptos,total_doador_repeticao_aptos,total_doador_repeticao_inaptos,total_doador_esporadico_aptos,total_doador_esporadico_inaptos,total_doador_masculino_aptos,total_doador_masculino_inaptos,total_doador_feminino_aptos,total_doador_feminino_inaptos,total_doador_menor_de_18_anos_aptos,total_doador_menor_de_18_anos_inaptos,total_doador_18_ate_29_anos_aptos,total_doador_18_ate_29_anos_inaptos,total_doador_acima_de_29_anos_aptos,total_doador_acima_de_29_anos_inaptos,total_candidatos_inaptos_anemia_masculino,total_candidatos_inaptos_anemia_feminino,total_candidatos_inaptos_anemia_total,total_candidatos_inaptos_hipertensao_masculino,total_candidatos_inaptos_hipertensao_feminino,total_candidatos_inaptos_hipertensao_total,total_candidatos_inaptos_hipotensao_masculino,total_candidatos_inaptos_hipotensao_feminino,total_candidatos_inaptos_hipotensao_total,total_candidatos_inaptos_alcoolismo_masculino,total_candidatos_inaptos_alcoolismo_feminino,total_candidatos_inaptos_alcoolismo_total,total_candidatos_inaptos_comportamento_risco_dst_masculino,total_candidatos_inaptos_comportamento_risco_dst_feminino,total_candidatos_inaptos_comportamento_risco_dst_total,total_candidatos_inaptos_uso_drogas_masculino,total_candidatos_inaptos_uso_drogas_feminino,total_candidatos_inaptos_uso_drogas_total,total_candidatos_inaptos_hepatite_masculino,total_candidatos_inaptos_hepatite_feminino,total_candidatos_inaptos_hepatite_total,total_candidatos_inaptos_doenca_chagas_masculino,total_candidatos_inaptos_doenca_chagas_feminino,total_candidatos_inaptos_doenca_chagas_total,total_candidatos_inaptos_malaria_masculino,total_candidatos_inaptos_malaria_feminino,total_candidatos_inaptos_malaria_total,total_candidatos_inaptos_outras_masculino,total_candidatos_inaptos_outras_feminino,total_candidatos_inaptos_outras_total,coleta_total_candidatos_desistentes,total_interrupcoes_coleta_dificuldade_puncao_venosa,total_interrupcoes_coleta_reacao_vagal,total_interrupcoes_coleta_outros_motivos,total_coletas_sangue_total,total_coletas_aferese,hemoprod_1_observacoes,exames_triagem_doenca_doenca_chagas_amostras_testadas,exames_triagem_doenca_doenca_chagas_amostras_reagentes,exames_triagem_doenca_hiv_amostras_testadas,exames_triagem_doenca_hiv_amostras_reagentes,exames_triagem_doenca_sifilis_amostras_testadas,`exames_triagem_doenca_sifilis_amostras_reagentes,exames_triagem_doenca_hepatite_b_hbs_ag_amostras_testadas,exames_triagem_doenca_hepatite_b_hbs_ag_amostras_reagentes,exames_triagem_doenca_hepatite_b_anti_hbc_amostras_testadas,exames_triagem_doenca_hepatite_b_anti_hbc_amostras_reagentes,exames_triagem_doenca_hepatite_c_amostras_testadas,exames_triagem_doenca_hepatite_c_amostras_reagentes,exames_triagem_doenca_htlv_i_ii_amostras_testadas,exames_triagem_doenca_htlv_i_ii_amostras_reagentes,exames_triagem_doenca_malaria_amostras_testadas,exames_triagem_doenca_malaria_amostras_reagentes,exames_triagem_doenca_hbv_teste_nat_amostras_testadas,exames_triagem_doenca_hbv_teste_nat_amostras_reagentes,exames_triagem_doenca_hcv_teste_nat_amostras_testadas,exames_triagem_doenca_hcv_teste_nat_amostras_reagentes,exames_triagem_doenca_hiv_teste_nat_amostras_testadas,exames_triagem_doenca_hiv_teste_nat_amostras_reagentes,imunohematologia_a_positivo_doador,imunohematologia_a_positivo_receptor,imunohe

In [31]:
# --- 2. Defina as colunas para a chave e para ordenação ---
registros_antes = len(hemoprod_ba)
print(f"Total de registros ANTES da remoção de duplicatas: {registros_antes}")
    

colunas_chave = [
        'cnpj', 
        'ano_referencia', 
        'periodo_referencia', 
        'razao_social_nome_fantasia'
    ]

coluna_data = 'data_envio'

# --- 3. Verifique se as colunas necessárias existem ---
colunas_necessarias = colunas_chave + [coluna_data]

if not all(col in hemoprod_ba.columns for col in colunas_necessarias):
    print("\n--- ERRO ---")
    print("Uma ou mais colunas necessárias para a deduplicação não foram encontradas.")
    colunas_faltantes = [col for col in colunas_necessarias if col not in hemoprod_ba.columns]
    print(f"Colunas necessárias: {colunas_necessarias}")
    print(f"Colunas faltantes no DataFrame: {colunas_faltantes}")
else:
    # --- 4. Prepare a coluna de data e ordene os dados ---
    # Converte a coluna 'data_envio' para datetime para garantir a ordenação correta.
    # 'errors='coerce'' transformará datas inválidas em NaT (Not a Time), que são tratadas como nulas.
    hemoprod_ba[coluna_data] = pd.to_datetime(hemoprod_ba[coluna_data], errors='coerce')

    # Ordena o DataFrame. Os registros com data de envio mais recente ficarão por último.
    print(f"\nOrdenando os dados por '{coluna_data}'...")
    hemoprod_ba_ordenado = hemoprod_ba.sort_values(by=coluna_data, ascending=True)

    # --- 5. Identifique e separe os registros duplicados e únicos ---
    # Em vez de usar drop_duplicates() diretamente, vamos usar duplicated()
    # para criar uma máscara booleana.
    # 'keep='last'' marca todas as ocorrências de uma chave como True, EXCETO a última (a mais recente).
    print(f"Identificando duplicatas com base na chave: {colunas_chave}...")
    mascara_duplicatas = hemoprod_ba_ordenado.duplicated(subset=colunas_chave, keep='last')

    # O DataFrame de removidos conterá todas as linhas marcadas como True
    hemoprod_removidos = hemoprod_ba_ordenado[mascara_duplicatas]
    
    # O DataFrame deduplicado conterá o INVERSO (~) da máscara (linhas marcadas como False)
    hemoprod_ba_deduplicado = hemoprod_ba_ordenado[~mascara_duplicatas]
    
    # Agora podemos contar os registros diretamente dos novos DataFrames
    registros_depois = len(hemoprod_ba_deduplicado)
    registros_removidos = len(hemoprod_removidos) 
    
    # --- 6. Exiba o resultado ---
    print("\n--- Processo Concluído ---")
    print(f"Registros removidos: {registros_removidos}")
    print(f"Total de registros DEPOIS da remoção de duplicatas: {registros_depois}")

    # (Opcional) Exibe a amostra dos removidos
    print("\nAmostra dos dados REMOVIDOS (os mais antigos/duplicados):")
    display(hemoprod_removidos.head(10))

    # Você pode continuar a usar o DataFrame 'hemoprod_ba_deduplicado' para suas análises
    print("\nAmostra dos dados únicos (os mais recentes para cada chave):")
    display(hemoprod_ba_deduplicado.head(10))
    
    # Agora você tem o DataFrame 'hemoprod_removidos' salvo

Total de registros ANTES da remoção de duplicatas: 975

Ordenando os dados por 'data_envio'...
Identificando duplicatas com base na chave: ['cnpj', 'ano_referencia', 'periodo_referencia', 'razao_social_nome_fantasia']...

--- Processo Concluído ---
Registros removidos: 267
Total de registros DEPOIS da remoção de duplicatas: 708

Amostra dos dados REMOVIDOS (os mais antigos/duplicados):


,id,data_envio,ultima_pagina,idioma_inicial,semente,codigo_acesso,data_inicio,data_ultima_acao,ip,identificacao_dado,tipo_envio,ano_referencia,periodo_referencia,identificacao_estabelecimento,municipio,razao_social_nome_fantasia,razao_social_nome_fantasia_outros,cnpj,tipo_estabelecimento,natureza_estabelecimento,dados_informados_referem_se,rede_estabelecimento,cnes,endereco,triagem_clinica_total_doacao_espontanea_aptos,triagem_clinica_total_doacao_espontanea_inaptos,triagem_clinica_total_doacao_reposicao_aptos,triagem_clinica_total_doacao_reposicao_inaptos,triagem_clinica_total_doacao_autologa_aptos,triagem_clinica_total_doacao_autologa_inaptos,total_doador_primeira_vez_aptos,total_doador_primeira_vez_inaptos,total_doador_repeticao_aptos,total_doador_repeticao_inaptos,total_doador_esporadico_aptos,total_doador_esporadico_inaptos,total_doador_masculino_aptos,total_doador_masculino_inaptos,total_doador_feminino_aptos,total_doador_feminino_inaptos,total_doador_menor_de_18_anos_aptos,total_doador_menor_de_18_anos_inaptos,total_doador_18_ate_29_anos_aptos,total_doador_18_ate_29_anos_inaptos,total_doador_acima_de_29_anos_aptos,total_doador_acima_de_29_anos_inaptos,total_candidatos_inaptos_anemia_masculino,total_candidatos_inaptos_anemia_feminino,total_candidatos_inaptos_anemia_total,total_candidatos_inaptos_hipertensao_masculino,total_candidatos_inaptos_hipertensao_feminino,total_candidatos_inaptos_hipertensao_total,total_candidatos_inaptos_hipotensao_masculino,total_candidatos_inaptos_hipotensao_feminino,total_candidatos_inaptos_hipotensao_total,total_candidatos_inaptos_alcoolismo_masculino,total_candidatos_inaptos_alcoolismo_feminino,total_candidatos_inaptos_alcoolismo_total,total_candidatos_inaptos_comportamento_risco_dst_masculino,total_candidatos_inaptos_comportamento_risco_dst_feminino,total_candidatos_inaptos_comportamento_risco_dst_total,total_candidatos_inaptos_uso_drogas_masculino,total_candidatos_inaptos_uso_drogas_feminino,total_candidatos_inaptos_uso_drogas_total,total_candidatos_inaptos_hepatite_masculino,total_candidatos_inaptos_hepatite_feminino,total_candidatos_inaptos_hepatite_total,total_candidatos_inaptos_doenca_chagas_masculino,total_candidatos_inaptos_doenca_chagas_feminino,total_candidatos_inaptos_doenca_chagas_total,total_candidatos_inaptos_malaria_masculino,total_candidatos_inaptos_malaria_feminino,total_candidatos_inaptos_malaria_total,total_candidatos_inaptos_outras_masculino,total_candidatos_inaptos_outras_feminino,total_candidatos_inaptos_outras_total,coleta_total_candidatos_desistentes,total_interrupcoes_coleta_dificuldade_puncao_venosa,total_interrupcoes_coleta_reacao_vagal,total_interrupcoes_coleta_outros_motivos,total_coletas_sangue_total,total_coletas_aferese,hemoprod_1_observacoes,exames_triagem_doenca_doenca_chagas_amostras_testadas,exames_triagem_doenca_doenca_chagas_amostras_reagentes,exames_triagem_doenca_hiv_amostras_testadas,exames_triagem_doenca_hiv_amostras_reagentes,exames_triagem_doenca_sifilis_amostras_testadas,`exames_triagem_doenca_sifilis_amostras_reagentes,exames_triagem_doenca_hepatite_b_hbs_ag_amostras_testadas,exames_triagem_doenca_hepatite_b_hbs_ag_amostras_reagentes,exames_triagem_doenca_hepatite_b_anti_hbc_amostras_testadas,exames_triagem_doenca_hepatite_b_anti_hbc_amostras_reagentes,exames_triagem_doenca_hepatite_c_amostras_testadas,exames_triagem_doenca_hepatite_c_amostras_reagentes,exames_triagem_doenca_htlv_i_ii_amostras_testadas,exames_triagem_doenca_htlv_i_ii_amostras_reagentes,exames_triagem_doenca_malaria_amostras_testadas,exames_triagem_doenca_malaria_amostras_reagentes,exames_triagem_doenca_hbv_teste_nat_amostras_testadas,exames_triagem_doenca_hbv_teste_nat_amostras_reagentes,exames_triagem_doenca_hcv_teste_nat_amostras_testadas,exames_triagem_doenca_hcv_teste_nat_amostras_reagentes,exames_triagem_doenca_hiv_teste_nat_amostras_testadas,exames_triagem_doenca_hiv_teste_nat_amostras_reagentes,imunohematologia_a_positivo_doador,imunohematologia_a_positivo_receptor,imunohe


Amostra dos dados únicos (os mais recentes para cada chave):


,id,data_envio,ultima_pagina,idioma_inicial,semente,codigo_acesso,data_inicio,data_ultima_acao,ip,identificacao_dado,tipo_envio,ano_referencia,periodo_referencia,identificacao_estabelecimento,municipio,razao_social_nome_fantasia,razao_social_nome_fantasia_outros,cnpj,tipo_estabelecimento,natureza_estabelecimento,dados_informados_referem_se,rede_estabelecimento,cnes,endereco,triagem_clinica_total_doacao_espontanea_aptos,triagem_clinica_total_doacao_espontanea_inaptos,triagem_clinica_total_doacao_reposicao_aptos,triagem_clinica_total_doacao_reposicao_inaptos,triagem_clinica_total_doacao_autologa_aptos,triagem_clinica_total_doacao_autologa_inaptos,total_doador_primeira_vez_aptos,total_doador_primeira_vez_inaptos,total_doador_repeticao_aptos,total_doador_repeticao_inaptos,total_doador_esporadico_aptos,total_doador_esporadico_inaptos,total_doador_masculino_aptos,total_doador_masculino_inaptos,total_doador_feminino_aptos,total_doador_feminino_inaptos,total_doador_menor_de_18_anos_aptos,total_doador_menor_de_18_anos_inaptos,total_doador_18_ate_29_anos_aptos,total_doador_18_ate_29_anos_inaptos,total_doador_acima_de_29_anos_aptos,total_doador_acima_de_29_anos_inaptos,total_candidatos_inaptos_anemia_masculino,total_candidatos_inaptos_anemia_feminino,total_candidatos_inaptos_anemia_total,total_candidatos_inaptos_hipertensao_masculino,total_candidatos_inaptos_hipertensao_feminino,total_candidatos_inaptos_hipertensao_total,total_candidatos_inaptos_hipotensao_masculino,total_candidatos_inaptos_hipotensao_feminino,total_candidatos_inaptos_hipotensao_total,total_candidatos_inaptos_alcoolismo_masculino,total_candidatos_inaptos_alcoolismo_feminino,total_candidatos_inaptos_alcoolismo_total,total_candidatos_inaptos_comportamento_risco_dst_masculino,total_candidatos_inaptos_comportamento_risco_dst_feminino,total_candidatos_inaptos_comportamento_risco_dst_total,total_candidatos_inaptos_uso_drogas_masculino,total_candidatos_inaptos_uso_drogas_feminino,total_candidatos_inaptos_uso_drogas_total,total_candidatos_inaptos_hepatite_masculino,total_candidatos_inaptos_hepatite_feminino,total_candidatos_inaptos_hepatite_total,total_candidatos_inaptos_doenca_chagas_masculino,total_candidatos_inaptos_doenca_chagas_feminino,total_candidatos_inaptos_doenca_chagas_total,total_candidatos_inaptos_malaria_masculino,total_candidatos_inaptos_malaria_feminino,total_candidatos_inaptos_malaria_total,total_candidatos_inaptos_outras_masculino,total_candidatos_inaptos_outras_feminino,total_candidatos_inaptos_outras_total,coleta_total_candidatos_desistentes,total_interrupcoes_coleta_dificuldade_puncao_venosa,total_interrupcoes_coleta_reacao_vagal,total_interrupcoes_coleta_outros_motivos,total_coletas_sangue_total,total_coletas_aferese,hemoprod_1_observacoes,exames_triagem_doenca_doenca_chagas_amostras_testadas,exames_triagem_doenca_doenca_chagas_amostras_reagentes,exames_triagem_doenca_hiv_amostras_testadas,exames_triagem_doenca_hiv_amostras_reagentes,exames_triagem_doenca_sifilis_amostras_testadas,`exames_triagem_doenca_sifilis_amostras_reagentes,exames_triagem_doenca_hepatite_b_hbs_ag_amostras_testadas,exames_triagem_doenca_hepatite_b_hbs_ag_amostras_reagentes,exames_triagem_doenca_hepatite_b_anti_hbc_amostras_testadas,exames_triagem_doenca_hepatite_b_anti_hbc_amostras_reagentes,exames_triagem_doenca_hepatite_c_amostras_testadas,exames_triagem_doenca_hepatite_c_amostras_reagentes,exames_triagem_doenca_htlv_i_ii_amostras_testadas,exames_triagem_doenca_htlv_i_ii_amostras_reagentes,exames_triagem_doenca_malaria_amostras_testadas,exames_triagem_doenca_malaria_amostras_reagentes,exames_triagem_doenca_hbv_teste_nat_amostras_testadas,exames_triagem_doenca_hbv_teste_nat_amostras_reagentes,exames_triagem_doenca_hcv_teste_nat_amostras_testadas,exames_triagem_doenca_hcv_teste_nat_amostras_reagentes,exames_triagem_doenca_hiv_teste_nat_amostras_testadas,exames_triagem_doenca_hiv_teste_nat_amostras_reagentes,imunohematologia_a_positivo_doador,imunohematologia_a_positivo_receptor,imunohe

In [32]:
hemoprod_ba_deduplicado.to_excel('dados_processados/hemoprod_ba.xlsx', index=False)

## Hemoprod Ceara

In [ ]:
import os
import pandas as pd

dados_brutos_path = 'dados_brutos'

arquivo_dados_path = os.path.join(dados_brutos_path, 'Hemoprod_CE.xlsx')
nome_planilha = 'Planilha1'

dicionario_path_ce = ('./dicionario_colunas_269.xlsx')
# dicionario_path_ap = ('./dicionario_colunas_270.xlsx')

dicionario_ce = pd.read_excel(dicionario_path_ce, sheet_name='Sheet1')

# --- 2. Carregue os dados e o dicionário ---
try:
    # Carrega o arquivo de dados
    hemoprod_ce = pd.read_excel(arquivo_dados_path, sheet_name=nome_planilha)
    print("Arquivo de dados carregado com sucesso.")
    print(f"Número de colunas original: {len(hemoprod_ce.columns)}")

    # Carrega o arquivo de dicionário
    dicionario = pd.read_excel(dicionario_path_ce)
    print("Arquivo de dicionário carregado com sucesso.")

    # --- 3. Extraia a lista de novos nomes ---
    # Pega os valores da coluna 'nome_sql' e converte para uma lista
    novos_nomes = dicionario['nome_sql'].tolist()
    print(f"Número de novos nomes no dicionário: {len(novos_nomes)}")

    # --- 4. Verificação de segurança (MUITO IMPORTANTE) ---
    # Garante que o número de colunas é o mesmo antes de renomear
    if len(hemoprod_ce.columns) == len(novos_nomes):
        print("\nO número de colunas corresponde. Renomeando...")
        
        # --- 5. Substitua os nomes das colunas ---
        # Esta é a linha principal que faz a substituição direta
        hemoprod_ce.columns = novos_nomes
        
        print("Colunas renomeadas com sucesso!")
        
        # --- 6. Verifique o resultado ---
        print("\nInformações do DataFrame com as novas colunas:")
        hemoprod_ce.info()
        
        print("\nAs 5 primeiras linhas com as novas colunas:")
        display(hemoprod_ce.head())

    else:
        # Mensagem de erro se o número de colunas for diferente
        print("\n--- ERRO ---")
        print("A renomeação foi cancelada. O número de colunas no arquivo de dados não é igual ao número de nomes no dicionário.")
        print(f"Colunas no arquivo de dados: {len(hemoprod_ce.columns)}")
        print(f"Nomes no dicionário: {len(novos_nomes)}")

except FileNotFoundError as e:
    print(f"\nErro de arquivo não encontrado: {e}")
except KeyError as e:
    print(f"\nErro de coluna não encontrada: {e}. Verifique se a coluna 'nome_sql' existe no seu arquivo de dicionário.")
except Exception as e:
    print(f"\nOcorreu um erro inesperado: {e}")



Arquivo de dados carregado com sucesso.
Número de colunas original: 269
Arquivo de dicionário carregado com sucesso.
Número de novos nomes no dicionário: 269

O número de colunas corresponde. Renomeando...
Colunas renomeadas com sucesso!

Informações do DataFrame com as novas colunas:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 228 entries, 0 to 227
Columns: 269 entries, id to hemoprod_3_observacoes
dtypes: datetime64[ns](3), float64(253), object(13)
memory usage: 479.3+ KB

As 5 primeiras linhas com as novas colunas:


,id,data_envio,ultima_pagina,idioma_inicial,semente,codigo_acesso,data_inicio,data_ultima_acao,ip,identificacao_dado,tipo_envio,ano_referencia,periodo_referencia,identificacao_estabelecimento,municipio,razao_social_nome_fantasia,razao_social_nome_fantasia_outros,cnpj,tipo_estabelecimento,natureza_estabelecimento,dados_informados_referem_se,rede_estabelecimento,cnes,endereco,triagem_clinica_total_doacao_espontanea_aptos,triagem_clinica_total_doacao_espontanea_inaptos,triagem_clinica_total_doacao_reposicao_aptos,triagem_clinica_total_doacao_reposicao_inaptos,triagem_clinica_total_doacao_autologa_aptos,triagem_clinica_total_doacao_autologa_inaptos,total_doador_primeira_vez_aptos,total_doador_primeira_vez_inaptos,total_doador_repeticao_aptos,total_doador_repeticao_inaptos,total_doador_esporadico_aptos,total_doador_esporadico_inaptos,total_doador_masculino_aptos,total_doador_masculino_inaptos,total_doador_feminino_aptos,total_doador_feminino_inaptos,total_doador_menor_de_18_anos_aptos,total_doador_menor_de_18_anos_inaptos,total_doador_18_ate_29_anos_aptos,total_doador_18_ate_29_anos_inaptos,total_doador_acima_de_29_anos_aptos,total_doador_acima_de_29_anos_inaptos,total_candidatos_inaptos_anemia_masculino,total_candidatos_inaptos_anemia_feminino,total_candidatos_inaptos_anemia_total,total_candidatos_inaptos_hipertensao_masculino,total_candidatos_inaptos_hipertensao_feminino,total_candidatos_inaptos_hipertensao_total,total_candidatos_inaptos_hipotensao_masculino,total_candidatos_inaptos_hipotensao_feminino,total_candidatos_inaptos_hipotensao_total,total_candidatos_inaptos_alcoolismo_masculino,total_candidatos_inaptos_alcoolismo_feminino,total_candidatos_inaptos_alcoolismo_total,total_candidatos_inaptos_comportamento_risco_dst_masculino,total_candidatos_inaptos_comportamento_risco_dst_feminino,total_candidatos_inaptos_comportamento_risco_dst_total,total_candidatos_inaptos_uso_drogas_masculino,total_candidatos_inaptos_uso_drogas_feminino,total_candidatos_inaptos_uso_drogas_total,total_candidatos_inaptos_hepatite_masculino,total_candidatos_inaptos_hepatite_feminino,total_candidatos_inaptos_hepatite_total,total_candidatos_inaptos_doenca_chagas_masculino,total_candidatos_inaptos_doenca_chagas_feminino,total_candidatos_inaptos_doenca_chagas_total,total_candidatos_inaptos_malaria_masculino,total_candidatos_inaptos_malaria_feminino,total_candidatos_inaptos_malaria_total,total_candidatos_inaptos_outras_masculino,total_candidatos_inaptos_outras_feminino,total_candidatos_inaptos_outras_total,coleta_total_candidatos_desistentes,total_interrupcoes_coleta_dificuldade_puncao_venosa,total_interrupcoes_coleta_reacao_vagal,total_interrupcoes_coleta_outros_motivos,total_coletas_sangue_total,total_coletas_aferese,hemoprod_1_observacoes,exames_triagem_doenca_doenca_chagas_amostras_testadas,exames_triagem_doenca_doenca_chagas_amostras_reagentes,exames_triagem_doenca_hiv_amostras_testadas,exames_triagem_doenca_hiv_amostras_reagentes,exames_triagem_doenca_sifilis_amostras_testadas,`exames_triagem_doenca_sifilis_amostras_reagentes,exames_triagem_doenca_hepatite_b_hbs_ag_amostras_testadas,exames_triagem_doenca_hepatite_b_hbs_ag_amostras_reagentes,exames_triagem_doenca_hepatite_b_anti_hbc_amostras_testadas,exames_triagem_doenca_hepatite_b_anti_hbc_amostras_reagentes,exames_triagem_doenca_hepatite_c_amostras_testadas,exames_triagem_doenca_hepatite_c_amostras_reagentes,exames_triagem_doenca_htlv_i_ii_amostras_testadas,exames_triagem_doenca_htlv_i_ii_amostras_reagentes,exames_triagem_doenca_malaria_amostras_testadas,exames_triagem_doenca_malaria_amostras_reagentes,exames_triagem_doenca_hbv_teste_nat_amostras_testadas,exames_triagem_doenca_hbv_teste_nat_amostras_reagentes,exames_triagem_doenca_hcv_teste_nat_amostras_testadas,exames_triagem_doenca_hcv_teste_nat_amostras_reagentes,exames_triagem_doenca_hiv_teste_nat_amostras_testadas,exames_triagem_doenca_hiv_teste_nat_amostras_reagentes,imunohematologia_a_positivo_doador,imunohematologia_a_positivo_receptor,imunohe

In [40]:
# --- 2. Defina as colunas para a chave e para ordenação ---
registros_antes = len(hemoprod_ce)
print(f"Total de registros ANTES da remoção de duplicatas: {registros_antes}")
    

colunas_chave = [
        'cnpj', 
        'ano_referencia', 
        'periodo_referencia', 
        'razao_social_nome_fantasia'
    ]

coluna_data = 'data_envio'

# --- 3. Verifique se as colunas necessárias existem ---
colunas_necessarias = colunas_chave + [coluna_data]

if not all(col in hemoprod_ce.columns for col in colunas_necessarias):
    print("\n--- ERRO ---")
    print("Uma ou mais colunas necessárias para a deduplicação não foram encontradas.")
    colunas_faltantes = [col for col in colunas_necessarias if col not in hemoprod_ce.columns]
    print(f"Colunas necessárias: {colunas_necessarias}")
    print(f"Colunas faltantes no DataFrame: {colunas_faltantes}")
else:
    # --- 4. Prepare a coluna de data e ordene os dados ---
    # Converte a coluna 'data_envio' para datetime para garantir a ordenação correta.
    # 'errors='coerce'' transformará datas inválidas em NaT (Not a Time), que são tratadas como nulas.
    hemoprod_ce[coluna_data] = pd.to_datetime(hemoprod_ce[coluna_data], errors='coerce')

    # Ordena o DataFrame. Os registros com data de envio mais recente ficarão por último.
    print(f"\nOrdenando os dados por '{coluna_data}'...")
    hemoprod_ce_ordenado = hemoprod_ce.sort_values(by=coluna_data, ascending=True)

    # --- 5. Identifique e separe os registros duplicados e únicos ---
    # Em vez de usar drop_duplicates() diretamente, vamos usar duplicated()
    # para criar uma máscara booleana.
    # 'keep='last'' marca todas as ocorrências de uma chave como True, EXCETO a última (a mais recente).
    print(f"Identificando duplicatas com cese na chave: {colunas_chave}...")
    mascara_duplicatas = hemoprod_ce_ordenado.duplicated(subset=colunas_chave, keep='last')

    # O DataFrame de removidos conterá todas as linhas marcadas como True
    hemoprod_removidos = hemoprod_ce_ordenado[mascara_duplicatas]
    
    # O DataFrame deduplicado conterá o INVERSO (~) da máscara (linhas marcadas como False)
    hemoprod_ce_deduplicado = hemoprod_ce_ordenado[~mascara_duplicatas]
    
    # Agora podemos contar os registros diretamente dos novos DataFrames
    registros_depois = len(hemoprod_ce_deduplicado)
    registros_removidos = len(hemoprod_removidos) 
    
    # --- 6. Exice o resultado ---
    print("\n--- Processo Concluído ---")
    print(f"Registros removidos: {registros_removidos}")
    print(f"Total de registros DEPOIS da remoção de duplicatas: {registros_depois}")

    # (Opcional) Exibe a amostra dos removidos
    print("\nAmostra dos dados REMOVIDOS (os mais antigos/duplicados):")
    display(hemoprod_removidos.head(10))

    # Você pode continuar a usar o DataFrame 'hemoprod_ce_deduplicado' para suas análises
    print("\nAmostra dos dados únicos (os mais recentes para cada chave):")
    display(hemoprod_ce_deduplicado.head(10))
    
    # Agora você tem o DataFrame 'hemoprod_removidos' salvo

Total de registros ANTES da remoção de duplicatas: 228

Ordenando os dados por 'data_envio'...
Identificando duplicatas com cese na chave: ['cnpj', 'ano_referencia', 'periodo_referencia', 'razao_social_nome_fantasia']...

--- Processo Concluído ---
Registros removidos: 162
Total de registros DEPOIS da remoção de duplicatas: 66

Amostra dos dados REMOVIDOS (os mais antigos/duplicados):


,id,data_envio,ultima_pagina,idioma_inicial,semente,codigo_acesso,data_inicio,data_ultima_acao,ip,identificacao_dado,tipo_envio,ano_referencia,periodo_referencia,identificacao_estabelecimento,municipio,razao_social_nome_fantasia,razao_social_nome_fantasia_outros,cnpj,tipo_estabelecimento,natureza_estabelecimento,dados_informados_referem_se,rede_estabelecimento,cnes,endereco,triagem_clinica_total_doacao_espontanea_aptos,triagem_clinica_total_doacao_espontanea_inaptos,triagem_clinica_total_doacao_reposicao_aptos,triagem_clinica_total_doacao_reposicao_inaptos,triagem_clinica_total_doacao_autologa_aptos,triagem_clinica_total_doacao_autologa_inaptos,total_doador_primeira_vez_aptos,total_doador_primeira_vez_inaptos,total_doador_repeticao_aptos,total_doador_repeticao_inaptos,total_doador_esporadico_aptos,total_doador_esporadico_inaptos,total_doador_masculino_aptos,total_doador_masculino_inaptos,total_doador_feminino_aptos,total_doador_feminino_inaptos,total_doador_menor_de_18_anos_aptos,total_doador_menor_de_18_anos_inaptos,total_doador_18_ate_29_anos_aptos,total_doador_18_ate_29_anos_inaptos,total_doador_acima_de_29_anos_aptos,total_doador_acima_de_29_anos_inaptos,total_candidatos_inaptos_anemia_masculino,total_candidatos_inaptos_anemia_feminino,total_candidatos_inaptos_anemia_total,total_candidatos_inaptos_hipertensao_masculino,total_candidatos_inaptos_hipertensao_feminino,total_candidatos_inaptos_hipertensao_total,total_candidatos_inaptos_hipotensao_masculino,total_candidatos_inaptos_hipotensao_feminino,total_candidatos_inaptos_hipotensao_total,total_candidatos_inaptos_alcoolismo_masculino,total_candidatos_inaptos_alcoolismo_feminino,total_candidatos_inaptos_alcoolismo_total,total_candidatos_inaptos_comportamento_risco_dst_masculino,total_candidatos_inaptos_comportamento_risco_dst_feminino,total_candidatos_inaptos_comportamento_risco_dst_total,total_candidatos_inaptos_uso_drogas_masculino,total_candidatos_inaptos_uso_drogas_feminino,total_candidatos_inaptos_uso_drogas_total,total_candidatos_inaptos_hepatite_masculino,total_candidatos_inaptos_hepatite_feminino,total_candidatos_inaptos_hepatite_total,total_candidatos_inaptos_doenca_chagas_masculino,total_candidatos_inaptos_doenca_chagas_feminino,total_candidatos_inaptos_doenca_chagas_total,total_candidatos_inaptos_malaria_masculino,total_candidatos_inaptos_malaria_feminino,total_candidatos_inaptos_malaria_total,total_candidatos_inaptos_outras_masculino,total_candidatos_inaptos_outras_feminino,total_candidatos_inaptos_outras_total,coleta_total_candidatos_desistentes,total_interrupcoes_coleta_dificuldade_puncao_venosa,total_interrupcoes_coleta_reacao_vagal,total_interrupcoes_coleta_outros_motivos,total_coletas_sangue_total,total_coletas_aferese,hemoprod_1_observacoes,exames_triagem_doenca_doenca_chagas_amostras_testadas,exames_triagem_doenca_doenca_chagas_amostras_reagentes,exames_triagem_doenca_hiv_amostras_testadas,exames_triagem_doenca_hiv_amostras_reagentes,exames_triagem_doenca_sifilis_amostras_testadas,`exames_triagem_doenca_sifilis_amostras_reagentes,exames_triagem_doenca_hepatite_b_hbs_ag_amostras_testadas,exames_triagem_doenca_hepatite_b_hbs_ag_amostras_reagentes,exames_triagem_doenca_hepatite_b_anti_hbc_amostras_testadas,exames_triagem_doenca_hepatite_b_anti_hbc_amostras_reagentes,exames_triagem_doenca_hepatite_c_amostras_testadas,exames_triagem_doenca_hepatite_c_amostras_reagentes,exames_triagem_doenca_htlv_i_ii_amostras_testadas,exames_triagem_doenca_htlv_i_ii_amostras_reagentes,exames_triagem_doenca_malaria_amostras_testadas,exames_triagem_doenca_malaria_amostras_reagentes,exames_triagem_doenca_hbv_teste_nat_amostras_testadas,exames_triagem_doenca_hbv_teste_nat_amostras_reagentes,exames_triagem_doenca_hcv_teste_nat_amostras_testadas,exames_triagem_doenca_hcv_teste_nat_amostras_reagentes,exames_triagem_doenca_hiv_teste_nat_amostras_testadas,exames_triagem_doenca_hiv_teste_nat_amostras_reagentes,imunohematologia_a_positivo_doador,imunohematologia_a_positivo_receptor,imunohe


Amostra dos dados únicos (os mais recentes para cada chave):


,id,data_envio,ultima_pagina,idioma_inicial,semente,codigo_acesso,data_inicio,data_ultima_acao,ip,identificacao_dado,tipo_envio,ano_referencia,periodo_referencia,identificacao_estabelecimento,municipio,razao_social_nome_fantasia,razao_social_nome_fantasia_outros,cnpj,tipo_estabelecimento,natureza_estabelecimento,dados_informados_referem_se,rede_estabelecimento,cnes,endereco,triagem_clinica_total_doacao_espontanea_aptos,triagem_clinica_total_doacao_espontanea_inaptos,triagem_clinica_total_doacao_reposicao_aptos,triagem_clinica_total_doacao_reposicao_inaptos,triagem_clinica_total_doacao_autologa_aptos,triagem_clinica_total_doacao_autologa_inaptos,total_doador_primeira_vez_aptos,total_doador_primeira_vez_inaptos,total_doador_repeticao_aptos,total_doador_repeticao_inaptos,total_doador_esporadico_aptos,total_doador_esporadico_inaptos,total_doador_masculino_aptos,total_doador_masculino_inaptos,total_doador_feminino_aptos,total_doador_feminino_inaptos,total_doador_menor_de_18_anos_aptos,total_doador_menor_de_18_anos_inaptos,total_doador_18_ate_29_anos_aptos,total_doador_18_ate_29_anos_inaptos,total_doador_acima_de_29_anos_aptos,total_doador_acima_de_29_anos_inaptos,total_candidatos_inaptos_anemia_masculino,total_candidatos_inaptos_anemia_feminino,total_candidatos_inaptos_anemia_total,total_candidatos_inaptos_hipertensao_masculino,total_candidatos_inaptos_hipertensao_feminino,total_candidatos_inaptos_hipertensao_total,total_candidatos_inaptos_hipotensao_masculino,total_candidatos_inaptos_hipotensao_feminino,total_candidatos_inaptos_hipotensao_total,total_candidatos_inaptos_alcoolismo_masculino,total_candidatos_inaptos_alcoolismo_feminino,total_candidatos_inaptos_alcoolismo_total,total_candidatos_inaptos_comportamento_risco_dst_masculino,total_candidatos_inaptos_comportamento_risco_dst_feminino,total_candidatos_inaptos_comportamento_risco_dst_total,total_candidatos_inaptos_uso_drogas_masculino,total_candidatos_inaptos_uso_drogas_feminino,total_candidatos_inaptos_uso_drogas_total,total_candidatos_inaptos_hepatite_masculino,total_candidatos_inaptos_hepatite_feminino,total_candidatos_inaptos_hepatite_total,total_candidatos_inaptos_doenca_chagas_masculino,total_candidatos_inaptos_doenca_chagas_feminino,total_candidatos_inaptos_doenca_chagas_total,total_candidatos_inaptos_malaria_masculino,total_candidatos_inaptos_malaria_feminino,total_candidatos_inaptos_malaria_total,total_candidatos_inaptos_outras_masculino,total_candidatos_inaptos_outras_feminino,total_candidatos_inaptos_outras_total,coleta_total_candidatos_desistentes,total_interrupcoes_coleta_dificuldade_puncao_venosa,total_interrupcoes_coleta_reacao_vagal,total_interrupcoes_coleta_outros_motivos,total_coletas_sangue_total,total_coletas_aferese,hemoprod_1_observacoes,exames_triagem_doenca_doenca_chagas_amostras_testadas,exames_triagem_doenca_doenca_chagas_amostras_reagentes,exames_triagem_doenca_hiv_amostras_testadas,exames_triagem_doenca_hiv_amostras_reagentes,exames_triagem_doenca_sifilis_amostras_testadas,`exames_triagem_doenca_sifilis_amostras_reagentes,exames_triagem_doenca_hepatite_b_hbs_ag_amostras_testadas,exames_triagem_doenca_hepatite_b_hbs_ag_amostras_reagentes,exames_triagem_doenca_hepatite_b_anti_hbc_amostras_testadas,exames_triagem_doenca_hepatite_b_anti_hbc_amostras_reagentes,exames_triagem_doenca_hepatite_c_amostras_testadas,exames_triagem_doenca_hepatite_c_amostras_reagentes,exames_triagem_doenca_htlv_i_ii_amostras_testadas,exames_triagem_doenca_htlv_i_ii_amostras_reagentes,exames_triagem_doenca_malaria_amostras_testadas,exames_triagem_doenca_malaria_amostras_reagentes,exames_triagem_doenca_hbv_teste_nat_amostras_testadas,exames_triagem_doenca_hbv_teste_nat_amostras_reagentes,exames_triagem_doenca_hcv_teste_nat_amostras_testadas,exames_triagem_doenca_hcv_teste_nat_amostras_reagentes,exames_triagem_doenca_hiv_teste_nat_amostras_testadas,exames_triagem_doenca_hiv_teste_nat_amostras_reagentes,imunohematologia_a_positivo_doador,imunohematologia_a_positivo_receptor,imunohe

In [41]:
hemoprod_ce_deduplicado.to_excel('dados_processados/hemoprod_ce.xlsx', index=False)

## Hemoprod Distrito Federal

In [42]:
import os
import pandas as pd

dados_brutos_path = 'dados_brutos'

arquivo_dados_path = os.path.join(dados_brutos_path, 'Hemoprod_DF.xlsx')
nome_planilha = 'HEMOPROD - DISTRITOFEDERAL'

dicionario_path_df = ('./dicionario_colunas_269.xlsx')
# dicionario_path_ap = ('./dicionario_colunas_270.xlsx')

dicionario_df = pd.read_excel(dicionario_path_df, sheet_name='Sheet1')

# --- 2. Carregue os dados e o dicionário ---
try:
    # Carrega o arquivo de dados
    hemoprod_df = pd.read_excel(arquivo_dados_path, sheet_name=nome_planilha)
    print("Arquivo de dados carregado com sucesso.")
    print(f"Número de colunas original: {len(hemoprod_df.columns)}")

    # Carrega o arquivo de dicionário
    dicionario = pd.read_excel(dicionario_path_df)
    print("Arquivo de dicionário carregado com sucesso.")

    # --- 3. Extraia a lista de novos nomes ---
    # Pega os valores da coluna 'nome_sql' e converte para uma lista
    novos_nomes = dicionario['nome_sql'].tolist()
    print(f"Número de novos nomes no dicionário: {len(novos_nomes)}")

    # --- 4. Verificação de segurança (MUITO IMPORTANTE) ---
    # Garante que o número de colunas é o mesmo antes de renomear
    if len(hemoprod_df.columns) == len(novos_nomes):
        print("\nO número de colunas corresponde. Renomeando...")
        
        # --- 5. Substitua os nomes das colunas ---
        # Esta é a linha principal que faz a substituição direta
        hemoprod_df.columns = novos_nomes
        
        print("Colunas renomeadas com sucesso!")
        
        # --- 6. Verifique o resultado ---
        print("\nInformações do DataFrame com as novas colunas:")
        hemoprod_df.info()
        
        print("\nAs 5 primeiras linhas com as novas colunas:")
        display(hemoprod_df.head())

    else:
        # Mensagem de erro se o número de colunas for diferente
        print("\n--- ERRO ---")
        print("A renomeação foi cancelada. O número de colunas no arquivo de dados não é igual ao número de nomes no dicionário.")
        print(f"Colunas no arquivo de dados: {len(hemoprod_df.columns)}")
        print(f"Nomes no dicionário: {len(novos_nomes)}")

except FileNotFoundError as e:
    print(f"\nErro de arquivo não encontrado: {e}")
except KeyError as e:
    print(f"\nErro de coluna não encontrada: {e}. Verifique se a coluna 'nome_sql' existe no seu arquivo de dicionário.")
except Exception as e:
    print(f"\nOcorreu um erro inesperado: {e}")



Arquivo de dados carregado com sucesso.
Número de colunas original: 269
Arquivo de dicionário carregado com sucesso.
Número de novos nomes no dicionário: 269

O número de colunas corresponde. Renomeando...
Colunas renomeadas com sucesso!

Informações do DataFrame com as novas colunas:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 184 entries, 0 to 183
Columns: 269 entries, id to hemoprod_3_observacoes
dtypes: float64(250), int64(4), object(15)
memory usage: 386.8+ KB

As 5 primeiras linhas com as novas colunas:


,id,data_envio,ultima_pagina,idioma_inicial,semente,codigo_acesso,data_inicio,data_ultima_acao,ip,identificacao_dado,tipo_envio,ano_referencia,periodo_referencia,identificacao_estabelecimento,municipio,razao_social_nome_fantasia,razao_social_nome_fantasia_outros,cnpj,tipo_estabelecimento,natureza_estabelecimento,dados_informados_referem_se,rede_estabelecimento,cnes,endereco,triagem_clinica_total_doacao_espontanea_aptos,triagem_clinica_total_doacao_espontanea_inaptos,triagem_clinica_total_doacao_reposicao_aptos,triagem_clinica_total_doacao_reposicao_inaptos,triagem_clinica_total_doacao_autologa_aptos,triagem_clinica_total_doacao_autologa_inaptos,total_doador_primeira_vez_aptos,total_doador_primeira_vez_inaptos,total_doador_repeticao_aptos,total_doador_repeticao_inaptos,total_doador_esporadico_aptos,total_doador_esporadico_inaptos,total_doador_masculino_aptos,total_doador_masculino_inaptos,total_doador_feminino_aptos,total_doador_feminino_inaptos,total_doador_menor_de_18_anos_aptos,total_doador_menor_de_18_anos_inaptos,total_doador_18_ate_29_anos_aptos,total_doador_18_ate_29_anos_inaptos,total_doador_acima_de_29_anos_aptos,total_doador_acima_de_29_anos_inaptos,total_candidatos_inaptos_anemia_masculino,total_candidatos_inaptos_anemia_feminino,total_candidatos_inaptos_anemia_total,total_candidatos_inaptos_hipertensao_masculino,total_candidatos_inaptos_hipertensao_feminino,total_candidatos_inaptos_hipertensao_total,total_candidatos_inaptos_hipotensao_masculino,total_candidatos_inaptos_hipotensao_feminino,total_candidatos_inaptos_hipotensao_total,total_candidatos_inaptos_alcoolismo_masculino,total_candidatos_inaptos_alcoolismo_feminino,total_candidatos_inaptos_alcoolismo_total,total_candidatos_inaptos_comportamento_risco_dst_masculino,total_candidatos_inaptos_comportamento_risco_dst_feminino,total_candidatos_inaptos_comportamento_risco_dst_total,total_candidatos_inaptos_uso_drogas_masculino,total_candidatos_inaptos_uso_drogas_feminino,total_candidatos_inaptos_uso_drogas_total,total_candidatos_inaptos_hepatite_masculino,total_candidatos_inaptos_hepatite_feminino,total_candidatos_inaptos_hepatite_total,total_candidatos_inaptos_doenca_chagas_masculino,total_candidatos_inaptos_doenca_chagas_feminino,total_candidatos_inaptos_doenca_chagas_total,total_candidatos_inaptos_malaria_masculino,total_candidatos_inaptos_malaria_feminino,total_candidatos_inaptos_malaria_total,total_candidatos_inaptos_outras_masculino,total_candidatos_inaptos_outras_feminino,total_candidatos_inaptos_outras_total,coleta_total_candidatos_desistentes,total_interrupcoes_coleta_dificuldade_puncao_venosa,total_interrupcoes_coleta_reacao_vagal,total_interrupcoes_coleta_outros_motivos,total_coletas_sangue_total,total_coletas_aferese,hemoprod_1_observacoes,exames_triagem_doenca_doenca_chagas_amostras_testadas,exames_triagem_doenca_doenca_chagas_amostras_reagentes,exames_triagem_doenca_hiv_amostras_testadas,exames_triagem_doenca_hiv_amostras_reagentes,exames_triagem_doenca_sifilis_amostras_testadas,`exames_triagem_doenca_sifilis_amostras_reagentes,exames_triagem_doenca_hepatite_b_hbs_ag_amostras_testadas,exames_triagem_doenca_hepatite_b_hbs_ag_amostras_reagentes,exames_triagem_doenca_hepatite_b_anti_hbc_amostras_testadas,exames_triagem_doenca_hepatite_b_anti_hbc_amostras_reagentes,exames_triagem_doenca_hepatite_c_amostras_testadas,exames_triagem_doenca_hepatite_c_amostras_reagentes,exames_triagem_doenca_htlv_i_ii_amostras_testadas,exames_triagem_doenca_htlv_i_ii_amostras_reagentes,exames_triagem_doenca_malaria_amostras_testadas,exames_triagem_doenca_malaria_amostras_reagentes,exames_triagem_doenca_hbv_teste_nat_amostras_testadas,exames_triagem_doenca_hbv_teste_nat_amostras_reagentes,exames_triagem_doenca_hcv_teste_nat_amostras_testadas,exames_triagem_doenca_hcv_teste_nat_amostras_reagentes,exames_triagem_doenca_hiv_teste_nat_amostras_testadas,exames_triagem_doenca_hiv_teste_nat_amostras_reagentes,imunohematologia_a_positivo_doador,imunohematologia_a_positivo_receptor,imunohe

In [43]:
# --- 2. Defina as colunas para a chave e para ordenação ---
registros_antes = len(hemoprod_df)
print(f"Total de registros ANTES da remoção de duplicatas: {registros_antes}")
    

colunas_chave = [
        'cnpj', 
        'ano_referencia', 
        'periodo_referencia', 
        'razao_social_nome_fantasia'
    ]

coluna_data = 'data_envio'

# --- 3. Verifique se as colunas necessárias existem ---
colunas_necessarias = colunas_chave + [coluna_data]

if not all(col in hemoprod_df.columns for col in colunas_necessarias):
    print("\n--- ERRO ---")
    print("Uma ou mais colunas necessárias para a deduplicação não foram encontradas.")
    colunas_faltantes = [col for col in colunas_necessarias if col not in hemoprod_df.columns]
    print(f"Colunas necessárias: {colunas_necessarias}")
    print(f"Colunas faltantes no DataFrame: {colunas_faltantes}")
else:
    # --- 4. Prepare a coluna de data e ordene os dados ---
    # Converte a coluna 'data_envio' para datetime para garantir a ordenação correta.
    # 'errors='coerce'' transformará datas inválidas em NaT (Not a Time), que são tratadas como nulas.
    hemoprod_df[coluna_data] = pd.to_datetime(hemoprod_df[coluna_data], errors='coerce')

    # Ordena o DataFrame. Os registros com data de envio mais recente ficarão por último.
    print(f"\nOrdenando os dados por '{coluna_data}'...")
    hemoprod_df_ordenado = hemoprod_df.sort_values(by=coluna_data, ascending=True)

    # --- 5. Identifique e separe os registros duplicados e únicos ---
    # Em vez de usar drop_duplicates() diretamente, vamos usar duplicated()
    # para criar uma máscara booleana.
    # 'keep='last'' marca todas as ocorrências de uma chave como True, EXCETO a última (a mais recente).
    print(f"Identificando duplicatas com cese na chave: {colunas_chave}...")
    mascara_duplicatas = hemoprod_df_ordenado.duplicated(subset=colunas_chave, keep='last')

    # O DataFrame de removidos conterá todas as linhas marcadas como True
    hemoprod_removidos = hemoprod_df_ordenado[mascara_duplicatas]
    
    # O DataFrame deduplicado conterá o INVERSO (~) da máscara (linhas marcadas como False)
    hemoprod_df_deduplicado = hemoprod_df_ordenado[~mascara_duplicatas]
    
    # Agora podemos contar os registros diretamente dos novos DataFrames
    registros_depois = len(hemoprod_df_deduplicado)
    registros_removidos = len(hemoprod_removidos) 
    
    # --- 6. Exice o resultado ---
    print("\n--- Processo Concluído ---")
    print(f"Registros removidos: {registros_removidos}")
    print(f"Total de registros DEPOIS da remoção de duplicatas: {registros_depois}")

    # (Opcional) Exibe a amostra dos removidos
    print("\nAmostra dos dados REMOVIDOS (os mais antigos/duplicados):")
    display(hemoprod_removidos.head(10))

    # Você pode continuar a usar o DataFrame 'hemoprod_ce_deduplicado' para suas análises
    print("\nAmostra dos dados únicos (os mais recentes para cada chave):")
    display(hemoprod_df_deduplicado.head(10))
    
    # Agora você tem o DataFrame 'hemoprod_removidos' salvo

Total de registros ANTES da remoção de duplicatas: 184

Ordenando os dados por 'data_envio'...
Identificando duplicatas com cese na chave: ['cnpj', 'ano_referencia', 'periodo_referencia', 'razao_social_nome_fantasia']...

--- Processo Concluído ---
Registros removidos: 1
Total de registros DEPOIS da remoção de duplicatas: 183

Amostra dos dados REMOVIDOS (os mais antigos/duplicados):


,id,data_envio,ultima_pagina,idioma_inicial,semente,codigo_acesso,data_inicio,data_ultima_acao,ip,identificacao_dado,tipo_envio,ano_referencia,periodo_referencia,identificacao_estabelecimento,municipio,razao_social_nome_fantasia,razao_social_nome_fantasia_outros,cnpj,tipo_estabelecimento,natureza_estabelecimento,dados_informados_referem_se,rede_estabelecimento,cnes,endereco,triagem_clinica_total_doacao_espontanea_aptos,triagem_clinica_total_doacao_espontanea_inaptos,triagem_clinica_total_doacao_reposicao_aptos,triagem_clinica_total_doacao_reposicao_inaptos,triagem_clinica_total_doacao_autologa_aptos,triagem_clinica_total_doacao_autologa_inaptos,total_doador_primeira_vez_aptos,total_doador_primeira_vez_inaptos,total_doador_repeticao_aptos,total_doador_repeticao_inaptos,total_doador_esporadico_aptos,total_doador_esporadico_inaptos,total_doador_masculino_aptos,total_doador_masculino_inaptos,total_doador_feminino_aptos,total_doador_feminino_inaptos,total_doador_menor_de_18_anos_aptos,total_doador_menor_de_18_anos_inaptos,total_doador_18_ate_29_anos_aptos,total_doador_18_ate_29_anos_inaptos,total_doador_acima_de_29_anos_aptos,total_doador_acima_de_29_anos_inaptos,total_candidatos_inaptos_anemia_masculino,total_candidatos_inaptos_anemia_feminino,total_candidatos_inaptos_anemia_total,total_candidatos_inaptos_hipertensao_masculino,total_candidatos_inaptos_hipertensao_feminino,total_candidatos_inaptos_hipertensao_total,total_candidatos_inaptos_hipotensao_masculino,total_candidatos_inaptos_hipotensao_feminino,total_candidatos_inaptos_hipotensao_total,total_candidatos_inaptos_alcoolismo_masculino,total_candidatos_inaptos_alcoolismo_feminino,total_candidatos_inaptos_alcoolismo_total,total_candidatos_inaptos_comportamento_risco_dst_masculino,total_candidatos_inaptos_comportamento_risco_dst_feminino,total_candidatos_inaptos_comportamento_risco_dst_total,total_candidatos_inaptos_uso_drogas_masculino,total_candidatos_inaptos_uso_drogas_feminino,total_candidatos_inaptos_uso_drogas_total,total_candidatos_inaptos_hepatite_masculino,total_candidatos_inaptos_hepatite_feminino,total_candidatos_inaptos_hepatite_total,total_candidatos_inaptos_doenca_chagas_masculino,total_candidatos_inaptos_doenca_chagas_feminino,total_candidatos_inaptos_doenca_chagas_total,total_candidatos_inaptos_malaria_masculino,total_candidatos_inaptos_malaria_feminino,total_candidatos_inaptos_malaria_total,total_candidatos_inaptos_outras_masculino,total_candidatos_inaptos_outras_feminino,total_candidatos_inaptos_outras_total,coleta_total_candidatos_desistentes,total_interrupcoes_coleta_dificuldade_puncao_venosa,total_interrupcoes_coleta_reacao_vagal,total_interrupcoes_coleta_outros_motivos,total_coletas_sangue_total,total_coletas_aferese,hemoprod_1_observacoes,exames_triagem_doenca_doenca_chagas_amostras_testadas,exames_triagem_doenca_doenca_chagas_amostras_reagentes,exames_triagem_doenca_hiv_amostras_testadas,exames_triagem_doenca_hiv_amostras_reagentes,exames_triagem_doenca_sifilis_amostras_testadas,`exames_triagem_doenca_sifilis_amostras_reagentes,exames_triagem_doenca_hepatite_b_hbs_ag_amostras_testadas,exames_triagem_doenca_hepatite_b_hbs_ag_amostras_reagentes,exames_triagem_doenca_hepatite_b_anti_hbc_amostras_testadas,exames_triagem_doenca_hepatite_b_anti_hbc_amostras_reagentes,exames_triagem_doenca_hepatite_c_amostras_testadas,exames_triagem_doenca_hepatite_c_amostras_reagentes,exames_triagem_doenca_htlv_i_ii_amostras_testadas,exames_triagem_doenca_htlv_i_ii_amostras_reagentes,exames_triagem_doenca_malaria_amostras_testadas,exames_triagem_doenca_malaria_amostras_reagentes,exames_triagem_doenca_hbv_teste_nat_amostras_testadas,exames_triagem_doenca_hbv_teste_nat_amostras_reagentes,exames_triagem_doenca_hcv_teste_nat_amostras_testadas,exames_triagem_doenca_hcv_teste_nat_amostras_reagentes,exames_triagem_doenca_hiv_teste_nat_amostras_testadas,exames_triagem_doenca_hiv_teste_nat_amostras_reagentes,imunohematologia_a_positivo_doador,imunohematologia_a_positivo_receptor,imunohe


Amostra dos dados únicos (os mais recentes para cada chave):


,id,data_envio,ultima_pagina,idioma_inicial,semente,codigo_acesso,data_inicio,data_ultima_acao,ip,identificacao_dado,tipo_envio,ano_referencia,periodo_referencia,identificacao_estabelecimento,municipio,razao_social_nome_fantasia,razao_social_nome_fantasia_outros,cnpj,tipo_estabelecimento,natureza_estabelecimento,dados_informados_referem_se,rede_estabelecimento,cnes,endereco,triagem_clinica_total_doacao_espontanea_aptos,triagem_clinica_total_doacao_espontanea_inaptos,triagem_clinica_total_doacao_reposicao_aptos,triagem_clinica_total_doacao_reposicao_inaptos,triagem_clinica_total_doacao_autologa_aptos,triagem_clinica_total_doacao_autologa_inaptos,total_doador_primeira_vez_aptos,total_doador_primeira_vez_inaptos,total_doador_repeticao_aptos,total_doador_repeticao_inaptos,total_doador_esporadico_aptos,total_doador_esporadico_inaptos,total_doador_masculino_aptos,total_doador_masculino_inaptos,total_doador_feminino_aptos,total_doador_feminino_inaptos,total_doador_menor_de_18_anos_aptos,total_doador_menor_de_18_anos_inaptos,total_doador_18_ate_29_anos_aptos,total_doador_18_ate_29_anos_inaptos,total_doador_acima_de_29_anos_aptos,total_doador_acima_de_29_anos_inaptos,total_candidatos_inaptos_anemia_masculino,total_candidatos_inaptos_anemia_feminino,total_candidatos_inaptos_anemia_total,total_candidatos_inaptos_hipertensao_masculino,total_candidatos_inaptos_hipertensao_feminino,total_candidatos_inaptos_hipertensao_total,total_candidatos_inaptos_hipotensao_masculino,total_candidatos_inaptos_hipotensao_feminino,total_candidatos_inaptos_hipotensao_total,total_candidatos_inaptos_alcoolismo_masculino,total_candidatos_inaptos_alcoolismo_feminino,total_candidatos_inaptos_alcoolismo_total,total_candidatos_inaptos_comportamento_risco_dst_masculino,total_candidatos_inaptos_comportamento_risco_dst_feminino,total_candidatos_inaptos_comportamento_risco_dst_total,total_candidatos_inaptos_uso_drogas_masculino,total_candidatos_inaptos_uso_drogas_feminino,total_candidatos_inaptos_uso_drogas_total,total_candidatos_inaptos_hepatite_masculino,total_candidatos_inaptos_hepatite_feminino,total_candidatos_inaptos_hepatite_total,total_candidatos_inaptos_doenca_chagas_masculino,total_candidatos_inaptos_doenca_chagas_feminino,total_candidatos_inaptos_doenca_chagas_total,total_candidatos_inaptos_malaria_masculino,total_candidatos_inaptos_malaria_feminino,total_candidatos_inaptos_malaria_total,total_candidatos_inaptos_outras_masculino,total_candidatos_inaptos_outras_feminino,total_candidatos_inaptos_outras_total,coleta_total_candidatos_desistentes,total_interrupcoes_coleta_dificuldade_puncao_venosa,total_interrupcoes_coleta_reacao_vagal,total_interrupcoes_coleta_outros_motivos,total_coletas_sangue_total,total_coletas_aferese,hemoprod_1_observacoes,exames_triagem_doenca_doenca_chagas_amostras_testadas,exames_triagem_doenca_doenca_chagas_amostras_reagentes,exames_triagem_doenca_hiv_amostras_testadas,exames_triagem_doenca_hiv_amostras_reagentes,exames_triagem_doenca_sifilis_amostras_testadas,`exames_triagem_doenca_sifilis_amostras_reagentes,exames_triagem_doenca_hepatite_b_hbs_ag_amostras_testadas,exames_triagem_doenca_hepatite_b_hbs_ag_amostras_reagentes,exames_triagem_doenca_hepatite_b_anti_hbc_amostras_testadas,exames_triagem_doenca_hepatite_b_anti_hbc_amostras_reagentes,exames_triagem_doenca_hepatite_c_amostras_testadas,exames_triagem_doenca_hepatite_c_amostras_reagentes,exames_triagem_doenca_htlv_i_ii_amostras_testadas,exames_triagem_doenca_htlv_i_ii_amostras_reagentes,exames_triagem_doenca_malaria_amostras_testadas,exames_triagem_doenca_malaria_amostras_reagentes,exames_triagem_doenca_hbv_teste_nat_amostras_testadas,exames_triagem_doenca_hbv_teste_nat_amostras_reagentes,exames_triagem_doenca_hcv_teste_nat_amostras_testadas,exames_triagem_doenca_hcv_teste_nat_amostras_reagentes,exames_triagem_doenca_hiv_teste_nat_amostras_testadas,exames_triagem_doenca_hiv_teste_nat_amostras_reagentes,imunohematologia_a_positivo_doador,imunohematologia_a_positivo_receptor,imunohe

In [44]:
hemoprod_df_deduplicado.to_excel('dados_processados/hemoprod_df.xlsx', index=False)

## Hemoprod Espirito Santo

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)


In [ ]:
import os
import pandas as pd

dados_brutos_path = 'dados_brutos'

arquivo_dados_path = os.path.join(dados_brutos_path, 'Hemoprod_ES.xlsx')
nome_planilha = 'HEMOPROD - ESPIRITOSANTO'

dicionario_path_es = ('./dicionario_colunas_269.xlsx')
# dicionario_path_ap = ('./dicionario_colunas_270.xlsx')

dicionario_es = pd.read_excel(dicionario_path_es, sheet_name='Sheet1')

# --- 2. Carregue os dados e o dicionário ---
try:
    # Carrega o arquivo de dados
    hemoprod_es = pd.read_excel(arquivo_dados_path, sheet_name=nome_planilha)
    print("Arquivo de dados carregado com sucesso.")
    print(f"Número de colunas original: {len(hemoprod_es.columns)}")

    # Carrega o arquivo de dicionário
    dicionario = pd.read_excel(dicionario_path_es)
    print("Arquivo de dicionário carregado com sucesso.")

    # --- 3. Extraia a lista de novos nomes ---
    # Pega os valores da coluna 'nome_sql' e converte para uma lista
    novos_nomes = dicionario['nome_sql'].tolist()
    print(f"Número de novos nomes no dicionário: {len(novos_nomes)}")

    # --- 4. Verificação de segurança (MUITO IMPORTANTE) ---
    # Garante que o número de colunas é o mesmo antes de renomear
    if len(hemoprod_es.columns) == len(novos_nomes):
        print("\nO número de colunas corresponde. Renomeando...")
        
        # --- 5. Substitua os nomes das colunas ---
        # Esta é a linha principal que faz a substituição direta
        hemoprod_es.columns = novos_nomes
        
        print("Colunas renomeadas com sucesso!")
        
        # --- 6. Verifique o resultado ---
        print("\nInformações do DataFrame com as novas colunas:")
        hemoprod_es.info()
        
        print("\nAs 5 primeiras linhas com as novas colunas:")
        display(hemoprod_es.head())

    else:
        # Mensagem de erro se o número de colunas for diferente
        print("\n--- ERRO ---")
        print("A renomeação foi cancelada. O número de colunas no arquivo de dados não é igual ao número de nomes no dicionário.")
        print(f"Colunas no arquivo de dados: {len(hemoprod_es.columns)}")
        print(f"Nomes no dicionário: {len(novos_nomes)}")

except FileNotFoundError as e:
    print(f"\nErro de arquivo não encontrado: {e}")
except KeyError as e:
    print(f"\nErro de coluna não encontrada: {e}. Verifique se a coluna 'nome_sql' existe no seu arquivo de dicionário.")
except Exception as e:
    print(f"\nOcorreu um erro inesperado: {e}")



Arquivo de dados carregado com sucesso.
Número de colunas original: 257
Arquivo de dicionário carregado com sucesso.
Número de novos nomes no dicionário: 270

--- ERRO ---
A renomeação foi cancelada. O número de colunas no arquivo de dados não é igual ao número de nomes no dicionário.
Colunas no arquivo de dados: 257
Nomes no dicionário: 270


In [51]:
arquivo_dados_path = os.path.join(dados_brutos_path, 'Hemoprod_CE.xlsx')
nome_planilha = 'Planilha1'
hemoprod_ce1 = pd.read_excel(arquivo_dados_path, sheet_name=nome_planilha)

colunas_padrao = set(hemoprod_ce1.columns)
colunas_estado = set(hemoprod_es.columns)

colunas_faltantes = list(colunas_padrao - colunas_estado)

colunas_a_mais = list(colunas_estado - colunas_padrao)

print(f"Número de colunas faltantes: {len(colunas_faltantes)}")
print(f"Colunas faltantes (a serem adicionadas): {colunas_faltantes}")
print("-" * 30)
print(f"Número de colunas a mais: {len(colunas_a_mais)}")
print(f"Colunas a mais (a serem dropadas): {colunas_a_mais}")


Número de colunas faltantes: 15
Colunas faltantes (a serem adicionadas): ['6. Produção Hemoterápica  6.1(a) Entradas\xa0  [Concentrado de Plaquetas de Aférese][Recebidas]', 'Cite os estabelecimentos que compõem a rede  Informe o Tipo de Estabelecimento, o Nome Fantasia\xa0e o Município de localização de cada um. ', '6.1(b) Perdas  [Concentrado de Plaquetas de Aférese][Outros motivos]', 'Ano de referência\xa0 ', '6. Produção Hemoterápica  6.1(a) Entradas\xa0  [Concentrado de Plaquetas de Aférese][Devolvidas]', '6.1(b) Perdas  [Concentrado de Plaquetas de Aférese][Validade]', '6.1(d) Distribuição para outros serviços  \xa0  [Concentrado de Plaquetas de Aférese][Total]', '6.1(c) Tranfusões  [Concentrado de Plaquetas de Aférese][Total]', '6.1(d) Distribuição para outros serviços  \xa0  [Concentrado de Plaquetas de Aférese][Com exame pré-transfusional ]', '6. Produção Hemoterápica  6.1(a) Entradas\xa0  [Concentrado de Plaquetas de Aférese][Produzidas]', '6.1(d) Distribuição para outros serv

In [52]:
colunas_a_mais = ['Período de referência [Outros]', 'Município [Outros]', 'URL de referência']

# Conta quantos valores *não são nulos* em cada uma dessas colunas
contagem_nao_nulos = hemoprod_es[colunas_a_mais].count()

print("Contagem de valores NÃO-NULOS (preenchidos) por coluna:")
print(contagem_nao_nulos)

Contagem de valores NÃO-NULOS (preenchidos) por coluna:
Período de referência [Outros]     19
Município [Outros]                  0
URL de referência                 610
dtype: int64


In [53]:
if colunas_a_mais:
    hemoprod_es = hemoprod_es.drop(columns=colunas_a_mais, errors='ignore')
    print(f"Colunas a mais removidas: {colunas_a_mais}")
else:
    print("Nenhuma coluna para remover.")

# 2. ADICIONAR as colunas faltantes e preenchê-las com zero (0)
for coluna in colunas_faltantes:
    # Cria a nova coluna no hemoprod_es preenchida com 0
    hemoprod_es[coluna] = 0

print(f"Colunas faltantes adicionadas e zeradas: {colunas_faltantes}")

# 3. REORDENAR as colunas do hemoprod_es na mesma ordem do df_padrao
# A lista de colunas do df_padrao agora é o nosso gabarito final
ordem_padrao = hemoprod_ce1.columns.tolist()

# Reorganiza as colunas do hemoprod_es
hemoprod_es = hemoprod_es[ordem_padrao]

print("-" * 50)
print("Processo concluído:")
print(f"Total de colunas após o ajuste: {len(hemoprod_es.columns)}")
print("A ordem das colunas no df_estado agora corresponde à ordem do df_padrao.")

Colunas a mais removidas: ['Período de referência [Outros]', 'Município [Outros]', 'URL de referência']
Colunas faltantes adicionadas e zeradas: ['6. Produção Hemoterápica  6.1(a) Entradas\xa0  [Concentrado de Plaquetas de Aférese][Recebidas]', 'Cite os estabelecimentos que compõem a rede  Informe o Tipo de Estabelecimento, o Nome Fantasia\xa0e o Município de localização de cada um. ', '6.1(b) Perdas  [Concentrado de Plaquetas de Aférese][Outros motivos]', 'Ano de referência\xa0 ', '6. Produção Hemoterápica  6.1(a) Entradas\xa0  [Concentrado de Plaquetas de Aférese][Devolvidas]', '6.1(b) Perdas  [Concentrado de Plaquetas de Aférese][Validade]', '6.1(d) Distribuição para outros serviços  \xa0  [Concentrado de Plaquetas de Aférese][Total]', '6.1(c) Tranfusões  [Concentrado de Plaquetas de Aférese][Total]', '6.1(d) Distribuição para outros serviços  \xa0  [Concentrado de Plaquetas de Aférese][Com exame pré-transfusional ]', '6. Produção Hemoterápica  6.1(a) Entradas\xa0  [Concentrado de P

In [54]:
# Mapeia as colunas, limpando espaços em branco no início e fim
# e substituindo o caractere \xa0 por um espaço normal.
novos_nomes = {col: col.strip().replace('\xa0', ' ') for col in hemoprod_es.columns}

# Aplica a renomeação
hemoprod_es = hemoprod_es.rename(columns=novos_nomes)

print("Nomes das colunas limpos e padronizados.")

Nomes das colunas limpos e padronizados.


In [55]:
# Nomes das colunas após a limpeza (ajuste se a coluna original tiver nome diferente)
coluna_periodo_limpa = 'Período de referência'
coluna_ano_limpa = 'Ano de referência' # Ou apenas 'Ano de referencia'

# Nota: Certifique-se de que a sua coluna original do período esteja na ordem correta,
# caso contrário, a lógica de reordenação no final fará o trabalho.

# 1. Criar ou sobrescrever a coluna 'Ano de referencia' (limpa)
# O split divide a string e o str[1] pega o ano
hemoprod_es[coluna_ano_limpa] = hemoprod_es[coluna_periodo_limpa].astype(str).str.split('/').str[1]

# 2. Atualizar a coluna original 'Período de referência' (para ter só o mês)
# O split divide a string e o str[0] pega o mês
hemoprod_es[coluna_periodo_limpa] = hemoprod_es[coluna_periodo_limpa].astype(str).str.split('/').str[0]

print("Separação de mês e ano concluída nas colunas limpas.")
print(hemoprod_es[[coluna_periodo_limpa, coluna_ano_limpa]].head())

Separação de mês e ano concluída nas colunas limpas.
  Período de referência Ano de referência
0               Outubro              2022
1               Outubro              2022
2               Outubro              2022
3               Outubro              2022
4              Novembro              2022


In [56]:
set(hemoprod_es.columns)

{'2. Triagem Clínica  2.1 Total de candidatos quanto ao tipo de doação  [Autóloga][Aptos]',
 '2. Triagem Clínica  2.1 Total de candidatos quanto ao tipo de doação  [Autóloga][Inaptos]',
 '2. Triagem Clínica  2.1 Total de candidatos quanto ao tipo de doação  [Espontânea][Aptos]',
 '2. Triagem Clínica  2.1 Total de candidatos quanto ao tipo de doação  [Espontânea][Inaptos]',
 '2. Triagem Clínica  2.1 Total de candidatos quanto ao tipo de doação  [Reposição][Aptos]',
 '2. Triagem Clínica  2.1 Total de candidatos quanto ao tipo de doação  [Reposição][Inaptos]',
 '2.2 Total de candidatos quanto ao tipo de doador  [Esporádico][Aptos]',
 '2.2 Total de candidatos quanto ao tipo de doador  [Esporádico][Inaptos]',
 '2.2 Total de candidatos quanto ao tipo de doador  [Primeira vez][Aptos]',
 '2.2 Total de candidatos quanto ao tipo de doador  [Primeira vez][Inaptos]',
 '2.2 Total de candidatos quanto ao tipo de doador  [Repetição][Aptos]',
 '2.2 Total de candidatos quanto ao tipo de doador  [Repeti

In [57]:
hemoprod_es.head()

,ID da resposta,Data de envio,Última página,Idioma inicial,Semente,Código de acesso,Data de início,Data da última ação,Endereço IP,IDENTIFICAÇÃO DO DADO,"Tipo de Informação Antes de responder ao formulário, declare o tipo de informação que será inserida.",Ano de referência,Período de referência,IDENTIFICAÇÃO DO ESTABELECIMENTO,Município,Razão Social - Nome Fantasia,Razão Social - Nome Fantasia [Outros],CNPJ,Tipo de estabelecimento,Natureza do estabelecimento,Os dados informados referem-se à um(a):,"Cite os estabelecimentos que compõem a rede Informe o Tipo de Estabelecimento, o Nome Fantasia e o Município de localização de cada um.",CNES - Cadastro Nacional de Estabelecimentos de Saúde,Endereço,2. Triagem Clínica 2.1 Total de candidatos quanto ao tipo de doação [Espontânea][Aptos],2. Triagem Clínica 2.1 Total de candidatos quanto ao tipo de doação [Espontânea][Inaptos],2. Triagem Clínica 2.1 Total de candidatos quanto ao tipo de doação [Reposição][Aptos],2. Triagem Clínica 2.1 Total de candidatos quanto ao tipo de doação [Reposição][Inaptos],2. Triagem Clínica 2.1 Total de candidatos quanto ao tipo de doação [Autóloga][Aptos],2. Triagem Clínica 2.1 Total de candidatos quanto ao tipo de doação [Autóloga][Inaptos],2.2 Total de candidatos quanto ao tipo de doador [Primeira vez][Aptos],2.2 Total de candidatos quanto ao tipo de doador [Primeira vez][Inaptos],2.2 Total de candidatos quanto ao tipo de doador [Repetição][Aptos],2.2 Total de candidatos quanto ao tipo de doador [Repetição][Inaptos],2.2 Total de candidatos quanto ao tipo de doador [Esporádico][Aptos],2.2 Total de candidatos quanto ao tipo de doador [Esporádico][Inaptos],2.3 Total de candidatos quanto ao gênero do doador [Masculino][Aptos],2.3 Total de candidatos quanto ao gênero do doador [Masculino][Inaptos],2.3 Total de candidatos quanto ao gênero do doador [Feminino][Aptos],2.3 Total de candidatos quanto ao gênero do doador [Feminino][Inaptos],2.4 Total de candidatos quanto a idade do doador [Menor de 18 anos][Aptos],2.4 Total de candidatos quanto a idade do doador [Menor de 18 anos][Inaptos],2.4 Total de candidatos quanto a idade do doador [18 até 29 anos][Aptos],2.4 Total de candidatos quanto a idade do doador [18 até 29 anos][Inaptos],2.4 Total de candidatos quanto a idade do doador [Acima de 29 anos][Aptos],2.4 Total de candidatos quanto a idade do doador [Acima de 29 anos][Inaptos],2.5 Total de candidatos inaptos por motivo de inaptidão e por gênero [Anemia][Masculino],2.5 Total de candidatos inaptos por motivo de inaptidão e por gênero [Anemia][Feminino],2.5 Total de candidatos inaptos por motivo de inaptidão e por gênero [Anemia][Total],2.5 Total de candidatos inaptos por motivo de inaptidão e por gênero [Hipertensão][Masculino],2.5 Total de candidatos inaptos por motivo de inaptidão e por gênero [Hipertensão][Feminino],2.5 Total de candidatos inaptos por motivo de inaptidão e por gênero [Hipertensão][Total],2.5 Total de candidatos inaptos por motivo de inaptidão e por gênero [Hipotensão][Masculino],2.5 Total de candidatos inaptos por motivo de inaptidão e por gênero [Hipotensão][Feminino],2.5 Total de candidatos inaptos por motivo de inaptidão e por gênero [Hipotensão][Total],2.5 Total de candidatos inaptos por motivo de inaptidão e por gênero [Alcoolismo][Masculino],2.5 Total de candidatos inaptos por motivo de inaptidão e por gênero [Alcoolismo][Feminino],2.5 Total de candidatos inaptos por motivo de inaptidão e por gênero [Alcoolismo][Total],2.5 Total de candidatos inaptos por motivo de inaptidão e por gênero [Comportamento de risco para DST][Masculino],2.5 Total de candidatos inaptos por motivo de inaptidão e por gênero [Comportamento de risco para DST][Feminino],2.5 Total de candidatos inaptos por motivo de inaptidão e por gênero [Comportamento de risco para DST][Total],2.5 Total de candidatos inaptos por motivo de inaptidão e por gênero [Uso de drogas][Masculino],2.5 Total de candidatos inaptos por motivo de inaptidão e por gênero [Uso de drogas][Feminin

In [19]:
hemoprod_es.head()

,ID da resposta,Data de envio,Última página,Idioma inicial,Semente,Código de acesso,Data de início,Data da última ação,Endereço IP,IDENTIFICAÇÃO DO DADO,"Tipo de Informação Antes de responder ao formulário, declare o tipo de informação que será inserida.",Ano de referência,Período de referência,IDENTIFICAÇÃO DO ESTABELECIMENTO,Município,Razão Social - Nome Fantasia,Razão Social - Nome Fantasia [Outros],CNPJ,Tipo de estabelecimento,Natureza do estabelecimento,Os dados informados referem-se à um(a):,"Cite os estabelecimentos que compõem a rede Informe o Tipo de Estabelecimento, o Nome Fantasia e o Município de localização de cada um.",CNES - Cadastro Nacional de Estabelecimentos de Saúde,Endereço,2. Triagem Clínica 2.1 Total de candidatos quanto ao tipo de doação [Espontânea][Aptos],2. Triagem Clínica 2.1 Total de candidatos quanto ao tipo de doação [Espontânea][Inaptos],2. Triagem Clínica 2.1 Total de candidatos quanto ao tipo de doação [Reposição][Aptos],2. Triagem Clínica 2.1 Total de candidatos quanto ao tipo de doação [Reposição][Inaptos],2. Triagem Clínica 2.1 Total de candidatos quanto ao tipo de doação [Autóloga][Aptos],2. Triagem Clínica 2.1 Total de candidatos quanto ao tipo de doação [Autóloga][Inaptos],2.2 Total de candidatos quanto ao tipo de doador [Primeira vez][Aptos],2.2 Total de candidatos quanto ao tipo de doador [Primeira vez][Inaptos],2.2 Total de candidatos quanto ao tipo de doador [Repetição][Aptos],2.2 Total de candidatos quanto ao tipo de doador [Repetição][Inaptos],2.2 Total de candidatos quanto ao tipo de doador [Esporádico][Aptos],2.2 Total de candidatos quanto ao tipo de doador [Esporádico][Inaptos],2.3 Total de candidatos quanto ao gênero do doador [Masculino][Aptos],2.3 Total de candidatos quanto ao gênero do doador [Masculino][Inaptos],2.3 Total de candidatos quanto ao gênero do doador [Feminino][Aptos],2.3 Total de candidatos quanto ao gênero do doador [Feminino][Inaptos],2.4 Total de candidatos quanto a idade do doador [Menor de 18 anos][Aptos],2.4 Total de candidatos quanto a idade do doador [Menor de 18 anos][Inaptos],2.4 Total de candidatos quanto a idade do doador [18 até 29 anos][Aptos],2.4 Total de candidatos quanto a idade do doador [18 até 29 anos][Inaptos],2.4 Total de candidatos quanto a idade do doador [Acima de 29 anos][Aptos],2.4 Total de candidatos quanto a idade do doador [Acima de 29 anos][Inaptos],2.5 Total de candidatos inaptos por motivo de inaptidão e por gênero [Anemia][Masculino],2.5 Total de candidatos inaptos por motivo de inaptidão e por gênero [Anemia][Feminino],2.5 Total de candidatos inaptos por motivo de inaptidão e por gênero [Anemia][Total],2.5 Total de candidatos inaptos por motivo de inaptidão e por gênero [Hipertensão][Masculino],2.5 Total de candidatos inaptos por motivo de inaptidão e por gênero [Hipertensão][Feminino],2.5 Total de candidatos inaptos por motivo de inaptidão e por gênero [Hipertensão][Total],2.5 Total de candidatos inaptos por motivo de inaptidão e por gênero [Hipotensão][Masculino],2.5 Total de candidatos inaptos por motivo de inaptidão e por gênero [Hipotensão][Feminino],2.5 Total de candidatos inaptos por motivo de inaptidão e por gênero [Hipotensão][Total],2.5 Total de candidatos inaptos por motivo de inaptidão e por gênero [Alcoolismo][Masculino],2.5 Total de candidatos inaptos por motivo de inaptidão e por gênero [Alcoolismo][Feminino],2.5 Total de candidatos inaptos por motivo de inaptidão e por gênero [Alcoolismo][Total],2.5 Total de candidatos inaptos por motivo de inaptidão e por gênero [Comportamento de risco para DST][Masculino],2.5 Total de candidatos inaptos por motivo de inaptidão e por gênero [Comportamento de risco para DST][Feminino],2.5 Total de candidatos inaptos por motivo de inaptidão e por gênero [Comportamento de risco para DST][Total],2.5 Total de candidatos inaptos por motivo de inaptidão e por gênero [Uso de drogas][Masculino],2.5 Total de candidatos inaptos por motivo de inaptidão e por gênero [Uso de drogas][Feminin

In [61]:
# Carrega o arquivo de dicionário
dicionario_path_es = ('./dicionario_colunas_269.xlsx')
dicionario = pd.read_excel(dicionario_path_es)
print("Arquivo de dicionário carregado com sucesso.")

# --- 3. Extraia a lista de novos nomes ---
# Pega os valores da coluna 'nome_sql' e converte para uma lista
novos_nomes = dicionario['nome_sql'].tolist()
print(f"Número de novos nomes no dicionário: {len(novos_nomes)}")

# --- 4. Verificação de segurança (MUITO IMPORTANTE) ---
# Garante que o número de colunas é o mesmo antes de renomear
if len(hemoprod_es.columns) == len(novos_nomes):
    print("\nO número de colunas corresponde. Renomeando...")
    
    # --- 5. Substitua os nomes das colunas ---
    # Esta é a linha principal que faz a substituição direta
    hemoprod_es.columns = novos_nomes
    
    print("Colunas renomeadas com sucesso!")
    
    # --- 6. Verifique o resultado ---
    print("\nInformações do DataFrame com as novas colunas:")
    hemoprod_es.info()
    
    print("\nAs 5 primeiras linhas com as novas colunas:")
    display(hemoprod_es.head())

else:
    # Mensagem de erro se o número de colunas for diferente
    print("\n--- ERRO ---")
    print("A renomeação foi cancelada. O número de colunas no arquivo de dados não é igual ao número de nomes no dicionário.")
    print(f"Colunas no arquivo de dados: {len(hemoprod_es.columns)}")
    print(f"Nomes no dicionário: {len(novos_nomes)}")


Arquivo de dicionário carregado com sucesso.
Número de novos nomes no dicionário: 269

O número de colunas corresponde. Renomeando...
Colunas renomeadas com sucesso!

Informações do DataFrame com as novas colunas:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1827 entries, 0 to 1826
Columns: 269 entries, id to hemoprod_3_observacoes
dtypes: float64(237), int64(16), object(16)
memory usage: 3.7+ MB

As 5 primeiras linhas com as novas colunas:


,id,data_envio,ultima_pagina,idioma_inicial,semente,codigo_acesso,data_inicio,data_ultima_acao,ip,identificacao_dado,tipo_envio,ano_referencia,periodo_referencia,identificacao_estabelecimento,municipio,razao_social_nome_fantasia,razao_social_nome_fantasia_outros,cnpj,tipo_estabelecimento,natureza_estabelecimento,dados_informados_referem_se,rede_estabelecimento,cnes,endereco,triagem_clinica_total_doacao_espontanea_aptos,triagem_clinica_total_doacao_espontanea_inaptos,triagem_clinica_total_doacao_reposicao_aptos,triagem_clinica_total_doacao_reposicao_inaptos,triagem_clinica_total_doacao_autologa_aptos,triagem_clinica_total_doacao_autologa_inaptos,total_doador_primeira_vez_aptos,total_doador_primeira_vez_inaptos,total_doador_repeticao_aptos,total_doador_repeticao_inaptos,total_doador_esporadico_aptos,total_doador_esporadico_inaptos,total_doador_masculino_aptos,total_doador_masculino_inaptos,total_doador_feminino_aptos,total_doador_feminino_inaptos,total_doador_menor_de_18_anos_aptos,total_doador_menor_de_18_anos_inaptos,total_doador_18_ate_29_anos_aptos,total_doador_18_ate_29_anos_inaptos,total_doador_acima_de_29_anos_aptos,total_doador_acima_de_29_anos_inaptos,total_candidatos_inaptos_anemia_masculino,total_candidatos_inaptos_anemia_feminino,total_candidatos_inaptos_anemia_total,total_candidatos_inaptos_hipertensao_masculino,total_candidatos_inaptos_hipertensao_feminino,total_candidatos_inaptos_hipertensao_total,total_candidatos_inaptos_hipotensao_masculino,total_candidatos_inaptos_hipotensao_feminino,total_candidatos_inaptos_hipotensao_total,total_candidatos_inaptos_alcoolismo_masculino,total_candidatos_inaptos_alcoolismo_feminino,total_candidatos_inaptos_alcoolismo_total,total_candidatos_inaptos_comportamento_risco_dst_masculino,total_candidatos_inaptos_comportamento_risco_dst_feminino,total_candidatos_inaptos_comportamento_risco_dst_total,total_candidatos_inaptos_uso_drogas_masculino,total_candidatos_inaptos_uso_drogas_feminino,total_candidatos_inaptos_uso_drogas_total,total_candidatos_inaptos_hepatite_masculino,total_candidatos_inaptos_hepatite_feminino,total_candidatos_inaptos_hepatite_total,total_candidatos_inaptos_doenca_chagas_masculino,total_candidatos_inaptos_doenca_chagas_feminino,total_candidatos_inaptos_doenca_chagas_total,total_candidatos_inaptos_malaria_masculino,total_candidatos_inaptos_malaria_feminino,total_candidatos_inaptos_malaria_total,total_candidatos_inaptos_outras_masculino,total_candidatos_inaptos_outras_feminino,total_candidatos_inaptos_outras_total,coleta_total_candidatos_desistentes,total_interrupcoes_coleta_dificuldade_puncao_venosa,total_interrupcoes_coleta_reacao_vagal,total_interrupcoes_coleta_outros_motivos,total_coletas_sangue_total,total_coletas_aferese,hemoprod_1_observacoes,exames_triagem_doenca_doenca_chagas_amostras_testadas,exames_triagem_doenca_doenca_chagas_amostras_reagentes,exames_triagem_doenca_hiv_amostras_testadas,exames_triagem_doenca_hiv_amostras_reagentes,exames_triagem_doenca_sifilis_amostras_testadas,`exames_triagem_doenca_sifilis_amostras_reagentes,exames_triagem_doenca_hepatite_b_hbs_ag_amostras_testadas,exames_triagem_doenca_hepatite_b_hbs_ag_amostras_reagentes,exames_triagem_doenca_hepatite_b_anti_hbc_amostras_testadas,exames_triagem_doenca_hepatite_b_anti_hbc_amostras_reagentes,exames_triagem_doenca_hepatite_c_amostras_testadas,exames_triagem_doenca_hepatite_c_amostras_reagentes,exames_triagem_doenca_htlv_i_ii_amostras_testadas,exames_triagem_doenca_htlv_i_ii_amostras_reagentes,exames_triagem_doenca_malaria_amostras_testadas,exames_triagem_doenca_malaria_amostras_reagentes,exames_triagem_doenca_hbv_teste_nat_amostras_testadas,exames_triagem_doenca_hbv_teste_nat_amostras_reagentes,exames_triagem_doenca_hcv_teste_nat_amostras_testadas,exames_triagem_doenca_hcv_teste_nat_amostras_reagentes,exames_triagem_doenca_hiv_teste_nat_amostras_testadas,exames_triagem_doenca_hiv_teste_nat_amostras_reagentes,imunohematologia_a_positivo_doador,imunohematologia_a_positivo_receptor,imunohe

In [62]:
print("--- Nomes das Colunas (um por linha) ---")
for coluna in hemoprod_es.columns:
    print(coluna + ",") 

--- Nomes das Colunas (um por linha) ---
id,
data_envio,
ultima_pagina,
idioma_inicial,
semente,
codigo_acesso,
data_inicio,
data_ultima_acao,
ip,
identificacao_dado,
tipo_envio,
ano_referencia,
periodo_referencia,
identificacao_estabelecimento,
municipio,
razao_social_nome_fantasia,
razao_social_nome_fantasia_outros,
cnpj,
tipo_estabelecimento,
natureza_estabelecimento,
dados_informados_referem_se,
rede_estabelecimento,
cnes,
endereco,
triagem_clinica_total_doacao_espontanea_aptos,
triagem_clinica_total_doacao_espontanea_inaptos,
triagem_clinica_total_doacao_reposicao_aptos,
triagem_clinica_total_doacao_reposicao_inaptos,
triagem_clinica_total_doacao_autologa_aptos,
triagem_clinica_total_doacao_autologa_inaptos,
total_doador_primeira_vez_aptos,
total_doador_primeira_vez_inaptos,
total_doador_repeticao_aptos,
total_doador_repeticao_inaptos,
total_doador_esporadico_aptos,
total_doador_esporadico_inaptos,
total_doador_masculino_aptos,
total_doador_masculino_inaptos,
total_doador_feminino

In [63]:
import numpy as np
import pandas as pd

# Suponha que esta é a sua lista de colunas que devem ser vazias, e não 0
colunas_para_vazias = [
    'dados_informados_referem_se',
    'rede_estabelecimento',
    
]

# Itera sobre cada coluna na lista
for coluna in colunas_para_vazias:
    # Verifica se a coluna existe no DataFrame antes de tentar fazer a substituição
    if coluna in hemoprod_es.columns:
        # Substitui todos os valores '0' por pd.NA (valor ausente/vazio)
        hemoprod_es[coluna] = hemoprod_es[coluna].replace(0, pd.NA)
        print(f"Coluna '{coluna}' corrigida: 0 substituído por pd.NA.")
    else:
        print(f"Atenção: Coluna '{coluna}' não encontrada no DataFrame.")

print("\nSubstituição concluída.")

Coluna 'dados_informados_referem_se' corrigida: 0 substituído por pd.NA.
Coluna 'rede_estabelecimento' corrigida: 0 substituído por pd.NA.

Substituição concluída.


In [67]:
# Coluna a ser verificada (já limpa e separada)
coluna_mes = 'periodo_referencia'

# O padrão Regex r'.*\d+.*' verifica se a string contém UM OU MAIS dígitos (0-9)
# O .str.contains() retorna True ou False para cada linha
registros_com_numero = hemoprod_es[coluna_mes].astype(str).str.contains(r'\d+', na=False)

# 1. Contar quantos registros têm números
total_com_numero = registros_com_numero.sum()

print(f"Total de registros na coluna '{coluna_mes}' que contêm números: {total_com_numero}")

if total_com_numero > 0:
    print("-" * 50)
    print("Amostra dos registros que contêm números:")
    
    # 2. Exibir os registros com problema (as 5 primeiras ocorrências)
    linhas_problema = hemoprod_es[registros_com_numero]
    print(linhas_problema[[coluna_mes]].head(55))
    
    # Opcional: Se você quiser ver a linha inteira para entender o contexto
    # print(linhas_problema.head())
    
else:
    print("Nenhum registro encontrado com números na coluna de meses. Os dados parecem limpos.")

Total de registros na coluna 'periodo_referencia' que contêm números: 53
--------------------------------------------------
Amostra dos registros que contêm números:
    periodo_referencia
46    Consolidado 2022
55    Consolidado 2022
57    Consolidado 2022
58    Consolidado 2022
61    Consolidado 2022
66    Consolidado 2022
70    Consolidado 2022
72    Consolidado 2022
80    Consolidado 2022
81    Consolidado 2022
91    Consolidado 2022
94    Consolidado 2022
95    Consolidado 2022
98    Consolidado 2022
103   Consolidado 2022
106   Consolidado 2022
107   Consolidado 2022
108   Consolidado 2022
109   Consolidado 2022
110   Consolidado 2022
111   Consolidado 2022
112   Consolidado 2022
115   Consolidado 2022
116   Consolidado 2022
117   Consolidado 2022
118   Consolidado 2022
119   Consolidado 2022
120   Consolidado 2022
121   Consolidado 2022
122   Consolidado 2022
123   Consolidado 2022
126   Consolidado 2022
127   Consolidado 2022
128   Consolidado 2022
129   Consolidado 2022
130   

In [68]:
import pandas as pd
import numpy as np
# Colunas a serem corrigidas
coluna_periodo = 'periodo_referencia'
coluna_ano = 'ano_referencia'

# 1. Identificar as linhas onde o período ainda contém o ano (e assumimos que é um caso de "Consolidado XXXX")
# Regex para encontrar qualquer texto seguido por 4 dígitos (o ano)
filtro_consolidado = hemoprod_es[coluna_periodo].astype(str).str.contains(r'\d{4}', na=False)

if filtro_consolidado.any():
    print(f"Encontrados {filtro_consolidado.sum()} registros que precisam de correção manual.")
    
    # 2. Extrair o Ano (4 dígitos consecutivos)
    # A função extract() é ideal para isso
    ano_extraido = hemoprod_es.loc[filtro_consolidado, coluna_periodo].str.extract(r'(\d{4})', expand=False)
    
    # 3. Preencher a coluna 'Ano de referencia' com o valor extraído
    hemoprod_es.loc[filtro_consolidado, coluna_ano] = ano_extraido
    
    # 4. Limpar a coluna 'Período de referência'
    # Remove os 4 dígitos e qualquer espaço antes ou depois deles, deixando apenas o texto como "Consolidado"
    hemoprod_es.loc[filtro_consolidado, coluna_periodo] = (
        hemoprod_es.loc[filtro_consolidado, coluna_periodo]
                 .str.replace(r'\s*\d{4}', '', regex=True) # Remove o ano e espaços
                 .str.strip() # Remove qualquer espaço residual no início/fim
    )
    
    print("Correção aplicada com sucesso.")
    print("-" * 50)
    print("Amostra dos registros corrigidos:")
    # Mostrar as 5 primeiras linhas que foram corrigidas
    print(hemoprod_es.loc[filtro_consolidado, [coluna_periodo, coluna_ano]].head())

else:
    print("Nenhum caso de período que ainda contenha o ano foi encontrado. O DataFrame está limpo.")

Encontrados 53 registros que precisam de correção manual.
Correção aplicada com sucesso.
--------------------------------------------------
Amostra dos registros corrigidos:
   periodo_referencia ano_referencia
46        Consolidado           2022
55        Consolidado           2022
57        Consolidado           2022
58        Consolidado           2022
61        Consolidado           2022


In [69]:
# --- 2. Defina as colunas para a chave e para ordenação ---
registros_antes = len(hemoprod_es)
print(f"Total de registros ANTES da remoção de duplicatas: {registros_antes}")
    

colunas_chave = [
        'cnpj', 
        'ano_referencia', 
        'periodo_referencia', 
        'razao_social_nome_fantasia'
    ]

coluna_data = 'data_envio'

# --- 3. Verifique se as colunas necessárias existem ---
colunas_necessarias = colunas_chave + [coluna_data]

if not all(col in hemoprod_es.columns for col in colunas_necessarias):
    print("\n--- ERRO ---")
    print("Uma ou mais colunas necessárias para a deduplicação não foram encontradas.")
    colunas_faltantes = [col for col in colunas_necessarias if col not in hemoprod_es.columns]
    print(f"Colunas necessárias: {colunas_necessarias}")
    print(f"Colunas faltantes no DataFrame: {colunas_faltantes}")
else:
    # --- 4. Prepare a coluna de data e ordene os dados ---
    # Converte a coluna 'data_envio' para datetime para garantir a ordenação correta.
    # 'errors='coerce'' transformará datas inválidas em NaT (Not a Time), que são tratadas como nulas.
    hemoprod_es[coluna_data] = pd.to_datetime(hemoprod_es[coluna_data], errors='coerce')

    # Ordena o DataFrame. Os registros com data de envio mais recente ficarão por último.
    print(f"\nOrdenando os dados por '{coluna_data}'...")
    hemoprod_es_ordenado = hemoprod_es.sort_values(by=coluna_data, ascending=True)

    # --- 5. Identifique e separe os registros duplicados e únicos ---
    # Em vez de usar drop_duplicates() diretamente, vamos usar duplicated()
    # para criar uma máscara booleana.
    # 'keep='last'' marca todas as ocorrências de uma chave como True, EXCETO a última (a mais recente).
    print(f"Identificando duplicatas com cese na chave: {colunas_chave}...")
    mascara_duplicatas = hemoprod_es_ordenado.duplicated(subset=colunas_chave, keep='last')

    # O DataFrame de removidos conterá todas as linhas marcadas como True
    hemoprod_removidos = hemoprod_es_ordenado[mascara_duplicatas]
    
    # O DataFrame deduplicado conterá o INVERSO (~) da máscara (linhas marcadas como False)
    hemoprod_es_deduplicado = hemoprod_es_ordenado[~mascara_duplicatas]
    
    # Agora podemos contar os registros diretamente dos novos DataFrames
    registros_depois = len(hemoprod_es_deduplicado)
    registros_removidos = len(hemoprod_removidos) 
    
    # --- 6. Exice o resultado ---
    print("\n--- Processo Concluído ---")
    print(f"Registros removidos: {registros_removidos}")
    print(f"Total de registros DEPOIS da remoção de duplicatas: {registros_depois}")

    # (Opcional) Exibe a amostra dos removidos
    print("\nAmostra dos dados REMOVIDOS (os mais antigos/duplicados):")
    display(hemoprod_removidos.head(10))

    # Você pode continuar a usar o DataFrame 'hemoprod_ce_deduplicado' para suas análises
    print("\nAmostra dos dados únicos (os mais recentes para cada chave):")
    display(hemoprod_es_deduplicado.head(10))
    
    # Agora você tem o DataFrame 'hemoprod_removidos' salvo

Total de registros ANTES da remoção de duplicatas: 1827

Ordenando os dados por 'data_envio'...
Identificando duplicatas com cese na chave: ['cnpj', 'ano_referencia', 'periodo_referencia', 'razao_social_nome_fantasia']...

--- Processo Concluído ---
Registros removidos: 83
Total de registros DEPOIS da remoção de duplicatas: 1744

Amostra dos dados REMOVIDOS (os mais antigos/duplicados):


,id,data_envio,ultima_pagina,idioma_inicial,semente,codigo_acesso,data_inicio,data_ultima_acao,ip,identificacao_dado,tipo_envio,ano_referencia,periodo_referencia,identificacao_estabelecimento,municipio,razao_social_nome_fantasia,razao_social_nome_fantasia_outros,cnpj,tipo_estabelecimento,natureza_estabelecimento,dados_informados_referem_se,rede_estabelecimento,cnes,endereco,triagem_clinica_total_doacao_espontanea_aptos,triagem_clinica_total_doacao_espontanea_inaptos,triagem_clinica_total_doacao_reposicao_aptos,triagem_clinica_total_doacao_reposicao_inaptos,triagem_clinica_total_doacao_autologa_aptos,triagem_clinica_total_doacao_autologa_inaptos,total_doador_primeira_vez_aptos,total_doador_primeira_vez_inaptos,total_doador_repeticao_aptos,total_doador_repeticao_inaptos,total_doador_esporadico_aptos,total_doador_esporadico_inaptos,total_doador_masculino_aptos,total_doador_masculino_inaptos,total_doador_feminino_aptos,total_doador_feminino_inaptos,total_doador_menor_de_18_anos_aptos,total_doador_menor_de_18_anos_inaptos,total_doador_18_ate_29_anos_aptos,total_doador_18_ate_29_anos_inaptos,total_doador_acima_de_29_anos_aptos,total_doador_acima_de_29_anos_inaptos,total_candidatos_inaptos_anemia_masculino,total_candidatos_inaptos_anemia_feminino,total_candidatos_inaptos_anemia_total,total_candidatos_inaptos_hipertensao_masculino,total_candidatos_inaptos_hipertensao_feminino,total_candidatos_inaptos_hipertensao_total,total_candidatos_inaptos_hipotensao_masculino,total_candidatos_inaptos_hipotensao_feminino,total_candidatos_inaptos_hipotensao_total,total_candidatos_inaptos_alcoolismo_masculino,total_candidatos_inaptos_alcoolismo_feminino,total_candidatos_inaptos_alcoolismo_total,total_candidatos_inaptos_comportamento_risco_dst_masculino,total_candidatos_inaptos_comportamento_risco_dst_feminino,total_candidatos_inaptos_comportamento_risco_dst_total,total_candidatos_inaptos_uso_drogas_masculino,total_candidatos_inaptos_uso_drogas_feminino,total_candidatos_inaptos_uso_drogas_total,total_candidatos_inaptos_hepatite_masculino,total_candidatos_inaptos_hepatite_feminino,total_candidatos_inaptos_hepatite_total,total_candidatos_inaptos_doenca_chagas_masculino,total_candidatos_inaptos_doenca_chagas_feminino,total_candidatos_inaptos_doenca_chagas_total,total_candidatos_inaptos_malaria_masculino,total_candidatos_inaptos_malaria_feminino,total_candidatos_inaptos_malaria_total,total_candidatos_inaptos_outras_masculino,total_candidatos_inaptos_outras_feminino,total_candidatos_inaptos_outras_total,coleta_total_candidatos_desistentes,total_interrupcoes_coleta_dificuldade_puncao_venosa,total_interrupcoes_coleta_reacao_vagal,total_interrupcoes_coleta_outros_motivos,total_coletas_sangue_total,total_coletas_aferese,hemoprod_1_observacoes,exames_triagem_doenca_doenca_chagas_amostras_testadas,exames_triagem_doenca_doenca_chagas_amostras_reagentes,exames_triagem_doenca_hiv_amostras_testadas,exames_triagem_doenca_hiv_amostras_reagentes,exames_triagem_doenca_sifilis_amostras_testadas,`exames_triagem_doenca_sifilis_amostras_reagentes,exames_triagem_doenca_hepatite_b_hbs_ag_amostras_testadas,exames_triagem_doenca_hepatite_b_hbs_ag_amostras_reagentes,exames_triagem_doenca_hepatite_b_anti_hbc_amostras_testadas,exames_triagem_doenca_hepatite_b_anti_hbc_amostras_reagentes,exames_triagem_doenca_hepatite_c_amostras_testadas,exames_triagem_doenca_hepatite_c_amostras_reagentes,exames_triagem_doenca_htlv_i_ii_amostras_testadas,exames_triagem_doenca_htlv_i_ii_amostras_reagentes,exames_triagem_doenca_malaria_amostras_testadas,exames_triagem_doenca_malaria_amostras_reagentes,exames_triagem_doenca_hbv_teste_nat_amostras_testadas,exames_triagem_doenca_hbv_teste_nat_amostras_reagentes,exames_triagem_doenca_hcv_teste_nat_amostras_testadas,exames_triagem_doenca_hcv_teste_nat_amostras_reagentes,exames_triagem_doenca_hiv_teste_nat_amostras_testadas,exames_triagem_doenca_hiv_teste_nat_amostras_reagentes,imunohematologia_a_positivo_doador,imunohematologia_a_positivo_receptor,imunohe


Amostra dos dados únicos (os mais recentes para cada chave):


,id,data_envio,ultima_pagina,idioma_inicial,semente,codigo_acesso,data_inicio,data_ultima_acao,ip,identificacao_dado,tipo_envio,ano_referencia,periodo_referencia,identificacao_estabelecimento,municipio,razao_social_nome_fantasia,razao_social_nome_fantasia_outros,cnpj,tipo_estabelecimento,natureza_estabelecimento,dados_informados_referem_se,rede_estabelecimento,cnes,endereco,triagem_clinica_total_doacao_espontanea_aptos,triagem_clinica_total_doacao_espontanea_inaptos,triagem_clinica_total_doacao_reposicao_aptos,triagem_clinica_total_doacao_reposicao_inaptos,triagem_clinica_total_doacao_autologa_aptos,triagem_clinica_total_doacao_autologa_inaptos,total_doador_primeira_vez_aptos,total_doador_primeira_vez_inaptos,total_doador_repeticao_aptos,total_doador_repeticao_inaptos,total_doador_esporadico_aptos,total_doador_esporadico_inaptos,total_doador_masculino_aptos,total_doador_masculino_inaptos,total_doador_feminino_aptos,total_doador_feminino_inaptos,total_doador_menor_de_18_anos_aptos,total_doador_menor_de_18_anos_inaptos,total_doador_18_ate_29_anos_aptos,total_doador_18_ate_29_anos_inaptos,total_doador_acima_de_29_anos_aptos,total_doador_acima_de_29_anos_inaptos,total_candidatos_inaptos_anemia_masculino,total_candidatos_inaptos_anemia_feminino,total_candidatos_inaptos_anemia_total,total_candidatos_inaptos_hipertensao_masculino,total_candidatos_inaptos_hipertensao_feminino,total_candidatos_inaptos_hipertensao_total,total_candidatos_inaptos_hipotensao_masculino,total_candidatos_inaptos_hipotensao_feminino,total_candidatos_inaptos_hipotensao_total,total_candidatos_inaptos_alcoolismo_masculino,total_candidatos_inaptos_alcoolismo_feminino,total_candidatos_inaptos_alcoolismo_total,total_candidatos_inaptos_comportamento_risco_dst_masculino,total_candidatos_inaptos_comportamento_risco_dst_feminino,total_candidatos_inaptos_comportamento_risco_dst_total,total_candidatos_inaptos_uso_drogas_masculino,total_candidatos_inaptos_uso_drogas_feminino,total_candidatos_inaptos_uso_drogas_total,total_candidatos_inaptos_hepatite_masculino,total_candidatos_inaptos_hepatite_feminino,total_candidatos_inaptos_hepatite_total,total_candidatos_inaptos_doenca_chagas_masculino,total_candidatos_inaptos_doenca_chagas_feminino,total_candidatos_inaptos_doenca_chagas_total,total_candidatos_inaptos_malaria_masculino,total_candidatos_inaptos_malaria_feminino,total_candidatos_inaptos_malaria_total,total_candidatos_inaptos_outras_masculino,total_candidatos_inaptos_outras_feminino,total_candidatos_inaptos_outras_total,coleta_total_candidatos_desistentes,total_interrupcoes_coleta_dificuldade_puncao_venosa,total_interrupcoes_coleta_reacao_vagal,total_interrupcoes_coleta_outros_motivos,total_coletas_sangue_total,total_coletas_aferese,hemoprod_1_observacoes,exames_triagem_doenca_doenca_chagas_amostras_testadas,exames_triagem_doenca_doenca_chagas_amostras_reagentes,exames_triagem_doenca_hiv_amostras_testadas,exames_triagem_doenca_hiv_amostras_reagentes,exames_triagem_doenca_sifilis_amostras_testadas,`exames_triagem_doenca_sifilis_amostras_reagentes,exames_triagem_doenca_hepatite_b_hbs_ag_amostras_testadas,exames_triagem_doenca_hepatite_b_hbs_ag_amostras_reagentes,exames_triagem_doenca_hepatite_b_anti_hbc_amostras_testadas,exames_triagem_doenca_hepatite_b_anti_hbc_amostras_reagentes,exames_triagem_doenca_hepatite_c_amostras_testadas,exames_triagem_doenca_hepatite_c_amostras_reagentes,exames_triagem_doenca_htlv_i_ii_amostras_testadas,exames_triagem_doenca_htlv_i_ii_amostras_reagentes,exames_triagem_doenca_malaria_amostras_testadas,exames_triagem_doenca_malaria_amostras_reagentes,exames_triagem_doenca_hbv_teste_nat_amostras_testadas,exames_triagem_doenca_hbv_teste_nat_amostras_reagentes,exames_triagem_doenca_hcv_teste_nat_amostras_testadas,exames_triagem_doenca_hcv_teste_nat_amostras_reagentes,exames_triagem_doenca_hiv_teste_nat_amostras_testadas,exames_triagem_doenca_hiv_teste_nat_amostras_reagentes,imunohematologia_a_positivo_doador,imunohematologia_a_positivo_receptor,imunohe

In [70]:
hemoprod_es_deduplicado.to_excel('dados_processados/hemoprod_es.xlsx', index=False)

## Hemoprod Goiás

In [83]:
import os
import pandas as pd

dados_brutos_path = 'dados_brutos'

arquivo_dados_path = os.path.join(dados_brutos_path, 'Hemoprod_GO.xlsx')
nome_planilha = 'HEMOPROD - GOIAS'

dicionario_path_go = ('./dicionario_colunas_269.xlsx')
# dicionario_path_ap = ('./dicionario_colunas_270.xlsx')

dicionario_go = pd.read_excel(dicionario_path_go, sheet_name='Sheet1')

# --- 2. Carregue os dados e o dicionário ---
try:
    # Carrega o arquivo de dados
    hemoprod_go = pd.read_excel(arquivo_dados_path, sheet_name=nome_planilha)
    print("Arquivo de dados carregado com sucesso.")
    print(f"Número de colunas original: {len(hemoprod_go.columns)}")

    # Carrega o arquivo de dicionário
    dicionario = pd.read_excel(dicionario_path_go)
    print("Arquivo de dicionário carregado com sucesso.")

    # --- 3. Extraia a lista de novos nomes ---
    # Pega os valores da coluna 'nome_sql' e converte para uma lista
    novos_nomes = dicionario['nome_sql'].tolist()
    print(f"Número de novos nomes no dicionário: {len(novos_nomes)}")

    # --- 4. Verificação de segurança (MUITO IMPORTANTE) ---
    # Garante que o número de colunas é o mesmo antes de renomear
    if len(hemoprod_go.columns) == len(novos_nomes):
        print("\nO número de colunas corresponde. Renomeando...")
        
        # --- 5. Substitua os nomes das colunas ---
        # Esta é a linha principal que faz a substituição direta
        hemoprod_go.columns = novos_nomes
        
        print("Colunas renomeadas com sucesso!")
        
        # --- 6. Verifique o resultado ---
        print("\nInformações do DataFrame com as novas colunas:")
        hemoprod_go.info()
        
        print("\nAs 5 primeiras linhas com as novas colunas:")
        display(hemoprod_go.head())

    else:
        # Mensagem de erro se o número de colunas for diferente
        print("\n--- ERRO ---")
        print("A renomeação foi cancelada. O número de colunas no arquivo de dados não é igual ao número de nomes no dicionário.")
        print(f"Colunas no arquivo de dados: {len(hemoprod_go.columns)}")
        print(f"Nomes no dicionário: {len(novos_nomes)}")

except FileNotFoundError as e:
    print(f"\nErro de arquivo não encontrado: {e}")
except KeyError as e:
    print(f"\nErro de coluna não encontrada: {e}. Verifique se a coluna 'nome_sql' existe no seu arquivo de dicionário.")
except Exception as e:
    print(f"\nOcorreu um erro inesperado: {e}")



Arquivo de dados carregado com sucesso.
Número de colunas original: 257
Arquivo de dicionário carregado com sucesso.
Número de novos nomes no dicionário: 269

--- ERRO ---
A renomeação foi cancelada. O número de colunas no arquivo de dados não é igual ao número de nomes no dicionário.
Colunas no arquivo de dados: 257
Nomes no dicionário: 269


In [89]:
arquivo_dados_path = os.path.join(dados_brutos_path, 'Hemoprod_CE.xlsx')
nome_planilha = 'Planilha1'
hemoprod_ce1 = pd.read_excel(arquivo_dados_path, sheet_name=nome_planilha)

colunas_padrao = set(hemoprod_ce1.columns)
colunas_estado = set(hemoprod_go.columns)

colunas_faltantes = list(colunas_padrao - colunas_estado)

colunas_a_mais = list(colunas_estado - colunas_padrao)

print(f"Número de colunas faltantes: {len(colunas_faltantes)}")
print(f"Colunas faltantes (a serem adicionadas): {colunas_faltantes}")
print("-" * 30)
print(f"Número de colunas a mais: {len(colunas_a_mais)}")
print(f"Colunas a mais (a serem dropadas): {colunas_a_mais}")


Número de colunas faltantes: 0
Colunas faltantes (a serem adicionadas): []
------------------------------
Número de colunas a mais: 0
Colunas a mais (a serem dropadas): []


In [88]:
import os
import pandas as pd
import numpy as np # Adicionado para usar pd.NA

# --- Assumindo que você já carregou os DataFrames e calculou as listas ---
# hemoprod_ce1 (DataFrame Padrão)
# hemoprod_es (DataFrame do Estado)
# colunas_padrao, colunas_estado, colunas_faltantes, colunas_a_mais

# Nomes das colunas problemáticas
COLUNA_ERRADA = 'Ano de referência '  # A coluna que está 'a mais'
COLUNA_CORRETA = 'Ano de referência\xa0 ' # A coluna que está 'faltando'

# 1. TRATAMENTO DA COLUNA 'ANO DE REFERÊNCIA': RENOMEAR EM VEZ DE DROPAR/ADICIONAR

# if COLUNA_ERRADA in colunas_a_mais and COLUNA_CORRETA in colunas_faltantes:
#     # 1.1. Renomear a coluna no DataFrame do estado (preservando os dados)
#     hemoprod_go = hemoprod_go.rename(columns={COLUNA_ERRADA: COLUNA_CORRETA})
    
#     print(f"COLUNA RENOMEADA: '{COLUNA_ERRADA.strip()}' -> '{COLUNA_CORRETA.strip()}'")
    
#     # 1.2. Remover as colunas tratadas das listas de faltantes e a mais
#     colunas_a_mais.remove(COLUNA_ERRADA)
#     colunas_faltantes.remove(COLUNA_CORRETA)
    
#     print("A coluna de Ano de Referência foi removida das listas de ajuste.")
# else:
#     # Caso os nomes não correspondam ou a coluna não esteja nas listas (improvável, mas seguro)
#     print("A coluna de Ano de Referência não foi tratada por renomeação. Prosseguindo com drop/adição.")


# # 2. CONTINUAR com as colunas restantes: DROP e ADIÇÃO

print("\n--- INICIANDO DROP E ADIÇÃO ---")

# 2.1. DROP das colunas A MAIS restantes
if colunas_a_mais:
    # Observe que COLUNA_ERRADA não está mais aqui
    hemoprod_go = hemoprod_go.drop(columns=colunas_a_mais, errors='ignore')
    print(f"Colunas a mais (restantes) removidas: {colunas_a_mais}")
else:
    print("Nenhuma coluna a mais restante para remover.")


# 2.2. ADICIONAR as colunas FALTANTES restantes e preenchê-las com zero (0)
# A coluna COLUNA_CORRETA também não está mais aqui
colunas_de_texto_para_vazias = ['Os dados informados referem-se à um(a):'] # Adicione outras colunas de texto aqui, se houver

for coluna in colunas_faltantes:
    if coluna in colunas_de_texto_para_vazias:
        # Cria a nova coluna preenchida com valor ausente (pd.NA)
        hemoprod_go[coluna] = pd.NA
    else:
        # Cria a nova coluna preenchida com 0
        hemoprod_go[coluna] = 0

print(f"Colunas faltantes (restantes) adicionadas: {colunas_faltantes}")


# 3. REORDENAR as colunas do hemoprod_go na mesma ordem do padrão
ordem_padrao = hemoprod_ce1.columns.tolist()

# Reorganiza as colunas do hemoprod_go
hemoprod_go = hemoprod_go[ordem_padrao]

print("-" * 50)
print("Processo concluído:")
print(f"Total de colunas após o ajuste: {len(hemoprod_go.columns)}")
print("A ordem das colunas agora corresponde à ordem do hemoprod_ce1.")


--- INICIANDO DROP E ADIÇÃO ---
Nenhuma coluna a mais restante para remover.
Colunas faltantes (restantes) adicionadas: ['6. Produção Hemoterápica  6.1(a) Entradas\xa0  [Concentrado de Plaquetas de Aférese][Recebidas]', '6.1(b) Perdas  [Concentrado de Plaquetas de Aférese][Outros motivos]', '6. Produção Hemoterápica  6.1(a) Entradas\xa0  [Concentrado de Plaquetas de Aférese][Devolvidas]', '6.1(b) Perdas  [Concentrado de Plaquetas de Aférese][Validade]', '6.1(d) Distribuição para outros serviços  \xa0  [Concentrado de Plaquetas de Aférese][Total]', '6.1(c) Tranfusões  [Concentrado de Plaquetas de Aférese][Total]', '6.1(d) Distribuição para outros serviços  \xa0  [Concentrado de Plaquetas de Aférese][Com exame pré-transfusional ]', '6. Produção Hemoterápica  6.1(a) Entradas\xa0  [Concentrado de Plaquetas de Aférese][Produzidas]', '6.1(d) Distribuição para outros serviços  \xa0  [Concentrado de Plaquetas de Aférese][Sem exame pré-transfusional ]', '6.1(c) Tranfusões  [Concentrado de Plaq

In [90]:
# Mapeia as colunas, limpando espaços em branco no início e fim
# e substituindo o caractere \xa0 por um espaço normal.
novos_nomes = {col: col.strip().replace('\xa0', ' ') for col in hemoprod_go.columns}

# Aplica a renomeação
hemoprod_go = hemoprod_go.rename(columns=novos_nomes)

print("Nomes das colunas limpos e padronizados.")

Nomes das colunas limpos e padronizados.


In [91]:
print("--- Nomes das Colunas (um por linha) ---")
for coluna in hemoprod_go.columns:
    print(coluna + ",") 

--- Nomes das Colunas (um por linha) ---
ID da resposta,
Data de envio,
Última página,
Idioma inicial,
Semente,
Código de acesso,
Data de início,
Data da última ação,
Endereço IP,
IDENTIFICAÇÃO DO DADO,
Tipo de Informação  Antes de responder ao formulário, declare o tipo de informação que será inserida.,
Ano de referência,
Período de referência,
IDENTIFICAÇÃO DO ESTABELECIMENTO,
Município,
Razão Social - Nome Fantasia,
Razão Social - Nome Fantasia  [Outros],
CNPJ,
Tipo de estabelecimento,
Natureza do estabelecimento,
Os dados informados referem-se à um(a):,
Cite os estabelecimentos que compõem a rede  Informe o Tipo de Estabelecimento, o Nome Fantasia e o Município de localização de cada um.,
CNES - Cadastro Nacional de Estabelecimentos de Saúde,
Endereço,
2. Triagem Clínica  2.1 Total de candidatos quanto ao tipo de doação  [Espontânea][Aptos],
2. Triagem Clínica  2.1 Total de candidatos quanto ao tipo de doação  [Espontânea][Inaptos],
2. Triagem Clínica  2.1 Total de candidatos quant

In [92]:
hemoprod_go.head()

,ID da resposta,Data de envio,Última página,Idioma inicial,Semente,Código de acesso,Data de início,Data da última ação,Endereço IP,IDENTIFICAÇÃO DO DADO,"Tipo de Informação Antes de responder ao formulário, declare o tipo de informação que será inserida.",Ano de referência,Período de referência,IDENTIFICAÇÃO DO ESTABELECIMENTO,Município,Razão Social - Nome Fantasia,Razão Social - Nome Fantasia [Outros],CNPJ,Tipo de estabelecimento,Natureza do estabelecimento,Os dados informados referem-se à um(a):,"Cite os estabelecimentos que compõem a rede Informe o Tipo de Estabelecimento, o Nome Fantasia e o Município de localização de cada um.",CNES - Cadastro Nacional de Estabelecimentos de Saúde,Endereço,2. Triagem Clínica 2.1 Total de candidatos quanto ao tipo de doação [Espontânea][Aptos],2. Triagem Clínica 2.1 Total de candidatos quanto ao tipo de doação [Espontânea][Inaptos],2. Triagem Clínica 2.1 Total de candidatos quanto ao tipo de doação [Reposição][Aptos],2. Triagem Clínica 2.1 Total de candidatos quanto ao tipo de doação [Reposição][Inaptos],2. Triagem Clínica 2.1 Total de candidatos quanto ao tipo de doação [Autóloga][Aptos],2. Triagem Clínica 2.1 Total de candidatos quanto ao tipo de doação [Autóloga][Inaptos],2.2 Total de candidatos quanto ao tipo de doador [Primeira vez][Aptos],2.2 Total de candidatos quanto ao tipo de doador [Primeira vez][Inaptos],2.2 Total de candidatos quanto ao tipo de doador [Repetição][Aptos],2.2 Total de candidatos quanto ao tipo de doador [Repetição][Inaptos],2.2 Total de candidatos quanto ao tipo de doador [Esporádico][Aptos],2.2 Total de candidatos quanto ao tipo de doador [Esporádico][Inaptos],2.3 Total de candidatos quanto ao gênero do doador [Masculino][Aptos],2.3 Total de candidatos quanto ao gênero do doador [Masculino][Inaptos],2.3 Total de candidatos quanto ao gênero do doador [Feminino][Aptos],2.3 Total de candidatos quanto ao gênero do doador [Feminino][Inaptos],2.4 Total de candidatos quanto a idade do doador [Menor de 18 anos][Aptos],2.4 Total de candidatos quanto a idade do doador [Menor de 18 anos][Inaptos],2.4 Total de candidatos quanto a idade do doador [18 até 29 anos][Aptos],2.4 Total de candidatos quanto a idade do doador [18 até 29 anos][Inaptos],2.4 Total de candidatos quanto a idade do doador [Acima de 29 anos][Aptos],2.4 Total de candidatos quanto a idade do doador [Acima de 29 anos][Inaptos],2.5 Total de candidatos inaptos por motivo de inaptidão e por gênero [Anemia][Masculino],2.5 Total de candidatos inaptos por motivo de inaptidão e por gênero [Anemia][Feminino],2.5 Total de candidatos inaptos por motivo de inaptidão e por gênero [Anemia][Total],2.5 Total de candidatos inaptos por motivo de inaptidão e por gênero [Hipertensão][Masculino],2.5 Total de candidatos inaptos por motivo de inaptidão e por gênero [Hipertensão][Feminino],2.5 Total de candidatos inaptos por motivo de inaptidão e por gênero [Hipertensão][Total],2.5 Total de candidatos inaptos por motivo de inaptidão e por gênero [Hipotensão][Masculino],2.5 Total de candidatos inaptos por motivo de inaptidão e por gênero [Hipotensão][Feminino],2.5 Total de candidatos inaptos por motivo de inaptidão e por gênero [Hipotensão][Total],2.5 Total de candidatos inaptos por motivo de inaptidão e por gênero [Alcoolismo][Masculino],2.5 Total de candidatos inaptos por motivo de inaptidão e por gênero [Alcoolismo][Feminino],2.5 Total de candidatos inaptos por motivo de inaptidão e por gênero [Alcoolismo][Total],2.5 Total de candidatos inaptos por motivo de inaptidão e por gênero [Comportamento de risco para DST][Masculino],2.5 Total de candidatos inaptos por motivo de inaptidão e por gênero [Comportamento de risco para DST][Feminino],2.5 Total de candidatos inaptos por motivo de inaptidão e por gênero [Comportamento de risco para DST][Total],2.5 Total de candidatos inaptos por motivo de inaptidão e por gênero [Uso de drogas][Masculino],2.5 Total de candidatos inaptos por motivo de inaptidão e por gênero [Uso de drogas][Feminin

In [93]:
# Carrega o arquivo de dicionário
dicionario_path_go = ('./dicionario_colunas_269.xlsx')
dicionario = pd.read_excel(dicionario_path_go)
print("Arquivo de dicionário carregado com sucesso.")

# --- 3. Extraia a lista de novos nomes ---
# Pega os valores da coluna 'nome_sql' e converte para uma lista
novos_nomes = dicionario['nome_sql'].tolist()
print(f"Número de novos nomes no dicionário: {len(novos_nomes)}")

# --- 4. Verificação de segurança (MUITO IMPORTANTE) ---
# Garante que o número de colunas é o mesmo antes de renomear
if len(hemoprod_go.columns) == len(novos_nomes):
    print("\nO número de colunas corresponde. Renomeando...")
    
    # --- 5. Substitua os nomes das colunas ---
    # Esta é a linha principal que faz a substituição direta
    hemoprod_go.columns = novos_nomes
    
    print("Colunas renomeadas com sucesso!")
    
    # --- 6. Verifique o resultado ---
    print("\nInformações do DataFrame com as novas colunas:")
    hemoprod_go.info()
    
    print("\nAs 5 primeiras linhas com as novas colunas:")
    display(hemoprod_go.head())

else:
    # Mensagem de erro se o número de colunas for diferente
    print("\n--- ERRO ---")
    print("A renomeação foi cancelada. O número de colunas no arquivo de dados não é igual ao número de nomes no dicionário.")
    print(f"Colunas no arquivo de dados: {len(hemoprod_go.columns)}")
    print(f"Nomes no dicionário: {len(novos_nomes)}")


Arquivo de dicionário carregado com sucesso.
Número de novos nomes no dicionário: 269

O número de colunas corresponde. Renomeando...
Colunas renomeadas com sucesso!

Informações do DataFrame com as novas colunas:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 474 entries, 0 to 473
Columns: 269 entries, id to hemoprod_3_observacoes
dtypes: float64(92), int64(159), object(18)
memory usage: 996.3+ KB

As 5 primeiras linhas com as novas colunas:


,id,data_envio,ultima_pagina,idioma_inicial,semente,codigo_acesso,data_inicio,data_ultima_acao,ip,identificacao_dado,tipo_envio,ano_referencia,periodo_referencia,identificacao_estabelecimento,municipio,razao_social_nome_fantasia,razao_social_nome_fantasia_outros,cnpj,tipo_estabelecimento,natureza_estabelecimento,dados_informados_referem_se,rede_estabelecimento,cnes,endereco,triagem_clinica_total_doacao_espontanea_aptos,triagem_clinica_total_doacao_espontanea_inaptos,triagem_clinica_total_doacao_reposicao_aptos,triagem_clinica_total_doacao_reposicao_inaptos,triagem_clinica_total_doacao_autologa_aptos,triagem_clinica_total_doacao_autologa_inaptos,total_doador_primeira_vez_aptos,total_doador_primeira_vez_inaptos,total_doador_repeticao_aptos,total_doador_repeticao_inaptos,total_doador_esporadico_aptos,total_doador_esporadico_inaptos,total_doador_masculino_aptos,total_doador_masculino_inaptos,total_doador_feminino_aptos,total_doador_feminino_inaptos,total_doador_menor_de_18_anos_aptos,total_doador_menor_de_18_anos_inaptos,total_doador_18_ate_29_anos_aptos,total_doador_18_ate_29_anos_inaptos,total_doador_acima_de_29_anos_aptos,total_doador_acima_de_29_anos_inaptos,total_candidatos_inaptos_anemia_masculino,total_candidatos_inaptos_anemia_feminino,total_candidatos_inaptos_anemia_total,total_candidatos_inaptos_hipertensao_masculino,total_candidatos_inaptos_hipertensao_feminino,total_candidatos_inaptos_hipertensao_total,total_candidatos_inaptos_hipotensao_masculino,total_candidatos_inaptos_hipotensao_feminino,total_candidatos_inaptos_hipotensao_total,total_candidatos_inaptos_alcoolismo_masculino,total_candidatos_inaptos_alcoolismo_feminino,total_candidatos_inaptos_alcoolismo_total,total_candidatos_inaptos_comportamento_risco_dst_masculino,total_candidatos_inaptos_comportamento_risco_dst_feminino,total_candidatos_inaptos_comportamento_risco_dst_total,total_candidatos_inaptos_uso_drogas_masculino,total_candidatos_inaptos_uso_drogas_feminino,total_candidatos_inaptos_uso_drogas_total,total_candidatos_inaptos_hepatite_masculino,total_candidatos_inaptos_hepatite_feminino,total_candidatos_inaptos_hepatite_total,total_candidatos_inaptos_doenca_chagas_masculino,total_candidatos_inaptos_doenca_chagas_feminino,total_candidatos_inaptos_doenca_chagas_total,total_candidatos_inaptos_malaria_masculino,total_candidatos_inaptos_malaria_feminino,total_candidatos_inaptos_malaria_total,total_candidatos_inaptos_outras_masculino,total_candidatos_inaptos_outras_feminino,total_candidatos_inaptos_outras_total,coleta_total_candidatos_desistentes,total_interrupcoes_coleta_dificuldade_puncao_venosa,total_interrupcoes_coleta_reacao_vagal,total_interrupcoes_coleta_outros_motivos,total_coletas_sangue_total,total_coletas_aferese,hemoprod_1_observacoes,exames_triagem_doenca_doenca_chagas_amostras_testadas,exames_triagem_doenca_doenca_chagas_amostras_reagentes,exames_triagem_doenca_hiv_amostras_testadas,exames_triagem_doenca_hiv_amostras_reagentes,exames_triagem_doenca_sifilis_amostras_testadas,`exames_triagem_doenca_sifilis_amostras_reagentes,exames_triagem_doenca_hepatite_b_hbs_ag_amostras_testadas,exames_triagem_doenca_hepatite_b_hbs_ag_amostras_reagentes,exames_triagem_doenca_hepatite_b_anti_hbc_amostras_testadas,exames_triagem_doenca_hepatite_b_anti_hbc_amostras_reagentes,exames_triagem_doenca_hepatite_c_amostras_testadas,exames_triagem_doenca_hepatite_c_amostras_reagentes,exames_triagem_doenca_htlv_i_ii_amostras_testadas,exames_triagem_doenca_htlv_i_ii_amostras_reagentes,exames_triagem_doenca_malaria_amostras_testadas,exames_triagem_doenca_malaria_amostras_reagentes,exames_triagem_doenca_hbv_teste_nat_amostras_testadas,exames_triagem_doenca_hbv_teste_nat_amostras_reagentes,exames_triagem_doenca_hcv_teste_nat_amostras_testadas,exames_triagem_doenca_hcv_teste_nat_amostras_reagentes,exames_triagem_doenca_hiv_teste_nat_amostras_testadas,exames_triagem_doenca_hiv_teste_nat_amostras_reagentes,imunohematologia_a_positivo_doador,imunohematologia_a_positivo_receptor,imunohe

In [95]:
hemoprod_go.describe()

,id,ultima_pagina,semente,identificacao_dado,ano_referencia,identificacao_estabelecimento,cnpj,cnes,endereco,triagem_clinica_total_doacao_espontanea_aptos,triagem_clinica_total_doacao_espontanea_inaptos,triagem_clinica_total_doacao_reposicao_aptos,triagem_clinica_total_doacao_reposicao_inaptos,triagem_clinica_total_doacao_autologa_aptos,triagem_clinica_total_doacao_autologa_inaptos,total_doador_primeira_vez_aptos,total_doador_primeira_vez_inaptos,total_doador_repeticao_aptos,total_doador_repeticao_inaptos,total_doador_esporadico_aptos,total_doador_esporadico_inaptos,total_doador_masculino_aptos,total_doador_masculino_inaptos,total_doador_feminino_aptos,total_doador_feminino_inaptos,total_doador_menor_de_18_anos_aptos,total_doador_menor_de_18_anos_inaptos,total_doador_18_ate_29_anos_aptos,total_doador_18_ate_29_anos_inaptos,total_doador_acima_de_29_anos_aptos,total_doador_acima_de_29_anos_inaptos,total_candidatos_inaptos_anemia_masculino,total_candidatos_inaptos_anemia_feminino,total_candidatos_inaptos_anemia_total,total_candidatos_inaptos_hipertensao_masculino,total_candidatos_inaptos_hipertensao_feminino,total_candidatos_inaptos_hipertensao_total,total_candidatos_inaptos_hipotensao_masculino,total_candidatos_inaptos_hipotensao_feminino,total_candidatos_inaptos_hipotensao_total,total_candidatos_inaptos_alcoolismo_masculino,total_candidatos_inaptos_alcoolismo_feminino,total_candidatos_inaptos_alcoolismo_total,total_candidatos_inaptos_comportamento_risco_dst_masculino,total_candidatos_inaptos_comportamento_risco_dst_feminino,total_candidatos_inaptos_comportamento_risco_dst_total,total_candidatos_inaptos_uso_drogas_masculino,total_candidatos_inaptos_uso_drogas_feminino,total_candidatos_inaptos_uso_drogas_total,total_candidatos_inaptos_hepatite_masculino,total_candidatos_inaptos_hepatite_feminino,total_candidatos_inaptos_hepatite_total,total_candidatos_inaptos_doenca_chagas_masculino,total_candidatos_inaptos_doenca_chagas_feminino,total_candidatos_inaptos_doenca_chagas_total,total_candidatos_inaptos_malaria_masculino,total_candidatos_inaptos_malaria_feminino,total_candidatos_inaptos_malaria_total,total_candidatos_inaptos_outras_masculino,total_candidatos_inaptos_outras_feminino,total_candidatos_inaptos_outras_total,coleta_total_candidatos_desistentes,total_interrupcoes_coleta_dificuldade_puncao_venosa,total_interrupcoes_coleta_reacao_vagal,total_interrupcoes_coleta_outros_motivos,total_coletas_sangue_total,total_coletas_aferese,exames_triagem_doenca_doenca_chagas_amostras_testadas,exames_triagem_doenca_doenca_chagas_amostras_reagentes,exames_triagem_doenca_hiv_amostras_testadas,exames_triagem_doenca_hiv_amostras_reagentes,exames_triagem_doenca_sifilis_amostras_testadas,`exames_triagem_doenca_sifilis_amostras_reagentes,exames_triagem_doenca_hepatite_b_hbs_ag_amostras_testadas,exames_triagem_doenca_hepatite_b_hbs_ag_amostras_reagentes,exames_triagem_doenca_hepatite_b_anti_hbc_amostras_testadas,exames_triagem_doenca_hepatite_b_anti_hbc_amostras_reagentes,exames_triagem_doenca_hepatite_c_amostras_testadas,exames_triagem_doenca_hepatite_c_amostras_reagentes,exames_triagem_doenca_htlv_i_ii_amostras_testadas,exames_triagem_doenca_htlv_i_ii_amostras_reagentes,exames_triagem_doenca_malaria_amostras_testadas,exames_triagem_doenca_malaria_amostras_reagentes,exames_triagem_doenca_hbv_teste_nat_amostras_testadas,exames_triagem_doenca_hbv_teste_nat_amostras_reagentes,exames_triagem_doenca_hcv_teste_nat_amostras_testadas,exames_triagem_doenca_hcv_teste_nat_amostras_reagentes,exames_triagem_doenca_hiv_teste_nat_amostras_testadas,exames_triagem_doenca_hiv_teste_nat_amostras_reagentes,imunohematologia_a_positivo_doador,imunohematologia_a_positivo_receptor,imunohematologia_b_positivo_doador,imunohematologia_b_positivo_receptor,imunohematologia_ab_positivo_doador,imunohematologia_ab_positivo_receptor,imunohematologia_o_positivo_doador,imunohematologia_o_positivo_receptor,imunohematologia_a_negativo_doador,imunohematologia_a_negativo_receptor,imunohem

In [96]:
# --- 2. Defina as colunas para a chave e para ordenação ---
registros_antes = len(hemoprod_go)
print(f"Total de registros ANTES da remoção de duplicatas: {registros_antes}")
    

colunas_chave = [
        'cnpj', 
        'ano_referencia', 
        'periodo_referencia', 
        'razao_social_nome_fantasia'
    ]

coluna_data = 'data_envio'

# --- 3. Verifique se as colunas necessárias existem ---
colunas_necessarias = colunas_chave + [coluna_data]

if not all(col in hemoprod_go.columns for col in colunas_necessarias):
    print("\n--- ERRO ---")
    print("Uma ou mais colunas necessárias para a deduplicação não foram encontradas.")
    colunas_faltantes = [col for col in colunas_necessarias if col not in hemoprod_go.columns]
    print(f"Colunas necessárias: {colunas_necessarias}")
    print(f"Colunas faltantes no DataFrame: {colunas_faltantes}")
else:
    # --- 4. Prepare a coluna de data e ordene os dados ---
    # Converte a coluna 'data_envio' para datetime para garantir a ordenação correta.
    # 'errors='coerce'' transformará datas inválidas em NaT (Not a Time), que são tratadas como nulas.
    hemoprod_go[coluna_data] = pd.to_datetime(hemoprod_go[coluna_data], errors='coerce')

    # Ordena o DataFrame. Os registros com data de envio mais recente ficarão por último.
    print(f"\nOrdenando os dados por '{coluna_data}'...")
    hemoprod_go_ordenado = hemoprod_go.sort_values(by=coluna_data, ascending=True)

    # --- 5. Identifique e separe os registros duplicados e únicos ---
    # Em vez de usar drop_duplicates() diretamente, vamos usar duplicated()
    # para criar uma máscara booleana.
    # 'keep='last'' marca todas as ocorrências de uma chave como True, EXCETO a última (a mais recente).
    print(f"Identificando duplicatas com cese na chave: {colunas_chave}...")
    mascara_duplicatas = hemoprod_go_ordenado.duplicated(subset=colunas_chave, keep='last')

    # O DataFrame de removidos conterá todas as linhas marcadas como True
    hemoprod_removidos = hemoprod_go_ordenado[mascara_duplicatas]
    
    # O DataFrame deduplicado conterá o INVERSO (~) da máscara (linhas marcadas como False)
    hemoprod_go_deduplicado = hemoprod_go_ordenado[~mascara_duplicatas]
    
    # Agora podemos contar os registros diretamente dos novos DataFrames
    registros_depois = len(hemoprod_go_deduplicado)
    registros_removidos = len(hemoprod_removidos) 
    
    # --- 6. Exice o resultado ---
    print("\n--- Processo Concluído ---")
    print(f"Registros removidos: {registros_removidos}")
    print(f"Total de registros DEPOIS da remoção de duplicatas: {registros_depois}")

    # (Opcional) Exibe a amostra dos removidos
    print("\nAmostra dos dados REMOVIDOS (os mais antigos/duplicados):")
    display(hemoprod_removidos.head(10))

    # Você pode continuar a usar o DataFrame 'hemoprod_ce_deduplicado' para suas análises
    print("\nAmostra dos dados únicos (os mais recentes para cada chave):")
    display(hemoprod_go_deduplicado.head(10))
    
    # Agora você tem o DataFrame 'hemoprod_removidos' salvo

Total de registros ANTES da remoção de duplicatas: 474

Ordenando os dados por 'data_envio'...
Identificando duplicatas com cese na chave: ['cnpj', 'ano_referencia', 'periodo_referencia', 'razao_social_nome_fantasia']...

--- Processo Concluído ---
Registros removidos: 3
Total de registros DEPOIS da remoção de duplicatas: 471

Amostra dos dados REMOVIDOS (os mais antigos/duplicados):


,id,data_envio,ultima_pagina,idioma_inicial,semente,codigo_acesso,data_inicio,data_ultima_acao,ip,identificacao_dado,tipo_envio,ano_referencia,periodo_referencia,identificacao_estabelecimento,municipio,razao_social_nome_fantasia,razao_social_nome_fantasia_outros,cnpj,tipo_estabelecimento,natureza_estabelecimento,dados_informados_referem_se,rede_estabelecimento,cnes,endereco,triagem_clinica_total_doacao_espontanea_aptos,triagem_clinica_total_doacao_espontanea_inaptos,triagem_clinica_total_doacao_reposicao_aptos,triagem_clinica_total_doacao_reposicao_inaptos,triagem_clinica_total_doacao_autologa_aptos,triagem_clinica_total_doacao_autologa_inaptos,total_doador_primeira_vez_aptos,total_doador_primeira_vez_inaptos,total_doador_repeticao_aptos,total_doador_repeticao_inaptos,total_doador_esporadico_aptos,total_doador_esporadico_inaptos,total_doador_masculino_aptos,total_doador_masculino_inaptos,total_doador_feminino_aptos,total_doador_feminino_inaptos,total_doador_menor_de_18_anos_aptos,total_doador_menor_de_18_anos_inaptos,total_doador_18_ate_29_anos_aptos,total_doador_18_ate_29_anos_inaptos,total_doador_acima_de_29_anos_aptos,total_doador_acima_de_29_anos_inaptos,total_candidatos_inaptos_anemia_masculino,total_candidatos_inaptos_anemia_feminino,total_candidatos_inaptos_anemia_total,total_candidatos_inaptos_hipertensao_masculino,total_candidatos_inaptos_hipertensao_feminino,total_candidatos_inaptos_hipertensao_total,total_candidatos_inaptos_hipotensao_masculino,total_candidatos_inaptos_hipotensao_feminino,total_candidatos_inaptos_hipotensao_total,total_candidatos_inaptos_alcoolismo_masculino,total_candidatos_inaptos_alcoolismo_feminino,total_candidatos_inaptos_alcoolismo_total,total_candidatos_inaptos_comportamento_risco_dst_masculino,total_candidatos_inaptos_comportamento_risco_dst_feminino,total_candidatos_inaptos_comportamento_risco_dst_total,total_candidatos_inaptos_uso_drogas_masculino,total_candidatos_inaptos_uso_drogas_feminino,total_candidatos_inaptos_uso_drogas_total,total_candidatos_inaptos_hepatite_masculino,total_candidatos_inaptos_hepatite_feminino,total_candidatos_inaptos_hepatite_total,total_candidatos_inaptos_doenca_chagas_masculino,total_candidatos_inaptos_doenca_chagas_feminino,total_candidatos_inaptos_doenca_chagas_total,total_candidatos_inaptos_malaria_masculino,total_candidatos_inaptos_malaria_feminino,total_candidatos_inaptos_malaria_total,total_candidatos_inaptos_outras_masculino,total_candidatos_inaptos_outras_feminino,total_candidatos_inaptos_outras_total,coleta_total_candidatos_desistentes,total_interrupcoes_coleta_dificuldade_puncao_venosa,total_interrupcoes_coleta_reacao_vagal,total_interrupcoes_coleta_outros_motivos,total_coletas_sangue_total,total_coletas_aferese,hemoprod_1_observacoes,exames_triagem_doenca_doenca_chagas_amostras_testadas,exames_triagem_doenca_doenca_chagas_amostras_reagentes,exames_triagem_doenca_hiv_amostras_testadas,exames_triagem_doenca_hiv_amostras_reagentes,exames_triagem_doenca_sifilis_amostras_testadas,`exames_triagem_doenca_sifilis_amostras_reagentes,exames_triagem_doenca_hepatite_b_hbs_ag_amostras_testadas,exames_triagem_doenca_hepatite_b_hbs_ag_amostras_reagentes,exames_triagem_doenca_hepatite_b_anti_hbc_amostras_testadas,exames_triagem_doenca_hepatite_b_anti_hbc_amostras_reagentes,exames_triagem_doenca_hepatite_c_amostras_testadas,exames_triagem_doenca_hepatite_c_amostras_reagentes,exames_triagem_doenca_htlv_i_ii_amostras_testadas,exames_triagem_doenca_htlv_i_ii_amostras_reagentes,exames_triagem_doenca_malaria_amostras_testadas,exames_triagem_doenca_malaria_amostras_reagentes,exames_triagem_doenca_hbv_teste_nat_amostras_testadas,exames_triagem_doenca_hbv_teste_nat_amostras_reagentes,exames_triagem_doenca_hcv_teste_nat_amostras_testadas,exames_triagem_doenca_hcv_teste_nat_amostras_reagentes,exames_triagem_doenca_hiv_teste_nat_amostras_testadas,exames_triagem_doenca_hiv_teste_nat_amostras_reagentes,imunohematologia_a_positivo_doador,imunohematologia_a_positivo_receptor,imunohe


Amostra dos dados únicos (os mais recentes para cada chave):


,id,data_envio,ultima_pagina,idioma_inicial,semente,codigo_acesso,data_inicio,data_ultima_acao,ip,identificacao_dado,tipo_envio,ano_referencia,periodo_referencia,identificacao_estabelecimento,municipio,razao_social_nome_fantasia,razao_social_nome_fantasia_outros,cnpj,tipo_estabelecimento,natureza_estabelecimento,dados_informados_referem_se,rede_estabelecimento,cnes,endereco,triagem_clinica_total_doacao_espontanea_aptos,triagem_clinica_total_doacao_espontanea_inaptos,triagem_clinica_total_doacao_reposicao_aptos,triagem_clinica_total_doacao_reposicao_inaptos,triagem_clinica_total_doacao_autologa_aptos,triagem_clinica_total_doacao_autologa_inaptos,total_doador_primeira_vez_aptos,total_doador_primeira_vez_inaptos,total_doador_repeticao_aptos,total_doador_repeticao_inaptos,total_doador_esporadico_aptos,total_doador_esporadico_inaptos,total_doador_masculino_aptos,total_doador_masculino_inaptos,total_doador_feminino_aptos,total_doador_feminino_inaptos,total_doador_menor_de_18_anos_aptos,total_doador_menor_de_18_anos_inaptos,total_doador_18_ate_29_anos_aptos,total_doador_18_ate_29_anos_inaptos,total_doador_acima_de_29_anos_aptos,total_doador_acima_de_29_anos_inaptos,total_candidatos_inaptos_anemia_masculino,total_candidatos_inaptos_anemia_feminino,total_candidatos_inaptos_anemia_total,total_candidatos_inaptos_hipertensao_masculino,total_candidatos_inaptos_hipertensao_feminino,total_candidatos_inaptos_hipertensao_total,total_candidatos_inaptos_hipotensao_masculino,total_candidatos_inaptos_hipotensao_feminino,total_candidatos_inaptos_hipotensao_total,total_candidatos_inaptos_alcoolismo_masculino,total_candidatos_inaptos_alcoolismo_feminino,total_candidatos_inaptos_alcoolismo_total,total_candidatos_inaptos_comportamento_risco_dst_masculino,total_candidatos_inaptos_comportamento_risco_dst_feminino,total_candidatos_inaptos_comportamento_risco_dst_total,total_candidatos_inaptos_uso_drogas_masculino,total_candidatos_inaptos_uso_drogas_feminino,total_candidatos_inaptos_uso_drogas_total,total_candidatos_inaptos_hepatite_masculino,total_candidatos_inaptos_hepatite_feminino,total_candidatos_inaptos_hepatite_total,total_candidatos_inaptos_doenca_chagas_masculino,total_candidatos_inaptos_doenca_chagas_feminino,total_candidatos_inaptos_doenca_chagas_total,total_candidatos_inaptos_malaria_masculino,total_candidatos_inaptos_malaria_feminino,total_candidatos_inaptos_malaria_total,total_candidatos_inaptos_outras_masculino,total_candidatos_inaptos_outras_feminino,total_candidatos_inaptos_outras_total,coleta_total_candidatos_desistentes,total_interrupcoes_coleta_dificuldade_puncao_venosa,total_interrupcoes_coleta_reacao_vagal,total_interrupcoes_coleta_outros_motivos,total_coletas_sangue_total,total_coletas_aferese,hemoprod_1_observacoes,exames_triagem_doenca_doenca_chagas_amostras_testadas,exames_triagem_doenca_doenca_chagas_amostras_reagentes,exames_triagem_doenca_hiv_amostras_testadas,exames_triagem_doenca_hiv_amostras_reagentes,exames_triagem_doenca_sifilis_amostras_testadas,`exames_triagem_doenca_sifilis_amostras_reagentes,exames_triagem_doenca_hepatite_b_hbs_ag_amostras_testadas,exames_triagem_doenca_hepatite_b_hbs_ag_amostras_reagentes,exames_triagem_doenca_hepatite_b_anti_hbc_amostras_testadas,exames_triagem_doenca_hepatite_b_anti_hbc_amostras_reagentes,exames_triagem_doenca_hepatite_c_amostras_testadas,exames_triagem_doenca_hepatite_c_amostras_reagentes,exames_triagem_doenca_htlv_i_ii_amostras_testadas,exames_triagem_doenca_htlv_i_ii_amostras_reagentes,exames_triagem_doenca_malaria_amostras_testadas,exames_triagem_doenca_malaria_amostras_reagentes,exames_triagem_doenca_hbv_teste_nat_amostras_testadas,exames_triagem_doenca_hbv_teste_nat_amostras_reagentes,exames_triagem_doenca_hcv_teste_nat_amostras_testadas,exames_triagem_doenca_hcv_teste_nat_amostras_reagentes,exames_triagem_doenca_hiv_teste_nat_amostras_testadas,exames_triagem_doenca_hiv_teste_nat_amostras_reagentes,imunohematologia_a_positivo_doador,imunohematologia_a_positivo_receptor,imunohe

In [97]:
hemoprod_go_deduplicado.to_excel('dados_processados/hemoprod_go.xlsx', index=False)

## Hemoprod Hemominas

In [98]:
import os
import pandas as pd

dados_brutos_path = 'dados_brutos'

arquivo_dados_path = os.path.join(dados_brutos_path, 'Hemoprod_Hemominas.xlsx')
nome_planilha = 'HEMOPROD - HEMOMINAS'

dicionario_path_hm = ('./dicionario_colunas_269.xlsx')
# dicionario_path_ap = ('./dicionario_colunas_270.xlsx')

dicionario_hm = pd.read_excel(dicionario_path_hm, sheet_name='Sheet1')

# --- 2. Carregue os dados e o dicionário ---
try:
    # Carrega o arquivo de dados
    hemoprod_hm = pd.read_excel(arquivo_dados_path, sheet_name=nome_planilha)
    print("Arquivo de dados carregado com sucesso.")
    print(f"Número de colunas original: {len(hemoprod_hm.columns)}")

    # Carrega o arquivo de dicionário
    dicionario = pd.read_excel(dicionario_path_hm)
    print("Arquivo de dicionário carregado com sucesso.")

    # --- 3. Extraia a lista de novos nomes ---
    # Pega os valores da coluna 'nome_sql' e converte para uma lista
    novos_nomes = dicionario['nome_sql'].tolist()
    print(f"Número de novos nomes no dicionário: {len(novos_nomes)}")

    # --- 4. Verificação de segurança (MUITO IMPORTANTE) ---
    # Garante que o número de colunas é o mesmo antes de renomear
    if len(hemoprod_hm.columns) == len(novos_nomes):
        print("\nO número de colunas corresponde. Renomeando...")
        
        # --- 5. Substitua os nomes das colunas ---
        # Esta é a linha principal que faz a substituição direta
        hemoprod_hm.columns = novos_nomes
        
        print("Colunas renomeadas com sucesso!")
        
        # --- 6. Verifique o resultado ---
        print("\nInformações do DataFrame com as novas colunas:")
        hemoprod_hm.info()
        
        print("\nAs 5 primeiras linhas com as novas colunas:")
        display(hemoprod_hm.head())

    else:
        # Mensagem de erro se o número de colunas for diferente
        print("\n--- ERRO ---")
        print("A renomeação foi cancelada. O número de colunas no arquivo de dados não é igual ao número de nomes no dicionário.")
        print(f"Colunas no arquivo de dados: {len(hemoprod_hm.columns)}")
        print(f"Nomes no dicionário: {len(novos_nomes)}")

except FileNotFoundError as e:
    print(f"\nErro de arquivo não encontrado: {e}")
except KeyError as e:
    print(f"\nErro de coluna não encontrada: {e}. Verifique se a coluna 'nome_sql' existe no seu arquivo de dicionário.")
except Exception as e:
    print(f"\nOcorreu um erro inesperado: {e}")



Arquivo de dados carregado com sucesso.
Número de colunas original: 270
Arquivo de dicionário carregado com sucesso.
Número de novos nomes no dicionário: 269

--- ERRO ---
A renomeação foi cancelada. O número de colunas no arquivo de dados não é igual ao número de nomes no dicionário.
Colunas no arquivo de dados: 270
Nomes no dicionário: 269


In [99]:
arquivo_dados_path = os.path.join(dados_brutos_path, 'Hemoprod_CE.xlsx')
nome_planilha = 'Planilha1'
hemoprod_ce1 = pd.read_excel(arquivo_dados_path, sheet_name=nome_planilha)

colunas_padrao = set(hemoprod_ce1.columns)
colunas_estado = set(hemoprod_hm.columns)

colunas_faltantes = list(colunas_padrao - colunas_estado)

colunas_a_mais = list(colunas_estado - colunas_padrao)

print(f"Número de colunas faltantes: {len(colunas_faltantes)}")
print(f"Colunas faltantes (a serem adicionadas): {colunas_faltantes}")
print("-" * 30)
print(f"Número de colunas a mais: {len(colunas_a_mais)}")
print(f"Colunas a mais (a serem dropadas): {colunas_a_mais}")

Número de colunas faltantes: 0
Colunas faltantes (a serem adicionadas): []
------------------------------
Número de colunas a mais: 1
Colunas a mais (a serem dropadas): ['Município [Outros]']


In [100]:

print("\n--- INICIANDO DROP E ADIÇÃO ---")

# 2.1. DROP das colunas A MAIS restantes
if colunas_a_mais:
    # Observe que COLUNA_ERRADA não está mais aqui
    hemoprod_hm = hemoprod_hm.drop(columns=colunas_a_mais, errors='ignore')
    print(f"Colunas a mais (restantes) removidas: {colunas_a_mais}")
else:
    print("Nenhuma coluna a mais restante para remover.")


# 2.2. ADICIONAR as colunas FALTANTES restantes e preenchê-las com zero (0)
# A coluna COLUNA_CORRETA também não está mais aqui
colunas_de_texto_para_vazias = ['Os dados informados referem-se à um(a):'] # Adicione outras colunas de texto aqui, se houver

for coluna in colunas_faltantes:
    if coluna in colunas_de_texto_para_vazias:
        # Cria a nova coluna preenchida com valor ausente (pd.NA)
        hemoprod_hm[coluna] = pd.NA
    else:
        # Cria a nova coluna preenchida com 0
        hemoprod_hm[coluna] = 0

print(f"Colunas faltantes (restantes) adicionadas: {colunas_faltantes}")


# 3. REORDENAR as colunas do hemoprod_hm na mesma ordem do padrão
ordem_padrao = hemoprod_ce1.columns.tolist()

# Reorganiza as colunas do hemoprod_hm
hemoprod_hm = hemoprod_hm[ordem_padrao]

print("-" * 50)
print("Processo concluído:")
print(f"Total de colunas após o ajuste: {len(hemoprod_hm.columns)}")
print("A ordem das colunas agora corresponde à ordem do hemoprod_ce1.")


--- INICIANDO DROP E ADIÇÃO ---
Colunas a mais (restantes) removidas: ['Município [Outros]']
Colunas faltantes (restantes) adicionadas: []
--------------------------------------------------
Processo concluído:
Total de colunas após o ajuste: 269
A ordem das colunas agora corresponde à ordem do hemoprod_ce1.


In [101]:
 # Carrega o arquivo de dicionário
dicionario = pd.read_excel(dicionario_path_hm)
print("Arquivo de dicionário carregado com sucesso.")

# --- 3. Extraia a lista de novos nomes ---
# Pega os valores da coluna 'nome_sql' e converte para uma lista
novos_nomes = dicionario['nome_sql'].tolist()
print(f"Número de novos nomes no dicionário: {len(novos_nomes)}")

# --- 4. Verificação de segurança (MUITO IMPORTANTE) ---
# Garante que o número de colunas é o mesmo antes de renomear
if len(hemoprod_hm.columns) == len(novos_nomes):
    print("\nO número de colunas corresponde. Renomeando...")
    
    # --- 5. Substitua os nomes das colunas ---
    # Esta é a linha principal que faz a substituição direta
    hemoprod_hm.columns = novos_nomes
    
    print("Colunas renomeadas com sucesso!")
    
    # --- 6. Verifique o resultado ---
    print("\nInformações do DataFrame com as novas colunas:")
    hemoprod_hm.info()
    
    print("\nAs 5 primeiras linhas com as novas colunas:")
    display(hemoprod_hm.head())

else:
    # Mensagem de erro se o número de colunas for diferente
    print("\n--- ERRO ---")
    print("A renomeação foi cancelada. O número de colunas no arquivo de dados não é igual ao número de nomes no dicionário.")
    print(f"Colunas no arquivo de dados: {len(hemoprod_hm.columns)}")
    print(f"Nomes no dicionário: {len(novos_nomes)}")

Arquivo de dicionário carregado com sucesso.
Número de novos nomes no dicionário: 269

O número de colunas corresponde. Renomeando...
Colunas renomeadas com sucesso!

Informações do DataFrame com as novas colunas:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 930 entries, 0 to 929
Columns: 269 entries, id to hemoprod_3_observacoes
dtypes: float64(26), int64(231), object(12)
memory usage: 1.9+ MB

As 5 primeiras linhas com as novas colunas:


,id,data_envio,ultima_pagina,idioma_inicial,semente,codigo_acesso,data_inicio,data_ultima_acao,ip,identificacao_dado,tipo_envio,ano_referencia,periodo_referencia,identificacao_estabelecimento,municipio,razao_social_nome_fantasia,razao_social_nome_fantasia_outros,cnpj,tipo_estabelecimento,natureza_estabelecimento,dados_informados_referem_se,rede_estabelecimento,cnes,endereco,triagem_clinica_total_doacao_espontanea_aptos,triagem_clinica_total_doacao_espontanea_inaptos,triagem_clinica_total_doacao_reposicao_aptos,triagem_clinica_total_doacao_reposicao_inaptos,triagem_clinica_total_doacao_autologa_aptos,triagem_clinica_total_doacao_autologa_inaptos,total_doador_primeira_vez_aptos,total_doador_primeira_vez_inaptos,total_doador_repeticao_aptos,total_doador_repeticao_inaptos,total_doador_esporadico_aptos,total_doador_esporadico_inaptos,total_doador_masculino_aptos,total_doador_masculino_inaptos,total_doador_feminino_aptos,total_doador_feminino_inaptos,total_doador_menor_de_18_anos_aptos,total_doador_menor_de_18_anos_inaptos,total_doador_18_ate_29_anos_aptos,total_doador_18_ate_29_anos_inaptos,total_doador_acima_de_29_anos_aptos,total_doador_acima_de_29_anos_inaptos,total_candidatos_inaptos_anemia_masculino,total_candidatos_inaptos_anemia_feminino,total_candidatos_inaptos_anemia_total,total_candidatos_inaptos_hipertensao_masculino,total_candidatos_inaptos_hipertensao_feminino,total_candidatos_inaptos_hipertensao_total,total_candidatos_inaptos_hipotensao_masculino,total_candidatos_inaptos_hipotensao_feminino,total_candidatos_inaptos_hipotensao_total,total_candidatos_inaptos_alcoolismo_masculino,total_candidatos_inaptos_alcoolismo_feminino,total_candidatos_inaptos_alcoolismo_total,total_candidatos_inaptos_comportamento_risco_dst_masculino,total_candidatos_inaptos_comportamento_risco_dst_feminino,total_candidatos_inaptos_comportamento_risco_dst_total,total_candidatos_inaptos_uso_drogas_masculino,total_candidatos_inaptos_uso_drogas_feminino,total_candidatos_inaptos_uso_drogas_total,total_candidatos_inaptos_hepatite_masculino,total_candidatos_inaptos_hepatite_feminino,total_candidatos_inaptos_hepatite_total,total_candidatos_inaptos_doenca_chagas_masculino,total_candidatos_inaptos_doenca_chagas_feminino,total_candidatos_inaptos_doenca_chagas_total,total_candidatos_inaptos_malaria_masculino,total_candidatos_inaptos_malaria_feminino,total_candidatos_inaptos_malaria_total,total_candidatos_inaptos_outras_masculino,total_candidatos_inaptos_outras_feminino,total_candidatos_inaptos_outras_total,coleta_total_candidatos_desistentes,total_interrupcoes_coleta_dificuldade_puncao_venosa,total_interrupcoes_coleta_reacao_vagal,total_interrupcoes_coleta_outros_motivos,total_coletas_sangue_total,total_coletas_aferese,hemoprod_1_observacoes,exames_triagem_doenca_doenca_chagas_amostras_testadas,exames_triagem_doenca_doenca_chagas_amostras_reagentes,exames_triagem_doenca_hiv_amostras_testadas,exames_triagem_doenca_hiv_amostras_reagentes,exames_triagem_doenca_sifilis_amostras_testadas,`exames_triagem_doenca_sifilis_amostras_reagentes,exames_triagem_doenca_hepatite_b_hbs_ag_amostras_testadas,exames_triagem_doenca_hepatite_b_hbs_ag_amostras_reagentes,exames_triagem_doenca_hepatite_b_anti_hbc_amostras_testadas,exames_triagem_doenca_hepatite_b_anti_hbc_amostras_reagentes,exames_triagem_doenca_hepatite_c_amostras_testadas,exames_triagem_doenca_hepatite_c_amostras_reagentes,exames_triagem_doenca_htlv_i_ii_amostras_testadas,exames_triagem_doenca_htlv_i_ii_amostras_reagentes,exames_triagem_doenca_malaria_amostras_testadas,exames_triagem_doenca_malaria_amostras_reagentes,exames_triagem_doenca_hbv_teste_nat_amostras_testadas,exames_triagem_doenca_hbv_teste_nat_amostras_reagentes,exames_triagem_doenca_hcv_teste_nat_amostras_testadas,exames_triagem_doenca_hcv_teste_nat_amostras_reagentes,exames_triagem_doenca_hiv_teste_nat_amostras_testadas,exames_triagem_doenca_hiv_teste_nat_amostras_reagentes,imunohematologia_a_positivo_doador,imunohematologia_a_positivo_receptor,imunohe

In [102]:
# --- 2. Defina as colunas para a chave e para ordenação ---
registros_antes = len(hemoprod_hm)
print(f"Total de registros ANTES da remoção de duplicatas: {registros_antes}")
    

colunas_chave = [
        'cnpj', 
        'ano_referencia', 
        'periodo_referencia', 
        'razao_social_nome_fantasia'
    ]

coluna_data = 'data_envio'

# --- 3. Verifique se as colunas necessárias existem ---
colunas_necessarias = colunas_chave + [coluna_data]

if not all(col in hemoprod_hm.columns for col in colunas_necessarias):
    print("\n--- ERRO ---")
    print("Uma ou mais colunas necessárias para a deduplicação não foram encontradas.")
    colunas_faltantes = [col for col in colunas_necessarias if col not in hemoprod_hm.columns]
    print(f"Colunas necessárias: {colunas_necessarias}")
    print(f"Colunas faltantes no DataFrame: {colunas_faltantes}")
else:
    # --- 4. Prepare a coluna de data e ordene os dados ---
    # Converte a coluna 'data_envio' para datetime para garantir a ordenação correta.
    # 'errors='coerce'' transformará datas inválidas em NaT (Not a Time), que são tratadas como nulas.
    hemoprod_hm[coluna_data] = pd.to_datetime(hemoprod_hm[coluna_data], errors='coerce')

    # Ordena o DataFrame. Os registros com data de envio mais recente ficarão por último.
    print(f"\nOrdenando os dados por '{coluna_data}'...")
    hemoprod_hm_ordenado = hemoprod_hm.sort_values(by=coluna_data, ascending=True)

    # --- 5. Identifique e separe os registros duplicados e únicos ---
    # Em vez de usar drop_duplicates() diretamente, vamos usar duplicated()
    # para criar uma máscara booleana.
    # 'keep='last'' marca todas as ocorrências de uma chave como True, EXCETO a última (a mais recente).
    print(f"Identificando duplicatas com cese na chave: {colunas_chave}...")
    mascara_duplicatas = hemoprod_hm_ordenado.duplicated(subset=colunas_chave, keep='last')

    # O DataFrame de removidos conterá todas as linhas marcadas como True
    hemoprod_removidos = hemoprod_hm_ordenado[mascara_duplicatas]
    
    # O DataFrame deduplicado conterá o INVERSO (~) da máscara (linhas marcadas como False)
    hemoprod_hm_deduplicado = hemoprod_hm_ordenado[~mascara_duplicatas]
    
    # Agora podemos contar os registros diretamente dos novos DataFrames
    registros_depois = len(hemoprod_hm_deduplicado)
    registros_removidos = len(hemoprod_removidos) 
    
    # --- 6. Exice o resultado ---
    print("\n--- Processo Concluído ---")
    print(f"Registros removidos: {registros_removidos}")
    print(f"Total de registros DEPOIS da remoção de duplicatas: {registros_depois}")

    # (Opcional) Exibe a amostra dos removidos
    print("\nAmostra dos dados REMOVIDOS (os mais antigos/duplicados):")
    display(hemoprod_removidos.head(10))

    # Você pode continuar a usar o DataFrame 'hemoprod_ce_deduplicado' para suas análises
    print("\nAmostra dos dados únicos (os mais recentes para cada chave):")
    display(hemoprod_hm_deduplicado.head(10))
    
    # Agora você tem o DataFrame 'hemoprod_removidos' salvo

Total de registros ANTES da remoção de duplicatas: 930

Ordenando os dados por 'data_envio'...
Identificando duplicatas com cese na chave: ['cnpj', 'ano_referencia', 'periodo_referencia', 'razao_social_nome_fantasia']...

--- Processo Concluído ---
Registros removidos: 0
Total de registros DEPOIS da remoção de duplicatas: 930

Amostra dos dados REMOVIDOS (os mais antigos/duplicados):


,id,data_envio,ultima_pagina,idioma_inicial,semente,codigo_acesso,data_inicio,data_ultima_acao,ip,identificacao_dado,tipo_envio,ano_referencia,periodo_referencia,identificacao_estabelecimento,municipio,razao_social_nome_fantasia,razao_social_nome_fantasia_outros,cnpj,tipo_estabelecimento,natureza_estabelecimento,dados_informados_referem_se,rede_estabelecimento,cnes,endereco,triagem_clinica_total_doacao_espontanea_aptos,triagem_clinica_total_doacao_espontanea_inaptos,triagem_clinica_total_doacao_reposicao_aptos,triagem_clinica_total_doacao_reposicao_inaptos,triagem_clinica_total_doacao_autologa_aptos,triagem_clinica_total_doacao_autologa_inaptos,total_doador_primeira_vez_aptos,total_doador_primeira_vez_inaptos,total_doador_repeticao_aptos,total_doador_repeticao_inaptos,total_doador_esporadico_aptos,total_doador_esporadico_inaptos,total_doador_masculino_aptos,total_doador_masculino_inaptos,total_doador_feminino_aptos,total_doador_feminino_inaptos,total_doador_menor_de_18_anos_aptos,total_doador_menor_de_18_anos_inaptos,total_doador_18_ate_29_anos_aptos,total_doador_18_ate_29_anos_inaptos,total_doador_acima_de_29_anos_aptos,total_doador_acima_de_29_anos_inaptos,total_candidatos_inaptos_anemia_masculino,total_candidatos_inaptos_anemia_feminino,total_candidatos_inaptos_anemia_total,total_candidatos_inaptos_hipertensao_masculino,total_candidatos_inaptos_hipertensao_feminino,total_candidatos_inaptos_hipertensao_total,total_candidatos_inaptos_hipotensao_masculino,total_candidatos_inaptos_hipotensao_feminino,total_candidatos_inaptos_hipotensao_total,total_candidatos_inaptos_alcoolismo_masculino,total_candidatos_inaptos_alcoolismo_feminino,total_candidatos_inaptos_alcoolismo_total,total_candidatos_inaptos_comportamento_risco_dst_masculino,total_candidatos_inaptos_comportamento_risco_dst_feminino,total_candidatos_inaptos_comportamento_risco_dst_total,total_candidatos_inaptos_uso_drogas_masculino,total_candidatos_inaptos_uso_drogas_feminino,total_candidatos_inaptos_uso_drogas_total,total_candidatos_inaptos_hepatite_masculino,total_candidatos_inaptos_hepatite_feminino,total_candidatos_inaptos_hepatite_total,total_candidatos_inaptos_doenca_chagas_masculino,total_candidatos_inaptos_doenca_chagas_feminino,total_candidatos_inaptos_doenca_chagas_total,total_candidatos_inaptos_malaria_masculino,total_candidatos_inaptos_malaria_feminino,total_candidatos_inaptos_malaria_total,total_candidatos_inaptos_outras_masculino,total_candidatos_inaptos_outras_feminino,total_candidatos_inaptos_outras_total,coleta_total_candidatos_desistentes,total_interrupcoes_coleta_dificuldade_puncao_venosa,total_interrupcoes_coleta_reacao_vagal,total_interrupcoes_coleta_outros_motivos,total_coletas_sangue_total,total_coletas_aferese,hemoprod_1_observacoes,exames_triagem_doenca_doenca_chagas_amostras_testadas,exames_triagem_doenca_doenca_chagas_amostras_reagentes,exames_triagem_doenca_hiv_amostras_testadas,exames_triagem_doenca_hiv_amostras_reagentes,exames_triagem_doenca_sifilis_amostras_testadas,`exames_triagem_doenca_sifilis_amostras_reagentes,exames_triagem_doenca_hepatite_b_hbs_ag_amostras_testadas,exames_triagem_doenca_hepatite_b_hbs_ag_amostras_reagentes,exames_triagem_doenca_hepatite_b_anti_hbc_amostras_testadas,exames_triagem_doenca_hepatite_b_anti_hbc_amostras_reagentes,exames_triagem_doenca_hepatite_c_amostras_testadas,exames_triagem_doenca_hepatite_c_amostras_reagentes,exames_triagem_doenca_htlv_i_ii_amostras_testadas,exames_triagem_doenca_htlv_i_ii_amostras_reagentes,exames_triagem_doenca_malaria_amostras_testadas,exames_triagem_doenca_malaria_amostras_reagentes,exames_triagem_doenca_hbv_teste_nat_amostras_testadas,exames_triagem_doenca_hbv_teste_nat_amostras_reagentes,exames_triagem_doenca_hcv_teste_nat_amostras_testadas,exames_triagem_doenca_hcv_teste_nat_amostras_reagentes,exames_triagem_doenca_hiv_teste_nat_amostras_testadas,exames_triagem_doenca_hiv_teste_nat_amostras_reagentes,imunohematologia_a_positivo_doador,imunohematologia_a_positivo_receptor,imunohe


Amostra dos dados únicos (os mais recentes para cada chave):


,id,data_envio,ultima_pagina,idioma_inicial,semente,codigo_acesso,data_inicio,data_ultima_acao,ip,identificacao_dado,tipo_envio,ano_referencia,periodo_referencia,identificacao_estabelecimento,municipio,razao_social_nome_fantasia,razao_social_nome_fantasia_outros,cnpj,tipo_estabelecimento,natureza_estabelecimento,dados_informados_referem_se,rede_estabelecimento,cnes,endereco,triagem_clinica_total_doacao_espontanea_aptos,triagem_clinica_total_doacao_espontanea_inaptos,triagem_clinica_total_doacao_reposicao_aptos,triagem_clinica_total_doacao_reposicao_inaptos,triagem_clinica_total_doacao_autologa_aptos,triagem_clinica_total_doacao_autologa_inaptos,total_doador_primeira_vez_aptos,total_doador_primeira_vez_inaptos,total_doador_repeticao_aptos,total_doador_repeticao_inaptos,total_doador_esporadico_aptos,total_doador_esporadico_inaptos,total_doador_masculino_aptos,total_doador_masculino_inaptos,total_doador_feminino_aptos,total_doador_feminino_inaptos,total_doador_menor_de_18_anos_aptos,total_doador_menor_de_18_anos_inaptos,total_doador_18_ate_29_anos_aptos,total_doador_18_ate_29_anos_inaptos,total_doador_acima_de_29_anos_aptos,total_doador_acima_de_29_anos_inaptos,total_candidatos_inaptos_anemia_masculino,total_candidatos_inaptos_anemia_feminino,total_candidatos_inaptos_anemia_total,total_candidatos_inaptos_hipertensao_masculino,total_candidatos_inaptos_hipertensao_feminino,total_candidatos_inaptos_hipertensao_total,total_candidatos_inaptos_hipotensao_masculino,total_candidatos_inaptos_hipotensao_feminino,total_candidatos_inaptos_hipotensao_total,total_candidatos_inaptos_alcoolismo_masculino,total_candidatos_inaptos_alcoolismo_feminino,total_candidatos_inaptos_alcoolismo_total,total_candidatos_inaptos_comportamento_risco_dst_masculino,total_candidatos_inaptos_comportamento_risco_dst_feminino,total_candidatos_inaptos_comportamento_risco_dst_total,total_candidatos_inaptos_uso_drogas_masculino,total_candidatos_inaptos_uso_drogas_feminino,total_candidatos_inaptos_uso_drogas_total,total_candidatos_inaptos_hepatite_masculino,total_candidatos_inaptos_hepatite_feminino,total_candidatos_inaptos_hepatite_total,total_candidatos_inaptos_doenca_chagas_masculino,total_candidatos_inaptos_doenca_chagas_feminino,total_candidatos_inaptos_doenca_chagas_total,total_candidatos_inaptos_malaria_masculino,total_candidatos_inaptos_malaria_feminino,total_candidatos_inaptos_malaria_total,total_candidatos_inaptos_outras_masculino,total_candidatos_inaptos_outras_feminino,total_candidatos_inaptos_outras_total,coleta_total_candidatos_desistentes,total_interrupcoes_coleta_dificuldade_puncao_venosa,total_interrupcoes_coleta_reacao_vagal,total_interrupcoes_coleta_outros_motivos,total_coletas_sangue_total,total_coletas_aferese,hemoprod_1_observacoes,exames_triagem_doenca_doenca_chagas_amostras_testadas,exames_triagem_doenca_doenca_chagas_amostras_reagentes,exames_triagem_doenca_hiv_amostras_testadas,exames_triagem_doenca_hiv_amostras_reagentes,exames_triagem_doenca_sifilis_amostras_testadas,`exames_triagem_doenca_sifilis_amostras_reagentes,exames_triagem_doenca_hepatite_b_hbs_ag_amostras_testadas,exames_triagem_doenca_hepatite_b_hbs_ag_amostras_reagentes,exames_triagem_doenca_hepatite_b_anti_hbc_amostras_testadas,exames_triagem_doenca_hepatite_b_anti_hbc_amostras_reagentes,exames_triagem_doenca_hepatite_c_amostras_testadas,exames_triagem_doenca_hepatite_c_amostras_reagentes,exames_triagem_doenca_htlv_i_ii_amostras_testadas,exames_triagem_doenca_htlv_i_ii_amostras_reagentes,exames_triagem_doenca_malaria_amostras_testadas,exames_triagem_doenca_malaria_amostras_reagentes,exames_triagem_doenca_hbv_teste_nat_amostras_testadas,exames_triagem_doenca_hbv_teste_nat_amostras_reagentes,exames_triagem_doenca_hcv_teste_nat_amostras_testadas,exames_triagem_doenca_hcv_teste_nat_amostras_reagentes,exames_triagem_doenca_hiv_teste_nat_amostras_testadas,exames_triagem_doenca_hiv_teste_nat_amostras_reagentes,imunohematologia_a_positivo_doador,imunohematologia_a_positivo_receptor,imunohe

In [103]:
hemoprod_hm_deduplicado.to_excel('dados_processados/hemoprod_hm.xlsx', index=False)

## Hemoprod Maranhão

In [106]:
import os
import pandas as pd

dados_brutos_path = 'dados_brutos'

arquivo_dados_path = os.path.join(dados_brutos_path, 'Hemoprod_MA.xlsx')
nome_planilha = 'HEMOPROD - MARANHAO'

dicionario_path_ma = ('./dicionario_colunas_269.xlsx')
# dicionario_path_ap = ('./dicionario_colunas_270.xlsx')

dicionario_ma = pd.read_excel(dicionario_path_ma, sheet_name='Sheet1')

# --- 2. Carregue os dados e o dicionário ---
try:
    # Carrega o arquivo de dados
    hemoprod_ma = pd.read_excel(arquivo_dados_path, sheet_name=nome_planilha)
    print("Arquivo de dados carregado com sucesso.")
    print(f"Número de colunas original: {len(hemoprod_ma.columns)}")

    # Carrega o arquivo de dicionário
    dicionario = pd.read_excel(dicionario_path_ma)
    print("Arquivo de dicionário carregado com sucesso.")

    # --- 3. Extraia a lista de novos nomes ---
    # Pega os valores da coluna 'nome_sql' e converte para uma lista
    novos_nomes = dicionario['nome_sql'].tolist()
    print(f"Número de novos nomes no dicionário: {len(novos_nomes)}")

    # --- 4. Verificação de segurança (MUITO IMPORTANTE) ---
    # Garante que o número de colunas é o mesmo antes de renomear
    if len(hemoprod_ma.columns) == len(novos_nomes):
        print("\nO número de colunas corresponde. Renomeando...")
        
        # --- 5. Substitua os nomes das colunas ---
        # Esta é a linha principal que faz a substituição direta
        hemoprod_ma.columns = novos_nomes
        
        print("Colunas renomeadas com sucesso!")
        
        # --- 6. Verifique o resultado ---
        print("\nInformações do DataFrame com as novas colunas:")
        hemoprod_ma.info()
        
        print("\nAs 5 primeiras linhas com as novas colunas:")
        display(hemoprod_ma.head())

    else:
        # Mensagem de erro se o número de colunas for diferente
        print("\n--- ERRO ---")
        print("A renomeação foi cancelada. O número de colunas no arquivo de dados não é igual ao número de nomes no dicionário.")
        print(f"Colunas no arquivo de dados: {len(hemoprod_ma.columns)}")
        print(f"Nomes no dicionário: {len(novos_nomes)}")

except FileNotFoundError as e:
    print(f"\nErro de arquivo não encontrado: {e}")
except KeyError as e:
    print(f"\nErro de coluna não encontrada: {e}. Verifique se a coluna 'nome_sql' existe no seu arquivo de dicionário.")
except Exception as e:
    print(f"\nOcorreu um erro inesperado: {e}")



Arquivo de dados carregado com sucesso.
Número de colunas original: 269
Arquivo de dicionário carregado com sucesso.
Número de novos nomes no dicionário: 269

O número de colunas corresponde. Renomeando...
Colunas renomeadas com sucesso!

Informações do DataFrame com as novas colunas:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45 entries, 0 to 44
Columns: 269 entries, id to hemoprod_3_observacoes
dtypes: float64(95), int64(160), object(14)
memory usage: 94.7+ KB

As 5 primeiras linhas com as novas colunas:


,id,data_envio,ultima_pagina,idioma_inicial,semente,codigo_acesso,data_inicio,data_ultima_acao,ip,identificacao_dado,tipo_envio,ano_referencia,periodo_referencia,identificacao_estabelecimento,municipio,razao_social_nome_fantasia,razao_social_nome_fantasia_outros,cnpj,tipo_estabelecimento,natureza_estabelecimento,dados_informados_referem_se,rede_estabelecimento,cnes,endereco,triagem_clinica_total_doacao_espontanea_aptos,triagem_clinica_total_doacao_espontanea_inaptos,triagem_clinica_total_doacao_reposicao_aptos,triagem_clinica_total_doacao_reposicao_inaptos,triagem_clinica_total_doacao_autologa_aptos,triagem_clinica_total_doacao_autologa_inaptos,total_doador_primeira_vez_aptos,total_doador_primeira_vez_inaptos,total_doador_repeticao_aptos,total_doador_repeticao_inaptos,total_doador_esporadico_aptos,total_doador_esporadico_inaptos,total_doador_masculino_aptos,total_doador_masculino_inaptos,total_doador_feminino_aptos,total_doador_feminino_inaptos,total_doador_menor_de_18_anos_aptos,total_doador_menor_de_18_anos_inaptos,total_doador_18_ate_29_anos_aptos,total_doador_18_ate_29_anos_inaptos,total_doador_acima_de_29_anos_aptos,total_doador_acima_de_29_anos_inaptos,total_candidatos_inaptos_anemia_masculino,total_candidatos_inaptos_anemia_feminino,total_candidatos_inaptos_anemia_total,total_candidatos_inaptos_hipertensao_masculino,total_candidatos_inaptos_hipertensao_feminino,total_candidatos_inaptos_hipertensao_total,total_candidatos_inaptos_hipotensao_masculino,total_candidatos_inaptos_hipotensao_feminino,total_candidatos_inaptos_hipotensao_total,total_candidatos_inaptos_alcoolismo_masculino,total_candidatos_inaptos_alcoolismo_feminino,total_candidatos_inaptos_alcoolismo_total,total_candidatos_inaptos_comportamento_risco_dst_masculino,total_candidatos_inaptos_comportamento_risco_dst_feminino,total_candidatos_inaptos_comportamento_risco_dst_total,total_candidatos_inaptos_uso_drogas_masculino,total_candidatos_inaptos_uso_drogas_feminino,total_candidatos_inaptos_uso_drogas_total,total_candidatos_inaptos_hepatite_masculino,total_candidatos_inaptos_hepatite_feminino,total_candidatos_inaptos_hepatite_total,total_candidatos_inaptos_doenca_chagas_masculino,total_candidatos_inaptos_doenca_chagas_feminino,total_candidatos_inaptos_doenca_chagas_total,total_candidatos_inaptos_malaria_masculino,total_candidatos_inaptos_malaria_feminino,total_candidatos_inaptos_malaria_total,total_candidatos_inaptos_outras_masculino,total_candidatos_inaptos_outras_feminino,total_candidatos_inaptos_outras_total,coleta_total_candidatos_desistentes,total_interrupcoes_coleta_dificuldade_puncao_venosa,total_interrupcoes_coleta_reacao_vagal,total_interrupcoes_coleta_outros_motivos,total_coletas_sangue_total,total_coletas_aferese,hemoprod_1_observacoes,exames_triagem_doenca_doenca_chagas_amostras_testadas,exames_triagem_doenca_doenca_chagas_amostras_reagentes,exames_triagem_doenca_hiv_amostras_testadas,exames_triagem_doenca_hiv_amostras_reagentes,exames_triagem_doenca_sifilis_amostras_testadas,`exames_triagem_doenca_sifilis_amostras_reagentes,exames_triagem_doenca_hepatite_b_hbs_ag_amostras_testadas,exames_triagem_doenca_hepatite_b_hbs_ag_amostras_reagentes,exames_triagem_doenca_hepatite_b_anti_hbc_amostras_testadas,exames_triagem_doenca_hepatite_b_anti_hbc_amostras_reagentes,exames_triagem_doenca_hepatite_c_amostras_testadas,exames_triagem_doenca_hepatite_c_amostras_reagentes,exames_triagem_doenca_htlv_i_ii_amostras_testadas,exames_triagem_doenca_htlv_i_ii_amostras_reagentes,exames_triagem_doenca_malaria_amostras_testadas,exames_triagem_doenca_malaria_amostras_reagentes,exames_triagem_doenca_hbv_teste_nat_amostras_testadas,exames_triagem_doenca_hbv_teste_nat_amostras_reagentes,exames_triagem_doenca_hcv_teste_nat_amostras_testadas,exames_triagem_doenca_hcv_teste_nat_amostras_reagentes,exames_triagem_doenca_hiv_teste_nat_amostras_testadas,exames_triagem_doenca_hiv_teste_nat_amostras_reagentes,imunohematologia_a_positivo_doador,imunohematologia_a_positivo_receptor,imunohe

In [107]:
# --- 2. Defina as colunas para a chave e para ordenação ---
registros_antes = len(hemoprod_ma)
print(f"Total de registros ANTES da remoção de duplicatas: {registros_antes}")
    

colunas_chave = [
        'cnpj', 
        'ano_referencia', 
        'periodo_referencia', 
        'razao_social_nome_fantasia'
    ]

coluna_data = 'data_envio'

# --- 3. Verifique se as colunas necessárias existem ---
colunas_necessarias = colunas_chave + [coluna_data]

if not all(col in hemoprod_ma.columns for col in colunas_necessarias):
    print("\n--- ERRO ---")
    print("Uma ou mais colunas necessárias para a deduplicação não foram encontradas.")
    colunas_faltantes = [col for col in colunas_necessarias if col not in hemoprod_ma.columns]
    print(f"Colunas necessárias: {colunas_necessarias}")
    print(f"Colunas faltantes no DataFrame: {colunas_faltantes}")
else:
    # --- 4. Prepare a coluna de data e ordene os dados ---
    # Converte a coluna 'data_envio' para datetime para garantir a ordenação correta.
    # 'errors='coerce'' transformará datas inválidas em NaT (Not a Time), que são tratadas como nulas.
    hemoprod_ma[coluna_data] = pd.to_datetime(hemoprod_ma[coluna_data], errors='coerce')

    # Ordena o DataFrame. Os registros com data de envio mais recente ficarão por último.
    print(f"\nOrdenando os dados por '{coluna_data}'...")
    hemoprod_ma_ordenado = hemoprod_ma.sort_values(by=coluna_data, ascending=True)

    # --- 5. Identifique e separe os registros duplicados e únicos ---
    # Em vez de usar drop_duplicates() diretamente, vamos usar duplicated()
    # para criar uma máscara booleana.
    # 'keep='last'' marca todas as ocorrências de uma chave como True, EXCETO a última (a mais recente).
    print(f"Identificando duplicatas com cese na chave: {colunas_chave}...")
    mascara_duplicatas = hemoprod_ma_ordenado.duplicated(subset=colunas_chave, keep='last')

    # O DataFrame de removidos conterá todas as linhas marcadas como True
    hemoprod_removidos = hemoprod_ma_ordenado[mascara_duplicatas]
    
    # O DataFrame deduplicado conterá o INVERSO (~) da máscara (linhas marcadas como False)
    hemoprod_ma_deduplicado = hemoprod_ma_ordenado[~mascara_duplicatas]
    
    # Agora podemos contar os registros diretamente dos novos DataFrames
    registros_depois = len(hemoprod_ma_deduplicado)
    registros_removidos = len(hemoprod_removidos) 
    
    # --- 6. Exice o resultado ---
    print("\n--- Processo Concluído ---")
    print(f"Registros removidos: {registros_removidos}")
    print(f"Total de registros DEPOIS da remoção de duplicatas: {registros_depois}")

    # (Opcional) Exibe a amostra dos removidos
    print("\nAmostra dos dados REMOVIDOS (os mais antigos/duplicados):")
    display(hemoprod_removidos.head(10))

    # Você pode continuar a usar o DataFrame 'hemoprod_ce_deduplicado' para suas análises
    print("\nAmostra dos dados únicos (os mais recentes para cada chave):")
    display(hemoprod_ma_deduplicado.head(10))
    
    # Agora você tem o DataFrame 'hemoprod_removidos' salvo

Total de registros ANTES da remoção de duplicatas: 45

Ordenando os dados por 'data_envio'...
Identificando duplicatas com cese na chave: ['cnpj', 'ano_referencia', 'periodo_referencia', 'razao_social_nome_fantasia']...

--- Processo Concluído ---
Registros removidos: 1
Total de registros DEPOIS da remoção de duplicatas: 44

Amostra dos dados REMOVIDOS (os mais antigos/duplicados):


,id,data_envio,ultima_pagina,idioma_inicial,semente,codigo_acesso,data_inicio,data_ultima_acao,ip,identificacao_dado,tipo_envio,ano_referencia,periodo_referencia,identificacao_estabelecimento,municipio,razao_social_nome_fantasia,razao_social_nome_fantasia_outros,cnpj,tipo_estabelecimento,natureza_estabelecimento,dados_informados_referem_se,rede_estabelecimento,cnes,endereco,triagem_clinica_total_doacao_espontanea_aptos,triagem_clinica_total_doacao_espontanea_inaptos,triagem_clinica_total_doacao_reposicao_aptos,triagem_clinica_total_doacao_reposicao_inaptos,triagem_clinica_total_doacao_autologa_aptos,triagem_clinica_total_doacao_autologa_inaptos,total_doador_primeira_vez_aptos,total_doador_primeira_vez_inaptos,total_doador_repeticao_aptos,total_doador_repeticao_inaptos,total_doador_esporadico_aptos,total_doador_esporadico_inaptos,total_doador_masculino_aptos,total_doador_masculino_inaptos,total_doador_feminino_aptos,total_doador_feminino_inaptos,total_doador_menor_de_18_anos_aptos,total_doador_menor_de_18_anos_inaptos,total_doador_18_ate_29_anos_aptos,total_doador_18_ate_29_anos_inaptos,total_doador_acima_de_29_anos_aptos,total_doador_acima_de_29_anos_inaptos,total_candidatos_inaptos_anemia_masculino,total_candidatos_inaptos_anemia_feminino,total_candidatos_inaptos_anemia_total,total_candidatos_inaptos_hipertensao_masculino,total_candidatos_inaptos_hipertensao_feminino,total_candidatos_inaptos_hipertensao_total,total_candidatos_inaptos_hipotensao_masculino,total_candidatos_inaptos_hipotensao_feminino,total_candidatos_inaptos_hipotensao_total,total_candidatos_inaptos_alcoolismo_masculino,total_candidatos_inaptos_alcoolismo_feminino,total_candidatos_inaptos_alcoolismo_total,total_candidatos_inaptos_comportamento_risco_dst_masculino,total_candidatos_inaptos_comportamento_risco_dst_feminino,total_candidatos_inaptos_comportamento_risco_dst_total,total_candidatos_inaptos_uso_drogas_masculino,total_candidatos_inaptos_uso_drogas_feminino,total_candidatos_inaptos_uso_drogas_total,total_candidatos_inaptos_hepatite_masculino,total_candidatos_inaptos_hepatite_feminino,total_candidatos_inaptos_hepatite_total,total_candidatos_inaptos_doenca_chagas_masculino,total_candidatos_inaptos_doenca_chagas_feminino,total_candidatos_inaptos_doenca_chagas_total,total_candidatos_inaptos_malaria_masculino,total_candidatos_inaptos_malaria_feminino,total_candidatos_inaptos_malaria_total,total_candidatos_inaptos_outras_masculino,total_candidatos_inaptos_outras_feminino,total_candidatos_inaptos_outras_total,coleta_total_candidatos_desistentes,total_interrupcoes_coleta_dificuldade_puncao_venosa,total_interrupcoes_coleta_reacao_vagal,total_interrupcoes_coleta_outros_motivos,total_coletas_sangue_total,total_coletas_aferese,hemoprod_1_observacoes,exames_triagem_doenca_doenca_chagas_amostras_testadas,exames_triagem_doenca_doenca_chagas_amostras_reagentes,exames_triagem_doenca_hiv_amostras_testadas,exames_triagem_doenca_hiv_amostras_reagentes,exames_triagem_doenca_sifilis_amostras_testadas,`exames_triagem_doenca_sifilis_amostras_reagentes,exames_triagem_doenca_hepatite_b_hbs_ag_amostras_testadas,exames_triagem_doenca_hepatite_b_hbs_ag_amostras_reagentes,exames_triagem_doenca_hepatite_b_anti_hbc_amostras_testadas,exames_triagem_doenca_hepatite_b_anti_hbc_amostras_reagentes,exames_triagem_doenca_hepatite_c_amostras_testadas,exames_triagem_doenca_hepatite_c_amostras_reagentes,exames_triagem_doenca_htlv_i_ii_amostras_testadas,exames_triagem_doenca_htlv_i_ii_amostras_reagentes,exames_triagem_doenca_malaria_amostras_testadas,exames_triagem_doenca_malaria_amostras_reagentes,exames_triagem_doenca_hbv_teste_nat_amostras_testadas,exames_triagem_doenca_hbv_teste_nat_amostras_reagentes,exames_triagem_doenca_hcv_teste_nat_amostras_testadas,exames_triagem_doenca_hcv_teste_nat_amostras_reagentes,exames_triagem_doenca_hiv_teste_nat_amostras_testadas,exames_triagem_doenca_hiv_teste_nat_amostras_reagentes,imunohematologia_a_positivo_doador,imunohematologia_a_positivo_receptor,imunohe


Amostra dos dados únicos (os mais recentes para cada chave):


,id,data_envio,ultima_pagina,idioma_inicial,semente,codigo_acesso,data_inicio,data_ultima_acao,ip,identificacao_dado,tipo_envio,ano_referencia,periodo_referencia,identificacao_estabelecimento,municipio,razao_social_nome_fantasia,razao_social_nome_fantasia_outros,cnpj,tipo_estabelecimento,natureza_estabelecimento,dados_informados_referem_se,rede_estabelecimento,cnes,endereco,triagem_clinica_total_doacao_espontanea_aptos,triagem_clinica_total_doacao_espontanea_inaptos,triagem_clinica_total_doacao_reposicao_aptos,triagem_clinica_total_doacao_reposicao_inaptos,triagem_clinica_total_doacao_autologa_aptos,triagem_clinica_total_doacao_autologa_inaptos,total_doador_primeira_vez_aptos,total_doador_primeira_vez_inaptos,total_doador_repeticao_aptos,total_doador_repeticao_inaptos,total_doador_esporadico_aptos,total_doador_esporadico_inaptos,total_doador_masculino_aptos,total_doador_masculino_inaptos,total_doador_feminino_aptos,total_doador_feminino_inaptos,total_doador_menor_de_18_anos_aptos,total_doador_menor_de_18_anos_inaptos,total_doador_18_ate_29_anos_aptos,total_doador_18_ate_29_anos_inaptos,total_doador_acima_de_29_anos_aptos,total_doador_acima_de_29_anos_inaptos,total_candidatos_inaptos_anemia_masculino,total_candidatos_inaptos_anemia_feminino,total_candidatos_inaptos_anemia_total,total_candidatos_inaptos_hipertensao_masculino,total_candidatos_inaptos_hipertensao_feminino,total_candidatos_inaptos_hipertensao_total,total_candidatos_inaptos_hipotensao_masculino,total_candidatos_inaptos_hipotensao_feminino,total_candidatos_inaptos_hipotensao_total,total_candidatos_inaptos_alcoolismo_masculino,total_candidatos_inaptos_alcoolismo_feminino,total_candidatos_inaptos_alcoolismo_total,total_candidatos_inaptos_comportamento_risco_dst_masculino,total_candidatos_inaptos_comportamento_risco_dst_feminino,total_candidatos_inaptos_comportamento_risco_dst_total,total_candidatos_inaptos_uso_drogas_masculino,total_candidatos_inaptos_uso_drogas_feminino,total_candidatos_inaptos_uso_drogas_total,total_candidatos_inaptos_hepatite_masculino,total_candidatos_inaptos_hepatite_feminino,total_candidatos_inaptos_hepatite_total,total_candidatos_inaptos_doenca_chagas_masculino,total_candidatos_inaptos_doenca_chagas_feminino,total_candidatos_inaptos_doenca_chagas_total,total_candidatos_inaptos_malaria_masculino,total_candidatos_inaptos_malaria_feminino,total_candidatos_inaptos_malaria_total,total_candidatos_inaptos_outras_masculino,total_candidatos_inaptos_outras_feminino,total_candidatos_inaptos_outras_total,coleta_total_candidatos_desistentes,total_interrupcoes_coleta_dificuldade_puncao_venosa,total_interrupcoes_coleta_reacao_vagal,total_interrupcoes_coleta_outros_motivos,total_coletas_sangue_total,total_coletas_aferese,hemoprod_1_observacoes,exames_triagem_doenca_doenca_chagas_amostras_testadas,exames_triagem_doenca_doenca_chagas_amostras_reagentes,exames_triagem_doenca_hiv_amostras_testadas,exames_triagem_doenca_hiv_amostras_reagentes,exames_triagem_doenca_sifilis_amostras_testadas,`exames_triagem_doenca_sifilis_amostras_reagentes,exames_triagem_doenca_hepatite_b_hbs_ag_amostras_testadas,exames_triagem_doenca_hepatite_b_hbs_ag_amostras_reagentes,exames_triagem_doenca_hepatite_b_anti_hbc_amostras_testadas,exames_triagem_doenca_hepatite_b_anti_hbc_amostras_reagentes,exames_triagem_doenca_hepatite_c_amostras_testadas,exames_triagem_doenca_hepatite_c_amostras_reagentes,exames_triagem_doenca_htlv_i_ii_amostras_testadas,exames_triagem_doenca_htlv_i_ii_amostras_reagentes,exames_triagem_doenca_malaria_amostras_testadas,exames_triagem_doenca_malaria_amostras_reagentes,exames_triagem_doenca_hbv_teste_nat_amostras_testadas,exames_triagem_doenca_hbv_teste_nat_amostras_reagentes,exames_triagem_doenca_hcv_teste_nat_amostras_testadas,exames_triagem_doenca_hcv_teste_nat_amostras_reagentes,exames_triagem_doenca_hiv_teste_nat_amostras_testadas,exames_triagem_doenca_hiv_teste_nat_amostras_reagentes,imunohematologia_a_positivo_doador,imunohematologia_a_positivo_receptor,imunohe

In [108]:
hemoprod_ma_deduplicado.to_excel('dados_processados/hemoprod_ma.xlsx', index=False)

## Hemoprod Minas Gerais

In [109]:
import os
import pandas as pd

dados_brutos_path = 'dados_brutos'

arquivo_dados_path = os.path.join(dados_brutos_path, 'Hemoprod_MG.xlsx')
nome_planilha = 'HEMOPROD - MINASGERAIS'

dicionario_path_mg = ('./dicionario_colunas_269.xlsx')
# dicionario_path_ap = ('./dicionario_colunas_270.xlsx')

dicionario_mg = pd.read_excel(dicionario_path_mg, sheet_name='Sheet1')

# --- 2. Carregue os dados e o dicionário ---
try:
    # Carrega o arquivo de dados
    hemoprod_mg = pd.read_excel(arquivo_dados_path, sheet_name=nome_planilha)
    print("Arquivo de dados carregado com sucesso.")
    print(f"Número de colunas original: {len(hemoprod_mg.columns)}")

    # Carrega o arquivo de dicionário
    dicionario = pd.read_excel(dicionario_path_mg)
    print("Arquivo de dicionário carregado com sucesso.")

    # --- 3. Extraia a lista de novos nomes ---
    # Pega os valores da coluna 'nome_sql' e converte para uma lista
    novos_nomes = dicionario['nome_sql'].tolist()
    print(f"Número de novos nomes no dicionário: {len(novos_nomes)}")

    # --- 4. Verificação de segurança (MUITO IMPORTANTE) ---
    # Garante que o número de colunas é o mesmo antes de renomear
    if len(hemoprod_mg.columns) == len(novos_nomes):
        print("\nO número de colunas corresponde. Renomeando...")
        
        # --- 5. Substitua os nomes das colunas ---
        # Esta é a linha principal que faz a substituição direta
        hemoprod_mg.columns = novos_nomes
        
        print("Colunas renomeadas com sucesso!")
        
        # --- 6. Verifique o resultado ---
        print("\nInformações do DataFrame com as novas colunas:")
        hemoprod_mg.info()
        
        print("\nAs 5 primeiras linhas com as novas colunas:")
        display(hemoprod_mg.head())

    else:
        # Mensagem de erro se o número de colunas for diferente
        print("\n--- ERRO ---")
        print("A renomeação foi cancelada. O número de colunas no arquivo de dados não é igual ao número de nomes no dicionário.")
        print(f"Colunas no arquivo de dados: {len(hemoprod_mg.columns)}")
        print(f"Nomes no dicionário: {len(novos_nomes)}")

except FileNotFoundError as e:
    print(f"\nErro de arquivo não encontrado: {e}")
except KeyError as e:
    print(f"\nErro de coluna não encontrada: {e}. Verifique se a coluna 'nome_sql' existe no seu arquivo de dicionário.")
except Exception as e:
    print(f"\nOcorreu um erro inesperado: {e}")



Arquivo de dados carregado com sucesso.
Número de colunas original: 270
Arquivo de dicionário carregado com sucesso.
Número de novos nomes no dicionário: 269

--- ERRO ---
A renomeação foi cancelada. O número de colunas no arquivo de dados não é igual ao número de nomes no dicionário.
Colunas no arquivo de dados: 270
Nomes no dicionário: 269


In [110]:
arquivo_dados_path = os.path.join(dados_brutos_path, 'Hemoprod_CE.xlsx')
nome_planilha = 'Planilha1'
hemoprod_ce1 = pd.read_excel(arquivo_dados_path, sheet_name=nome_planilha)

colunas_padrao = set(hemoprod_ce1.columns)
colunas_estado = set(hemoprod_mg.columns)

colunas_faltantes = list(colunas_padrao - colunas_estado)

colunas_a_mais = list(colunas_estado - colunas_padrao)

print(f"Número de colunas faltantes: {len(colunas_faltantes)}")
print(f"Colunas faltantes (a serem adicionadas): {colunas_faltantes}")
print("-" * 30)
print(f"Número de colunas a mais: {len(colunas_a_mais)}")
print(f"Colunas a mais (a serem dropadas): {colunas_a_mais}")

Número de colunas faltantes: 0
Colunas faltantes (a serem adicionadas): []
------------------------------
Número de colunas a mais: 1
Colunas a mais (a serem dropadas): ['Município [Outros]']


In [111]:

print("\n--- INICIANDO DROP E ADIÇÃO ---")

# 2.1. DROP das colunas A MAIS restantes
if colunas_a_mais:
    # Observe que COLUNA_ERRADA não está mais aqui
    hemoprod_mg = hemoprod_mg.drop(columns=colunas_a_mais, errors='ignore')
    print(f"Colunas a mais (restantes) removidas: {colunas_a_mais}")
else:
    print("Nenhuma coluna a mais restante para remover.")


# 2.2. ADICIONAR as colunas FALTANTES restantes e preenchê-las com zero (0)
# A coluna COLUNA_CORRETA também não está mais aqui
colunas_de_texto_para_vazias = ['Os dados informados referem-se à um(a):'] # Adicione outras colunas de texto aqui, se houver

for coluna in colunas_faltantes:
    if coluna in colunas_de_texto_para_vazias:
        # Cria a nova coluna preenchida com valor ausente (pd.NA)
        hemoprod_mg[coluna] = pd.NA
    else:
        # Cria a nova coluna preenchida com 0
        hemoprod_mg[coluna] = 0

print(f"Colunas faltantes (restantes) adicionadas: {colunas_faltantes}")


# 3. REORDENAR as colunas do hemoprod_mg na mesma ordem do padrão
ordem_padrao = hemoprod_ce1.columns.tolist()

# Reorganiza as colunas do hemoprod_mg
hemoprod_mg = hemoprod_mg[ordem_padrao]

print("-" * 50)
print("Processo concluído:")
print(f"Total de colunas após o ajuste: {len(hemoprod_mg.columns)}")
print("A ordem das colunas agora corresponde à ordem do hemoprod_ce1.")


--- INICIANDO DROP E ADIÇÃO ---
Colunas a mais (restantes) removidas: ['Município [Outros]']
Colunas faltantes (restantes) adicionadas: []
--------------------------------------------------
Processo concluído:
Total de colunas após o ajuste: 269
A ordem das colunas agora corresponde à ordem do hemoprod_ce1.


In [112]:
 # Carrega o arquivo de dicionário
dicionario = pd.read_excel(dicionario_path_mg)
print("Arquivo de dicionário carregado com sucesso.")

# --- 3. Extraia a lista de novos nomes ---
# Pega os valores da coluna 'nome_sql' e converte para uma lista
novos_nomes = dicionario['nome_sql'].tolist()
print(f"Número de novos nomes no dicionário: {len(novos_nomes)}")

# --- 4. Verificação de segurança (MUITO IMPORTANTE) ---
# Garante que o número de colunas é o mesmo antes de renomear
if len(hemoprod_mg.columns) == len(novos_nomes):
    print("\nO número de colunas corresponde. Renomeando...")
    
    # --- 5. Substitua os nomes das colunas ---
    # Esta é a linha principal que faz a substituição direta
    hemoprod_mg.columns = novos_nomes
    
    print("Colunas renomeadas com sucesso!")
    
    # --- 6. Verifique o resultado ---
    print("\nInformações do DataFrame com as novas colunas:")
    hemoprod_mg.info()
    
    print("\nAs 5 primeiras linhas com as novas colunas:")
    display(hemoprod_mg.head())

else:
    # Mensagem de erro se o número de colunas for diferente
    print("\n--- ERRO ---")
    print("A renomeação foi cancelada. O número de colunas no arquivo de dados não é igual ao número de nomes no dicionário.")
    print(f"Colunas no arquivo de dados: {len(hemoprod_mg.columns)}")
    print(f"Nomes no dicionário: {len(novos_nomes)}")

Arquivo de dicionário carregado com sucesso.
Número de novos nomes no dicionário: 269

O número de colunas corresponde. Renomeando...
Colunas renomeadas com sucesso!

Informações do DataFrame com as novas colunas:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 271 entries, 0 to 270
Columns: 269 entries, id to hemoprod_3_observacoes
dtypes: float64(92), int64(160), object(17)
memory usage: 569.7+ KB

As 5 primeiras linhas com as novas colunas:


,id,data_envio,ultima_pagina,idioma_inicial,semente,codigo_acesso,data_inicio,data_ultima_acao,ip,identificacao_dado,tipo_envio,ano_referencia,periodo_referencia,identificacao_estabelecimento,municipio,razao_social_nome_fantasia,razao_social_nome_fantasia_outros,cnpj,tipo_estabelecimento,natureza_estabelecimento,dados_informados_referem_se,rede_estabelecimento,cnes,endereco,triagem_clinica_total_doacao_espontanea_aptos,triagem_clinica_total_doacao_espontanea_inaptos,triagem_clinica_total_doacao_reposicao_aptos,triagem_clinica_total_doacao_reposicao_inaptos,triagem_clinica_total_doacao_autologa_aptos,triagem_clinica_total_doacao_autologa_inaptos,total_doador_primeira_vez_aptos,total_doador_primeira_vez_inaptos,total_doador_repeticao_aptos,total_doador_repeticao_inaptos,total_doador_esporadico_aptos,total_doador_esporadico_inaptos,total_doador_masculino_aptos,total_doador_masculino_inaptos,total_doador_feminino_aptos,total_doador_feminino_inaptos,total_doador_menor_de_18_anos_aptos,total_doador_menor_de_18_anos_inaptos,total_doador_18_ate_29_anos_aptos,total_doador_18_ate_29_anos_inaptos,total_doador_acima_de_29_anos_aptos,total_doador_acima_de_29_anos_inaptos,total_candidatos_inaptos_anemia_masculino,total_candidatos_inaptos_anemia_feminino,total_candidatos_inaptos_anemia_total,total_candidatos_inaptos_hipertensao_masculino,total_candidatos_inaptos_hipertensao_feminino,total_candidatos_inaptos_hipertensao_total,total_candidatos_inaptos_hipotensao_masculino,total_candidatos_inaptos_hipotensao_feminino,total_candidatos_inaptos_hipotensao_total,total_candidatos_inaptos_alcoolismo_masculino,total_candidatos_inaptos_alcoolismo_feminino,total_candidatos_inaptos_alcoolismo_total,total_candidatos_inaptos_comportamento_risco_dst_masculino,total_candidatos_inaptos_comportamento_risco_dst_feminino,total_candidatos_inaptos_comportamento_risco_dst_total,total_candidatos_inaptos_uso_drogas_masculino,total_candidatos_inaptos_uso_drogas_feminino,total_candidatos_inaptos_uso_drogas_total,total_candidatos_inaptos_hepatite_masculino,total_candidatos_inaptos_hepatite_feminino,total_candidatos_inaptos_hepatite_total,total_candidatos_inaptos_doenca_chagas_masculino,total_candidatos_inaptos_doenca_chagas_feminino,total_candidatos_inaptos_doenca_chagas_total,total_candidatos_inaptos_malaria_masculino,total_candidatos_inaptos_malaria_feminino,total_candidatos_inaptos_malaria_total,total_candidatos_inaptos_outras_masculino,total_candidatos_inaptos_outras_feminino,total_candidatos_inaptos_outras_total,coleta_total_candidatos_desistentes,total_interrupcoes_coleta_dificuldade_puncao_venosa,total_interrupcoes_coleta_reacao_vagal,total_interrupcoes_coleta_outros_motivos,total_coletas_sangue_total,total_coletas_aferese,hemoprod_1_observacoes,exames_triagem_doenca_doenca_chagas_amostras_testadas,exames_triagem_doenca_doenca_chagas_amostras_reagentes,exames_triagem_doenca_hiv_amostras_testadas,exames_triagem_doenca_hiv_amostras_reagentes,exames_triagem_doenca_sifilis_amostras_testadas,`exames_triagem_doenca_sifilis_amostras_reagentes,exames_triagem_doenca_hepatite_b_hbs_ag_amostras_testadas,exames_triagem_doenca_hepatite_b_hbs_ag_amostras_reagentes,exames_triagem_doenca_hepatite_b_anti_hbc_amostras_testadas,exames_triagem_doenca_hepatite_b_anti_hbc_amostras_reagentes,exames_triagem_doenca_hepatite_c_amostras_testadas,exames_triagem_doenca_hepatite_c_amostras_reagentes,exames_triagem_doenca_htlv_i_ii_amostras_testadas,exames_triagem_doenca_htlv_i_ii_amostras_reagentes,exames_triagem_doenca_malaria_amostras_testadas,exames_triagem_doenca_malaria_amostras_reagentes,exames_triagem_doenca_hbv_teste_nat_amostras_testadas,exames_triagem_doenca_hbv_teste_nat_amostras_reagentes,exames_triagem_doenca_hcv_teste_nat_amostras_testadas,exames_triagem_doenca_hcv_teste_nat_amostras_reagentes,exames_triagem_doenca_hiv_teste_nat_amostras_testadas,exames_triagem_doenca_hiv_teste_nat_amostras_reagentes,imunohematologia_a_positivo_doador,imunohematologia_a_positivo_receptor,imunohe

In [113]:
    # --- 2. Defina as colunas para a chave e para ordenação ---
registros_antes = len(hemoprod_mg)
print(f"Total de registros ANTES da remoção de duplicatas: {registros_antes}")
    

colunas_chave = [
        'cnpj', 
        'ano_referencia', 
        'periodo_referencia', 
        'razao_social_nome_fantasia'
    ]

coluna_data = 'data_envio'

# --- 3. Verifique se as colunas necessárias existem ---
colunas_necessarias = colunas_chave + [coluna_data]

if not all(col in hemoprod_mg.columns for col in colunas_necessarias):
    print("\n--- ERRO ---")
    print("Uma ou mais colunas necessárias para a deduplicação não foram encontradas.")
    colunas_faltantes = [col for col in colunas_necessarias if col not in hemoprod_mg.columns]
    print(f"Colunas necessárias: {colunas_necessarias}")
    print(f"Colunas faltantes no DataFrame: {colunas_faltantes}")
else:
    # --- 4. Prepare a coluna de data e ordene os dados ---
    # Converte a coluna 'data_envio' para datetime para garantir a ordenação correta.
    # 'errors='coerce'' transformará datas inválidas em NaT (Not a Time), que são tratadas como nulas.
    hemoprod_mg[coluna_data] = pd.to_datetime(hemoprod_mg[coluna_data], errors='coerce')

    # Ordena o DataFrame. Os registros com data de envio mais recente ficarão por último.
    print(f"\nOrdenando os dados por '{coluna_data}'...")
    hemoprod_mg_ordenado = hemoprod_mg.sort_values(by=coluna_data, ascending=True)

    # --- 5. Identifique e separe os registros duplicados e únicos ---
    # Em vez de usar drop_duplicates() diretamente, vamos usar duplicated()
    # para criar uma máscara booleana.
    # 'keep='last'' marca todas as ocorrências de uma chave como True, EXCETO a última (a mais recente).
    print(f"Identificando duplicatas com cese na chave: {colunas_chave}...")
    mascara_duplicatas = hemoprod_mg_ordenado.duplicated(subset=colunas_chave, keep='last')

    # O DataFrame de removidos conterá todas as linhas marcadas como True
    hemoprod_removidos = hemoprod_mg_ordenado[mascara_duplicatas]
    
    # O DataFrame deduplicado conterá o INVERSO (~) da máscara (linhas marcadas como False)
    hemoprod_mg_deduplicado = hemoprod_mg_ordenado[~mascara_duplicatas]
    
    # Agora podemos contar os registros diretamente dos novos DataFrames
    registros_depois = len(hemoprod_mg_deduplicado)
    registros_removidos = len(hemoprod_removidos) 
    
    # --- 6. Exice o resultado ---
    print("\n--- Processo Concluído ---")
    print(f"Registros removidos: {registros_removidos}")
    print(f"Total de registros DEPOIS da remoção de duplicatas: {registros_depois}")

    # (Opcional) Exibe a amostra dos removidos
    print("\nAmostra dos dados REMOVIDOS (os mais antigos/duplicados):")
    display(hemoprod_removidos.head(10))

    # Você pode continuar a usar o DataFrame 'hemoprod_ce_deduplicado' para suas análises
    print("\nAmostra dos dados únicos (os mais recentes para cada chave):")
    display(hemoprod_mg_deduplicado.head(10))
    
    # Agora você tem o DataFrame 'hemoprod_removidos' salvo

Total de registros ANTES da remoção de duplicatas: 271

Ordenando os dados por 'data_envio'...
Identificando duplicatas com cese na chave: ['cnpj', 'ano_referencia', 'periodo_referencia', 'razao_social_nome_fantasia']...

--- Processo Concluído ---
Registros removidos: 8
Total de registros DEPOIS da remoção de duplicatas: 263

Amostra dos dados REMOVIDOS (os mais antigos/duplicados):


,id,data_envio,ultima_pagina,idioma_inicial,semente,codigo_acesso,data_inicio,data_ultima_acao,ip,identificacao_dado,tipo_envio,ano_referencia,periodo_referencia,identificacao_estabelecimento,municipio,razao_social_nome_fantasia,razao_social_nome_fantasia_outros,cnpj,tipo_estabelecimento,natureza_estabelecimento,dados_informados_referem_se,rede_estabelecimento,cnes,endereco,triagem_clinica_total_doacao_espontanea_aptos,triagem_clinica_total_doacao_espontanea_inaptos,triagem_clinica_total_doacao_reposicao_aptos,triagem_clinica_total_doacao_reposicao_inaptos,triagem_clinica_total_doacao_autologa_aptos,triagem_clinica_total_doacao_autologa_inaptos,total_doador_primeira_vez_aptos,total_doador_primeira_vez_inaptos,total_doador_repeticao_aptos,total_doador_repeticao_inaptos,total_doador_esporadico_aptos,total_doador_esporadico_inaptos,total_doador_masculino_aptos,total_doador_masculino_inaptos,total_doador_feminino_aptos,total_doador_feminino_inaptos,total_doador_menor_de_18_anos_aptos,total_doador_menor_de_18_anos_inaptos,total_doador_18_ate_29_anos_aptos,total_doador_18_ate_29_anos_inaptos,total_doador_acima_de_29_anos_aptos,total_doador_acima_de_29_anos_inaptos,total_candidatos_inaptos_anemia_masculino,total_candidatos_inaptos_anemia_feminino,total_candidatos_inaptos_anemia_total,total_candidatos_inaptos_hipertensao_masculino,total_candidatos_inaptos_hipertensao_feminino,total_candidatos_inaptos_hipertensao_total,total_candidatos_inaptos_hipotensao_masculino,total_candidatos_inaptos_hipotensao_feminino,total_candidatos_inaptos_hipotensao_total,total_candidatos_inaptos_alcoolismo_masculino,total_candidatos_inaptos_alcoolismo_feminino,total_candidatos_inaptos_alcoolismo_total,total_candidatos_inaptos_comportamento_risco_dst_masculino,total_candidatos_inaptos_comportamento_risco_dst_feminino,total_candidatos_inaptos_comportamento_risco_dst_total,total_candidatos_inaptos_uso_drogas_masculino,total_candidatos_inaptos_uso_drogas_feminino,total_candidatos_inaptos_uso_drogas_total,total_candidatos_inaptos_hepatite_masculino,total_candidatos_inaptos_hepatite_feminino,total_candidatos_inaptos_hepatite_total,total_candidatos_inaptos_doenca_chagas_masculino,total_candidatos_inaptos_doenca_chagas_feminino,total_candidatos_inaptos_doenca_chagas_total,total_candidatos_inaptos_malaria_masculino,total_candidatos_inaptos_malaria_feminino,total_candidatos_inaptos_malaria_total,total_candidatos_inaptos_outras_masculino,total_candidatos_inaptos_outras_feminino,total_candidatos_inaptos_outras_total,coleta_total_candidatos_desistentes,total_interrupcoes_coleta_dificuldade_puncao_venosa,total_interrupcoes_coleta_reacao_vagal,total_interrupcoes_coleta_outros_motivos,total_coletas_sangue_total,total_coletas_aferese,hemoprod_1_observacoes,exames_triagem_doenca_doenca_chagas_amostras_testadas,exames_triagem_doenca_doenca_chagas_amostras_reagentes,exames_triagem_doenca_hiv_amostras_testadas,exames_triagem_doenca_hiv_amostras_reagentes,exames_triagem_doenca_sifilis_amostras_testadas,`exames_triagem_doenca_sifilis_amostras_reagentes,exames_triagem_doenca_hepatite_b_hbs_ag_amostras_testadas,exames_triagem_doenca_hepatite_b_hbs_ag_amostras_reagentes,exames_triagem_doenca_hepatite_b_anti_hbc_amostras_testadas,exames_triagem_doenca_hepatite_b_anti_hbc_amostras_reagentes,exames_triagem_doenca_hepatite_c_amostras_testadas,exames_triagem_doenca_hepatite_c_amostras_reagentes,exames_triagem_doenca_htlv_i_ii_amostras_testadas,exames_triagem_doenca_htlv_i_ii_amostras_reagentes,exames_triagem_doenca_malaria_amostras_testadas,exames_triagem_doenca_malaria_amostras_reagentes,exames_triagem_doenca_hbv_teste_nat_amostras_testadas,exames_triagem_doenca_hbv_teste_nat_amostras_reagentes,exames_triagem_doenca_hcv_teste_nat_amostras_testadas,exames_triagem_doenca_hcv_teste_nat_amostras_reagentes,exames_triagem_doenca_hiv_teste_nat_amostras_testadas,exames_triagem_doenca_hiv_teste_nat_amostras_reagentes,imunohematologia_a_positivo_doador,imunohematologia_a_positivo_receptor,imunohe


Amostra dos dados únicos (os mais recentes para cada chave):


,id,data_envio,ultima_pagina,idioma_inicial,semente,codigo_acesso,data_inicio,data_ultima_acao,ip,identificacao_dado,tipo_envio,ano_referencia,periodo_referencia,identificacao_estabelecimento,municipio,razao_social_nome_fantasia,razao_social_nome_fantasia_outros,cnpj,tipo_estabelecimento,natureza_estabelecimento,dados_informados_referem_se,rede_estabelecimento,cnes,endereco,triagem_clinica_total_doacao_espontanea_aptos,triagem_clinica_total_doacao_espontanea_inaptos,triagem_clinica_total_doacao_reposicao_aptos,triagem_clinica_total_doacao_reposicao_inaptos,triagem_clinica_total_doacao_autologa_aptos,triagem_clinica_total_doacao_autologa_inaptos,total_doador_primeira_vez_aptos,total_doador_primeira_vez_inaptos,total_doador_repeticao_aptos,total_doador_repeticao_inaptos,total_doador_esporadico_aptos,total_doador_esporadico_inaptos,total_doador_masculino_aptos,total_doador_masculino_inaptos,total_doador_feminino_aptos,total_doador_feminino_inaptos,total_doador_menor_de_18_anos_aptos,total_doador_menor_de_18_anos_inaptos,total_doador_18_ate_29_anos_aptos,total_doador_18_ate_29_anos_inaptos,total_doador_acima_de_29_anos_aptos,total_doador_acima_de_29_anos_inaptos,total_candidatos_inaptos_anemia_masculino,total_candidatos_inaptos_anemia_feminino,total_candidatos_inaptos_anemia_total,total_candidatos_inaptos_hipertensao_masculino,total_candidatos_inaptos_hipertensao_feminino,total_candidatos_inaptos_hipertensao_total,total_candidatos_inaptos_hipotensao_masculino,total_candidatos_inaptos_hipotensao_feminino,total_candidatos_inaptos_hipotensao_total,total_candidatos_inaptos_alcoolismo_masculino,total_candidatos_inaptos_alcoolismo_feminino,total_candidatos_inaptos_alcoolismo_total,total_candidatos_inaptos_comportamento_risco_dst_masculino,total_candidatos_inaptos_comportamento_risco_dst_feminino,total_candidatos_inaptos_comportamento_risco_dst_total,total_candidatos_inaptos_uso_drogas_masculino,total_candidatos_inaptos_uso_drogas_feminino,total_candidatos_inaptos_uso_drogas_total,total_candidatos_inaptos_hepatite_masculino,total_candidatos_inaptos_hepatite_feminino,total_candidatos_inaptos_hepatite_total,total_candidatos_inaptos_doenca_chagas_masculino,total_candidatos_inaptos_doenca_chagas_feminino,total_candidatos_inaptos_doenca_chagas_total,total_candidatos_inaptos_malaria_masculino,total_candidatos_inaptos_malaria_feminino,total_candidatos_inaptos_malaria_total,total_candidatos_inaptos_outras_masculino,total_candidatos_inaptos_outras_feminino,total_candidatos_inaptos_outras_total,coleta_total_candidatos_desistentes,total_interrupcoes_coleta_dificuldade_puncao_venosa,total_interrupcoes_coleta_reacao_vagal,total_interrupcoes_coleta_outros_motivos,total_coletas_sangue_total,total_coletas_aferese,hemoprod_1_observacoes,exames_triagem_doenca_doenca_chagas_amostras_testadas,exames_triagem_doenca_doenca_chagas_amostras_reagentes,exames_triagem_doenca_hiv_amostras_testadas,exames_triagem_doenca_hiv_amostras_reagentes,exames_triagem_doenca_sifilis_amostras_testadas,`exames_triagem_doenca_sifilis_amostras_reagentes,exames_triagem_doenca_hepatite_b_hbs_ag_amostras_testadas,exames_triagem_doenca_hepatite_b_hbs_ag_amostras_reagentes,exames_triagem_doenca_hepatite_b_anti_hbc_amostras_testadas,exames_triagem_doenca_hepatite_b_anti_hbc_amostras_reagentes,exames_triagem_doenca_hepatite_c_amostras_testadas,exames_triagem_doenca_hepatite_c_amostras_reagentes,exames_triagem_doenca_htlv_i_ii_amostras_testadas,exames_triagem_doenca_htlv_i_ii_amostras_reagentes,exames_triagem_doenca_malaria_amostras_testadas,exames_triagem_doenca_malaria_amostras_reagentes,exames_triagem_doenca_hbv_teste_nat_amostras_testadas,exames_triagem_doenca_hbv_teste_nat_amostras_reagentes,exames_triagem_doenca_hcv_teste_nat_amostras_testadas,exames_triagem_doenca_hcv_teste_nat_amostras_reagentes,exames_triagem_doenca_hiv_teste_nat_amostras_testadas,exames_triagem_doenca_hiv_teste_nat_amostras_reagentes,imunohematologia_a_positivo_doador,imunohematologia_a_positivo_receptor,imunohe

In [114]:
hemoprod_mg_deduplicado.to_excel('dados_processados/hemoprod_mg.xlsx', index=False)

## Hemoprod Mato Grosso do Sul

In [ ]:
import os
import pandas as pd

dados_brutos_path = 'dados_brutos'

arquivo_dados_path = os.path.join(dados_brutos_path, 'Hemoprod_MS.xlsx')
nome_planilha = 'HEMOPROD - MATOGROSSODOSUL'

dicionario_path_ms = ('./dicionario_colunas_270.xlsx')
# dicionario_path_ap = ('./dicionario_colunas_270.xlsx')

dicionario_ms = pd.read_excel(dicionario_path_ms, sheet_name='Sheet1')

# --- 2. Carregue os dados e o dicionário ---
try:
    # Carrega o arquivo de dados
    hemoprod_ms = pd.read_excel(arquivo_dados_path, sheet_name=nome_planilha)
    print("Arquivo de dados carregado com sucesso.")
    print(f"Número de colunas original: {len(hemoprod_ms.columns)}")

    # Carrega o arquivo de dicionário
    dicionario = pd.read_excel(dicionario_path_ms)
    print("Arquivo de dicionário carregado com sucesso.")

    # --- 3. Extraia a lista de novos nomes ---
    # Pega os valores da coluna 'nome_sql' e converte para uma lista
    novos_nomes = dicionario['nome_sql'].tolist()
    print(f"Número de novos nomes no dicionário: {len(novos_nomes)}")

    # --- 4. Verificação de segurança (MUITO IMPORTANTE) ---
    # Garante que o número de colunas é o mesmo antes de renomear
    if len(hemoprod_ms.columns) == len(novos_nomes):
        print("\nO número de colunas corresponde. Renomeando...")
        
        # --- 5. Substitua os nomes das colunas ---
        # Esta é a linha principal que faz a substituição direta
        hemoprod_ms.columns = novos_nomes
        
        print("Colunas renomeadas com sucesso!")
        
        # --- 6. Verifique o resultado ---
        print("\nInformações do DataFrame com as novas colunas:")
        hemoprod_ms.info()
        
        print("\nAs 5 primeiras linhas com as novas colunas:")
        display(hemoprod_ms.head())

    else:
        # Mensagem de erro se o número de colunas for diferente
        print("\n--- ERRO ---")
        print("A renomeação foi cancelada. O número de colunas no arquivo de dados não é igual ao número de nomes no dicionário.")
        print(f"Colunas no arquivo de dados: {len(hemoprod_ms.columns)}")
        print(f"Nomes no dicionário: {len(novos_nomes)}")

except FileNotFoundError as e:
    print(f"\nErro de arquivo não encontrado: {e}")
except KeyError as e:
    print(f"\nErro de coluna não encontrada: {e}. Verifique se a coluna 'nome_sql' existe no seu arquivo de dicionário.")
except Exception as e:
    print(f"\nOcorreu um erro inesperado: {e}")



Arquivo de dados carregado com sucesso.
Número de colunas original: 269
Arquivo de dicionário carregado com sucesso.
Número de novos nomes no dicionário: 270

--- ERRO ---
A renomeação foi cancelada. O número de colunas no arquivo de dados não é igual ao número de nomes no dicionário.
Colunas no arquivo de dados: 269
Nomes no dicionário: 270


In [6]:

arquivo_dados_path = os.path.join(dados_brutos_path, 'Hemoprod_AM.xlsx')
nome_planilha = 'HEMOPROD - AMAZONAS'
hemoprod_ce1 = pd.read_excel(arquivo_dados_path, sheet_name=nome_planilha)

colunas_padrao = set(hemoprod_ce1.columns)
colunas_estado = set(hemoprod_ms.columns)

colunas_faltantes = list(colunas_padrao - colunas_estado)

colunas_a_mais = list(colunas_estado - colunas_padrao)

print(f"Número de colunas faltantes: {len(colunas_faltantes)}")
print(f"Colunas faltantes (a serem adicionadas): {colunas_faltantes}")
print("-" * 30)
print(f"Número de colunas a mais: {len(colunas_a_mais)}")
print(f"Colunas a mais (a serem dropadas): {colunas_a_mais}")

Número de colunas faltantes: 3
Colunas faltantes (a serem adicionadas): ['Os dados informados referem-se à um(a): ', 'Cite os estabelecimentos que compõem a rede  Informe o Tipo de Estabelecimento, o Nome Fantasia\xa0e o Município de localização de cada um. ', 'Ano de referência ']
------------------------------
Número de colunas a mais: 2
Colunas a mais (a serem dropadas): ['Município [Outros]', 'Período de referência [Outros]']


## Analisando o auto exclusao


In [1]:
import pandas as pd
import os

# 1. Definir o caminho do arquivo
parquet_path = "dados_processados/base_nacional.parquet"
coluna_alvo = "descarte_bolsas_total_bolsas_descartadas_auto_exclusao"

# 2. Verificar se o arquivo existe antes de tentar carregar
if not os.path.exists(parquet_path):
    print(f"Erro: Arquivo não encontrado no caminho: {parquet_path}")
else:
    try:
        # 3. Carregar o DataFrame do arquivo Parquet
        # O pandas usa o pyarrow ou fastparquet internamente para ler o .parquet
        df = pd.read_parquet(parquet_path)
        print(f"DataFrame carregado com sucesso. Total de linhas: {len(df)}")
        
        # 4. Verificar se a coluna existe no DataFrame
        if coluna_alvo in df.columns:
            # 5. Calcular a soma da coluna
            total_descarte = df[coluna_alvo].sum()
            
            # 6. Exibir o resultado formatado
            # O f-string com ":," formata o número com separador de milhar
            print("\n--- Resultado ---")
            print(f"Soma total de '{coluna_alvo}':")
            print(f"-> {int(total_descarte):,}")
            
        else:
            print(f"\nErro: A coluna '{coluna_alvo}' não foi encontrada no arquivo Parquet.")
            print(f"Colunas disponíveis: {df.columns.tolist()}")

    except Exception as e:
        print(f"Ocorreu um erro ao carregar o arquivo Parquet: {e}")

DataFrame carregado com sucesso. Total de linhas: 15865

--- Resultado ---
Soma total de 'descarte_bolsas_total_bolsas_descartadas_auto_exclusao':
-> 20,579


In [2]:
import pandas as pd
import os

# 1. Defina o caminho do arquivo Parquet
# Certifique-se de que este caminho está correto em relação ao seu notebook.
parquet_path = "dados_processados/base_nacional.parquet"
coluna_municipio = "municipio"
coluna_descarte = "descarte_bolsas_total_bolsas_descartadas_auto_exclusao"

# 2. Verificar se o arquivo existe e carregar
if not os.path.exists(parquet_path):
    print(f"ERRO: Arquivo não encontrado no caminho: {parquet_path}")
    print("Verifique se o caminho está correto ou se o arquivo foi gerado.")
else:
    try:
        # Carregar o DataFrame
        df = pd.read_parquet(parquet_path)
        print(f"DataFrame carregado com sucesso. Total de linhas: {len(df):,}")

        # 3. Verificar a existência das colunas
        if coluna_municipio in df.columns and coluna_descarte in df.columns:
            
            # 4. Agrupar por Município e somar o total de descartes
            # O .fillna(0) garante que valores nulos não atrapalhem a soma.
            df_analise = (
                df.groupby(coluna_municipio)[coluna_descarte]
                .sum()
                .fillna(0) 
                .sort_values(ascending=False)
            )

            # 5. Selecionar o Top 50
            top_50_municipios = df_analise.head(50)

            # 6. Exibir o resultado
            print("\n--- TOP 50 Municípios por Total de Bolsas Descartadas ---")
            
            # Usando to_frame() e reset_index() para uma exibição formatada
            df_resultado = top_50_municipios.to_frame().reset_index()
            
            # Renomear coluna para clareza
            df_resultado.columns = ["Município", "Total Bolsas Descartadas"]
            
            # Formatação opcional para melhor visualização no notebook
            pd.options.display.float_format = '{:,.0f}'.format
            
            # Exibe a tabela no Jupyter/Pandas
            display(df_resultado)

        else:
            print("\nERRO: Uma ou ambas as colunas necessárias não foram encontradas no DataFrame.")
            print(f"Esperadas: '{coluna_municipio}' e '{coluna_descarte}'")
            
    except Exception as e:
        print(f"Ocorreu um erro ao carregar ou processar o arquivo Parquet: {e}")

DataFrame carregado com sucesso. Total de linhas: 15,865

--- TOP 50 Municípios por Total de Bolsas Descartadas ---


,Município,Total Bolsas Descartadas
0,Não se aplica,10083
1,Curitiba,1472
2,Campo Grande,1003
3,Goiânia,578
4,Natal,536
5,Vitória,491
6,João Pessoa,445
7,Manaus,363
8,Maringá,321
9,Belém,312


In [6]:
import pandas as pd
import os

# 1. Defina o caminho do arquivo Parquet
# Certifique-se de que este caminho está correto em relação ao seu notebook.
parquet_path = "dados_processados/base_nacional.parquet"

# 2. Definir as colunas de análise
coluna_municipio = "municipio"
colunas_de_soma = [
    "total_coletas_sangue_total",
]

# 3. Verificar se o arquivo existe e carregar
if not os.path.exists(parquet_path):
    print(f"ERRO: Arquivo não encontrado no caminho: {parquet_path}")
    print("Verifique se o caminho está correto ou se o arquivo foi gerado.")
else:
    try:
        # Carregar o DataFrame
        # O argumento columns= otimiza a leitura lendo apenas as colunas necessárias + a coluna de agrupamento
        cols_to_load = [coluna_municipio] + colunas_de_soma
        df = pd.read_parquet(parquet_path, columns=cols_to_load)
        print(f"DataFrame carregado com sucesso. Total de linhas: {len(df):,}")

        # 4. Agrupar por Município e somar as colunas
        agg_functions = {col: "sum" for col in colunas_de_soma}
        
        municipio_stats = (
            df.groupby(coluna_municipio)
            .agg(agg_functions)
            .fillna(0) # Trata possíveis valores nulos nas colunas de soma
            .reset_index()
        )

        # 5. Calcular a coluna "Total Coletas" (como no seu código Streamlit)
        municipio_stats["Total Coletas"] = (
            municipio_stats["total_coletas_sangue_total"]
        )
        
        # 6. Ordenar e selecionar o Top 50
        municipio_stats = municipio_stats.sort_values("Total Coletas", ascending=False)
        top_50_municipios = municipio_stats.head(5)

        # 7. Exibir o resultado
        print("\n--- TOP 50 Municípios por Total de Coletas (Sangue Total + Aférese) ---")
        
        # Formatação opcional para melhor visualização dos números no notebook
        pd.options.display.float_format = '{:,.0f}'.format
        
        # Exibe a tabela no Jupyter/Pandas
        display(top_50_municipios)

    except Exception as e:
        print(f"Ocorreu um erro ao carregar ou processar o arquivo Parquet: {e}")

DataFrame carregado com sucesso. Total de linhas: 15,865

--- TOP 50 Municípios por Total de Coletas (Sangue Total + Aférese) ---


,municipio,total_coletas_sangue_total,Total Coletas
234,Não se aplica,2832656,2832656
112,Curitiba,427623,427623
140,Fortaleza,339609,339609
294,Recife,281227,281227
398,Vitória,246585,246585
